# P0 — Enviroment Prep

In [ ]:
# ==============================================================
# P0 — Environment Prep (paths, dataset availability, output dirs)
# --------------------------------------------------------------
# Purpose:
#   Make the notebook executable from a clean kernel by preparing
#   directory paths, ensuring the OULAD files are available, and
#   creating the output structure used by the paper exports.
# ==============================================================

import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

BASE_PATH = Path.cwd()
CONTENT_DIR = BASE_PATH / "content"
OUTPUTS_DIR = BASE_PATH / "outputs_v2"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"
ZIP_PATH = BASE_PATH / "anonymisedData.zip"
DATASET_URL = "https://analyse.kmi.open.ac.uk/open-dataset/download"

DATASET_FILES = {
    "studentInfo": "studentInfo.csv",
    "studentVle": "studentVle.csv",
    "assessments": "assessments.csv",
    "studentAssessment": "studentAssessment.csv",
    "studentRegistration": "studentRegistration.csv",
    "courses": "courses.csv",
    "vle": "vle.csv",
}

def _dataset_is_ready() -> bool:
    return CONTENT_DIR.exists() and all((CONTENT_DIR / fname).exists() for fname in DATASET_FILES.values())

def _download_dataset() -> None:
    if ZIP_PATH.exists():
        print(f"Zip already available: {ZIP_PATH}")
        return

    print(f"Downloading dataset to {ZIP_PATH}...")
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0 Safari/537.36"
        )
    }
    response = requests.get(DATASET_URL, headers=headers, stream=True, timeout=120)
    response.raise_for_status()

    with open(ZIP_PATH, "wb") as handle:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                handle.write(chunk)

    print(f"Download completed: {ZIP_PATH}")

def _extract_dataset() -> None:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Zip file not found: {ZIP_PATH}")

    CONTENT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(CONTENT_DIR)

    print(f"Dataset extracted to: {CONTENT_DIR}")

def ensure_dataset_available() -> None:
    if _dataset_is_ready():
        print(f"Dataset already available in: {CONTENT_DIR}")
        return

    if not ZIP_PATH.exists():
        _download_dataset()

    _extract_dataset()

    missing = [fname for fname in DATASET_FILES.values() if not (CONTENT_DIR / fname).exists()]
    if missing:
        raise FileNotFoundError(f"Missing extracted CSV files: {missing}")

def prepare_output_dirs(reset_outputs: bool = False) -> None:
    if reset_outputs and OUTPUTS_DIR.exists():
        shutil.rmtree(OUTPUTS_DIR)
        print(f"Removed existing output directory: {OUTPUTS_DIR}")

    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Output tables dir : {TABLES_DIR}")
    print(f"Output figures dir: {FIGURES_DIR}")

ensure_dataset_available()
prepare_output_dirs(reset_outputs=False)

print("=== P0 — Environment Prep ===")
print(f"BASE_PATH   : {BASE_PATH}")
print(f"CONTENT_DIR : {CONTENT_DIR}")
print(f"ZIP_PATH    : {ZIP_PATH}")
print(f"Outputs dir : {OUTPUTS_DIR}")
print("Dataset files:")
for var_name, file_name in DATASET_FILES.items():
    print(f"- {var_name}: {CONTENT_DIR / file_name}")
print("=== P0 DONE ===")

# P1 — Data Import and Loading (OULAD CSV Sources)

In [ ]:
# ==============================================================
# P1 — Data Import and Loading (OULAD CSV Sources)
# --------------------------------------------------------------
# Purpose:
#   Load all OULAD CSV sources required by the paper pipeline and
#   expose them as notebook globals for downstream cells.
# ==============================================================

if "CONTENT_DIR" not in globals() or "DATASET_FILES" not in globals():
    raise NameError("P1 requires CONTENT_DIR and DATASET_FILES from P0.")

loaded_frames = {}
for var_name, file_name in DATASET_FILES.items():
    csv_path = CONTENT_DIR / file_name
    if not csv_path.exists():
        raise FileNotFoundError(f"P1 could not find required file: {csv_path}")
    loaded_frames[var_name] = pd.read_csv(csv_path)
    globals()[var_name] = loaded_frames[var_name]

print("=== P1 — Data Import and Loading ===")
for var_name, df in loaded_frames.items():
    print(f"{var_name:20s} shape={df.shape}")
print("\n[Columns snapshot]")
for var_name, df in loaded_frames.items():
    cols_preview = list(df.columns[:10])
    print(f"{var_name:20s} columns={cols_preview}")
print("=== P1 DONE ===")

# P2 — Unit of Analysis and Core Integration (Enrollment-Level Dataset)

In [918]:
# ==============================================================
# P2 — Unit of Analysis and Core Integration (Enrollment-Level Dataset)
# --------------------------------------------------------------
# Purpose:
#   Define the unit of analysis as an enrollment (student × module × presentation),
#   and integrate administrative unregistration timing from studentRegistration
#   into studentInfo using the shared enrollment keys.
#
# Outputs:
#   - KEYS: enrollment identifiers
#   - enrollments: one row per enrollment with final_result and date_unregistration
# ==============================================================

KEYS = ["id_student", "code_module", "code_presentation"]

# Keep only the columns needed for the enrollment-level backbone
enrollments = studentInfo[KEYS + ["final_result"]].copy()

# Merge unregistration timing (date_unregistration) from studentRegistration
enrollments = enrollments.merge(
    studentRegistration[KEYS + ["date_unregistration"]],
    on=KEYS,
    how="left"
)

# Sanity check: ensure one row per enrollment
enrollments = enrollments.drop_duplicates(subset=KEYS).reset_index(drop=True)

print("=== P2 — Enrollment-level backbone ===")
print("Rows (enrollments):", len(enrollments))
print("Unique enrollments:", enrollments[KEYS].drop_duplicates().shape[0])
print("Columns:", list(enrollments.columns))

=== P2 — Enrollment-level backbone ===
Rows (enrollments): 32593
Unique enrollments: 32593
Columns: ['id_student', 'code_module', 'code_presentation', 'final_result', 'date_unregistration']


# P2.1 — Guardrails: Prevent Duplicate Columns at the Source

In [919]:
# ==============================================================
# P2.1 — Guardrails: Prevent Duplicate Columns at the Source
# --------------------------------------------------------------
# Purpose:
#   Prevent duplicate columns from being introduced by merges/joins/selections.
#   This block provides:
#     (1) assert_no_duplicate_columns(df, name)
#     (2) safe_merge(left, right, on, how, validate, right_keep_cols, ...)
#
# Contract:
#   - This block DOES NOT "fix" duplicates silently.
#   - If duplicates appear, it FAILS FAST with a clear diagnostic,
#     so we can correct the upstream merge that created them.
#
# How to use (incrementally):
#   - After any critical merge:
#       assert_no_duplicate_columns(df, "df_name_after_merge")
#   - Prefer replacing pd.merge(...) with safe_merge(...)
# ==============================================================

import pandas as pd

def assert_no_duplicate_columns(df: pd.DataFrame, name: str) -> None:
    """
    Fail-fast if df contains duplicate column names.
    Prints a compact diagnostic to locate the source merge quickly.
    """
    cols = list(df.columns)
    dups = pd.Index(cols)[pd.Index(cols).duplicated()].unique().tolist()
    if dups:
        # Show positions of duplicated columns
        pos = {c: [i for i, cc in enumerate(cols) if cc == c] for c in dups}
        raise ValueError(
            f"[{name}][FATAL] Duplicate columns detected: {dups}\n"
            f"[{name}][DETAIL] Column index positions: {pos}\n"
            f"[{name}][HINT] A prior merge likely introduced the same column twice.\n"
            f"         Fix by restricting right-side columns in the merge."
        )

def safe_merge(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    on,
    how: str = "left",
    validate: str | None = None,
    suffixes: tuple[str, str] = ("__L", "__R"),
    right_keep_cols: list[str] | None = None,
    name: str = "safe_merge",
) -> pd.DataFrame:
    """
    Merge wrapper that prevents accidental duplicate columns.

    Parameters:
      - on: join keys
      - validate: pandas merge validate string ('1:1','m:1','1:m','m:m')
      - right_keep_cols: list of columns to keep from right INCLUDING keys.
        If provided, right is reduced to those columns before merge.
        This is the main mechanism to prevent collisions.
      - suffixes: set to explicit non-default suffixes so collisions are visible.

    Behavior:
      - Fails if duplicate columns exist in inputs or output.
      - Fails if output contains any '__L'/'__R' suffixed columns (means collision happened).
    """
    assert_no_duplicate_columns(left, f"{name}:left_input")
    assert_no_duplicate_columns(right, f"{name}:right_input")

    # Reduce right-side columns to prevent collisions
    if right_keep_cols is not None:
        missing = [c for c in right_keep_cols if c not in right.columns]
        if missing:
            raise KeyError(f"[{name}][FATAL] right_keep_cols missing in right: {missing}")
        right_use = right.loc[:, right_keep_cols].copy()
    else:
        right_use = right

    out = left.merge(
        right_use,
        on=on,
        how=how,
        validate=validate,
        suffixes=suffixes,
    )

    # Fail if suffixes appear (collision)
    bad = [c for c in out.columns if c.endswith(suffixes[0]) or c.endswith(suffixes[1])]
    if bad:
        raise ValueError(
            f"[{name}][FATAL] Merge produced suffixed columns (collision detected): {bad}\n"
            f"[{name}][HINT] Restrict right_keep_cols so you do not bring overlapping columns."
        )

    assert_no_duplicate_columns(out, f"{name}:output")
    return out

print("=== P2.1 — Guardrails loaded ===")
print("Functions available: assert_no_duplicate_columns(), safe_merge()")
print("=== P2.1 DONE ===")

=== P2.1 — Guardrails loaded ===
Functions available: assert_no_duplicate_columns(), safe_merge()
=== P2.1 DONE ===


# P2.2 — Column-List Guardrail (Prevent Duplicate Columns by Construction)

In [920]:
# ==============================================================
# P2.2 — Column-List Guardrail (Prevent Duplicate Columns by Construction)
# --------------------------------------------------------------
# Purpose:
#   Prevent duplicate columns created by *lists*, e.g. KEYS + ["code_module", ...]
#   which silently duplicates columns because KEYS already contains them.
#
# Why this matters:
#   - Pandas groupby fails with: "Grouper for 'code_module' not 1-dimensional"
#     when a DataFrame has duplicate column names.
#
# Provides:
#   - uniq_cols(cols): returns a de-duplicated list preserving order
#   - safe_select(df, cols, name): selects columns safely + fails fast if duplicates arise
#
# Contract:
#   - Use uniq_cols() anytime you build a list like KEYS + [...]
#   - Use safe_select() anytime you slice df[...] with a dynamic list
# ==============================================================

import pandas as pd

def uniq_cols(cols: list[str]) -> list[str]:
    """
    Deduplicate a column-name list while preserving first occurrence order.
    """
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

def safe_select(df: pd.DataFrame, cols: list[str], name: str = "safe_select") -> pd.DataFrame:
    """
    Select columns with:
      (1) de-duplicated list
      (2) missing-column check
      (3) post-check for duplicate columns in output
    """
    cols_u = uniq_cols(list(cols))
    missing = [c for c in cols_u if c not in df.columns]
    if missing:
        raise KeyError(f"[{name}][FATAL] Missing columns: {missing}")

    out = df.loc[:, cols_u].copy()
    assert_no_duplicate_columns(out, f"{name}:output")  # from P2.1
    return out

print("=== P2.2 — Column-list guardrail loaded ===")
print("Functions available: uniq_cols(), safe_select()")
print("=== P2.2 DONE ===")

=== P2.2 — Column-list guardrail loaded ===
Functions available: uniq_cols(), safe_select()
=== P2.2 DONE ===


# P3 — Event Definition, Censoring, and Final Time (Withdrawn as the Event)

In [921]:
# ==============================================================
# P3 — Event Definition, Censoring, and Final Time (Withdrawn)
# --------------------------------------------------------------
# SINGLE SOURCE OF TRUTH (aligned with P4):
#   event_withdrawn = 1  iff  final_result == "Withdrawn"
#                         AND date_unregistration is observed (and valid)
#
# Discrete-time (weekly) time variables:
#   t_event_week     = floor(date_unregistration / 7)   only if event_withdrawn==1
#   t_last_obs_week  = last observed VLE interaction week (>=0); if none, 0
#   t_final_week     = t_event_week if event occurred, else t_last_obs_week (right-censored)
#
# Output:
#   enrollments_eventtime  (enrollment-level table with event/time columns)
# ==============================================================

import numpy as np
import pandas as pd

INFO_RESULT_COL = "final_result"
REG_DATE_COL = "date_unregistration"
VLE_DATE_COL = "date"

# ------------------------------
# 0) Preconditions / sanity
# ------------------------------
if "enrollments" not in globals():
    raise ValueError("P3: 'enrollments' not found. Run P1/P2 first.")

if "studentVle" not in globals():
    raise ValueError("P3: 'studentVle' not found. Run P1/P2 first.")

required_cols_enr = KEYS + [INFO_RESULT_COL, REG_DATE_COL]
missing_enr = [c for c in required_cols_enr if c not in enrollments.columns]
if missing_enr:
    raise ValueError(f"P3: enrollments missing required columns: {missing_enr}")

required_cols_vle = KEYS + [VLE_DATE_COL]
missing_vle = [c for c in required_cols_vle if c not in studentVle.columns]
if missing_vle:
    raise ValueError(f"P3: studentVle missing required columns: {missing_vle}")

# Ensure 1 row per enrollment key before attaching times
dup_enr = enrollments.duplicated(subset=KEYS, keep=False)
if dup_enr.any():
    ndup = int(dup_enr.sum())
    raise ValueError(
        f"P3: duplicated enrollments detected (n={ndup}). "
        f"Fix upstream merges/dedup before computing event times."
    )

enrollments = enrollments.copy()

# ------------------------------
# 1) Clean/validate date_unregistration
# ------------------------------
# Treat negative unregistration days as invalid (not observed)
enrollments.loc[enrollments[REG_DATE_COL] < 0, REG_DATE_COL] = np.nan

# ------------------------------
# 2) Compute last observed VLE week (t_last_obs_week)
# ------------------------------
vle_tmp = studentVle[KEYS + [VLE_DATE_COL]].copy()

# Drop missing dates and negative day indices
vle_tmp = vle_tmp.dropna(subset=[VLE_DATE_COL])
vle_tmp = vle_tmp[vle_tmp[VLE_DATE_COL] >= 0]

if len(vle_tmp) == 0:
    # If there is no VLE activity after filtering, everyone is censored at week 0.
    enrollments["t_last_obs_week"] = 0
else:
    # Convert day index to discrete week index
    vle_tmp["week"] = np.floor(vle_tmp[VLE_DATE_COL] / 7.0).astype(int)
    vle_tmp.loc[vle_tmp["week"] < 0, "week"] = 0

    vle_last = (
        vle_tmp.groupby(KEYS)["week"]
        .max()
        .reset_index()
        .rename(columns={"week": "t_last_obs_week"})
    )

    # Merge VLE last-week info back to enrollments
    enrollments = enrollments.merge(vle_last, on=KEYS, how="left")

    # SAFEGUARD: ensure the column exists even if merge produced no new column (edge cases)
    if "t_last_obs_week" not in enrollments.columns:
        enrollments["t_last_obs_week"] = 0
    else:
        enrollments["t_last_obs_week"] = enrollments["t_last_obs_week"].fillna(0)

    enrollments["t_last_obs_week"] = enrollments["t_last_obs_week"].astype(int)

# ------------------------------
# 3) Define the ONLY event: Withdrawn with observed unregistration date
# ------------------------------
is_withdrawn = (
    enrollments[INFO_RESULT_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("withdrawn")
)

has_valid_unreg = enrollments[REG_DATE_COL].notna()

# Single source of truth event definition (must match P4 concept)
enrollments["event_withdrawn"] = (is_withdrawn & has_valid_unreg).astype(int)

# ------------------------------
# 4) Event week (t_event_week) — ONLY for event==1 (nullable Int64)
# ------------------------------
# IMPORTANT: np.where returns a NumPy array; convert to pandas Series before casting to "Int64".
t_event_week_arr = np.where(
    enrollments["event_withdrawn"].eq(1),
    np.floor(enrollments[REG_DATE_COL] / 7.0),
    np.nan
)
enrollments["t_event_week"] = pd.Series(t_event_week_arr, index=enrollments.index).astype("Int64")

# ------------------------------
# 5) Final week (t_final_week): event time if event, else censoring time
# ------------------------------
enrollments["t_final_week"] = np.where(
    enrollments["event_withdrawn"].eq(1),
    enrollments["t_event_week"].astype("Int64"),
    enrollments["t_last_obs_week"]
)

enrollments["t_final_week"] = pd.to_numeric(enrollments["t_final_week"], errors="coerce").fillna(0).astype(int)
enrollments.loc[enrollments["t_final_week"] < 0, "t_final_week"] = 0

# ------------------------------
# 6) Integrity checks
# ------------------------------
bad1 = enrollments[(enrollments["event_withdrawn"] == 1) & (enrollments["t_event_week"].isna())]
if len(bad1) > 0:
    raise ValueError(f"P3 integrity fail: event_withdrawn==1 but t_event_week is NA (n={len(bad1)})")

bad2 = enrollments[(enrollments["event_withdrawn"] == 0) & (enrollments["t_event_week"].notna())]
if len(bad2) > 0:
    raise ValueError(f"P3 integrity fail: t_event_week present but event_withdrawn==0 (n={len(bad2)})")

# ------------------------------
# 7) Summary prints (auditing)
# ------------------------------
print("=== P3 — Event/Censoring Summary (Aligned with P4 Single Event Definition) ===")
print("N_enrollments:", int(len(enrollments)))
print("N_unique_students:", int(enrollments["id_student"].nunique()))
print("N_events_withdrawn:", int(enrollments["event_withdrawn"].sum()))
print("Event rate:", round(float(enrollments["event_withdrawn"].mean()), 6))
print("max_t_last_obs_week:", int(enrollments["t_last_obs_week"].max()) if "t_last_obs_week" in enrollments.columns else 0)
print("max_t_final_week:", int(enrollments["t_final_week"].max()))

# Diagnostic only (NOT used as event): Withdrawn label but missing unregistration date
n_withdrawn_missing_unreg = int((is_withdrawn & ~has_valid_unreg).sum())
print("N_withdrawn_missing_unreg (censored under P4 definition):", n_withdrawn_missing_unreg)

# ------------------------------
# 8) Output
# ------------------------------
enrollments_eventtime = enrollments.copy()
print("P3 OK: enrollments_eventtime created. Columns =", len(enrollments_eventtime.columns))

=== P3 — Event/Censoring Summary (Aligned with P4 Single Event Definition) ===
N_enrollments: 32593
N_unique_students: 28785
N_events_withdrawn: 7387
Event rate: 0.226644
max_t_last_obs_week: 38
max_t_final_week: 63
N_withdrawn_missing_unreg (censored under P4 definition): 2769
P3 OK: enrollments_eventtime created. Columns = 9


# P4 — Person-Period Weekly Frame (Discrete-Time Survival Dataset)

In [922]:
# ==============================================================
# P4 — Person-Period Weekly Frame (Discrete-Time Survival Dataset)
# --------------------------------------------------------------
# IMPORTANT: P4 MUST NOT redefine the event. It consumes P3 output:
#   enrollments_eventtime with:
#     - event_withdrawn (single source of truth)
#     - t_event_week (only when event_withdrawn==1)
#     - t_last_obs_week
#     - t_final_week (event time if event, else censoring time)
#
# Output:
#   person_period_weekly  (one row per enrollment-week)
# ==============================================================

import numpy as np
import pandas as pd

# ------------------------------
# 0) Preconditions
# ------------------------------
if "enrollments_eventtime" not in globals():
    raise ValueError("P4: enrollments_eventtime not found. Run P3 first.")

required_cols = KEYS + [
    "event_withdrawn",
    "t_event_week",
    "t_last_obs_week",
    "t_final_week",
    "final_result",
    "date_unregistration",
]
missing = [c for c in required_cols if c not in enrollments_eventtime.columns]
if missing:
    raise ValueError(f"P4: missing required columns from P3: {missing}")

base = enrollments_eventtime[required_cols].copy()

# Ensure one row per enrollment key (avoid accidental duplication upstream)
dup = base.duplicated(subset=KEYS, keep=False)
if dup.any():
    ndup = int(dup.sum())
    raise ValueError(
        f"P4: duplicated enrollments detected in P3 output (n={ndup}). "
        f"Fix upstream merges/dedup before expanding."
    )

# Coerce types defensively
base["event_withdrawn"] = base["event_withdrawn"].astype(int)

# Integrity: event <-> t_event_week consistency (must already hold from P3)
bad1 = base[(base["event_withdrawn"] == 1) & (base["t_event_week"].isna())]
if len(bad1) > 0:
    raise ValueError(f"P4: event_withdrawn==1 but t_event_week is NA (n={len(bad1)})")

bad2 = base[(base["event_withdrawn"] == 0) & (base["t_event_week"].notna())]
if len(bad2) > 0:
    raise ValueError(f"P4: t_event_week present but event_withdrawn==0 (n={len(bad2)})")

# t_final_week must be non-negative integer
base["t_final_week"] = pd.to_numeric(base["t_final_week"], errors="coerce").fillna(0).astype(int)
if (base["t_final_week"] < 0).any():
    raise ValueError("P4: negative t_final_week detected (should not happen).")

# ------------------------------
# 1) Expand each enrollment into weeks 0..t_final_week
# ------------------------------
repeats = (base["t_final_week"].to_numpy() + 1).astype(int)

if (repeats <= 0).any():
    # repeats=0 would mean t_final_week=-1 (should not happen after checks), but keep explicit
    raise ValueError("P4: non-positive repeats detected; check t_final_week logic.")

idx = np.repeat(np.arange(len(base)), repeats)

person_period_weekly = base.iloc[idx].reset_index(drop=True)

# Create week sequence efficiently (per enrollment)
person_period_weekly["week"] = np.concatenate([np.arange(r, dtype=int) for r in repeats])

# ------------------------------
# 2) Create discrete-time event indicator in person-period frame
# ------------------------------
# event=1 only at week == t_event_week for enrollments with event_withdrawn==1
person_period_weekly["event"] = 0

# Use Int64 to compare safely with NAs
t_event = person_period_weekly["t_event_week"].astype("Int64")

mask_event = (person_period_weekly["event_withdrawn"] == 1) & (t_event.notna()) & (person_period_weekly["week"] == t_event)
person_period_weekly.loc[mask_event, "event"] = 1

# ------------------------------
# 3) Hard integrity checks (must pass)
# ------------------------------
# (a) Exactly one event row per event enrollment
event_counts = (
    person_period_weekly.groupby(KEYS, as_index=False)["event"]
    .sum()
    .rename(columns={"event": "n_events"})
)

bad_multi = event_counts[event_counts["n_events"] > 1]
if len(bad_multi) > 0:
    raise ValueError(f"P4 integrity fail: multiple events per enrollment (n={len(bad_multi)})")

# Enrollments that should have an event must have exactly one event row
should_have = base[base["event_withdrawn"] == 1][KEYS].merge(event_counts, on=KEYS, how="left")
should_have["n_events"] = should_have["n_events"].fillna(0).astype(int)
bad_zero = should_have[should_have["n_events"] != 1]
if len(bad_zero) > 0:
    raise ValueError(f"P4 integrity fail: event enrollments missing event row (n={len(bad_zero)})")

# (b) No rows after the event week for event enrollments
# Because we expand only to t_final_week and t_final_week==t_event_week for events, this should be impossible.
# Still assert explicitly.
post_event = person_period_weekly[
    (person_period_weekly["event_withdrawn"] == 1)
    & (t_event.notna())
    & (person_period_weekly["week"] > t_event)
]
if len(post_event) > 0:
    raise ValueError(f"P4 integrity fail: rows after event detected (n={len(post_event)})")

# ------------------------------
# 4) Summary / audit table
# ------------------------------
summary_p4 = pd.DataFrame([{
    "N_enrollments": int(len(base)),
    "N_unique_students": int(base["id_student"].nunique()),
    "N_events_withdrawn": int(base["event_withdrawn"].sum()),
    "Event_rate": float(base["event_withdrawn"].mean()) if len(base) else 0.0,
    "N_person_period_rows": int(len(person_period_weekly)),
    "max_t_last_obs_week": int(base["t_last_obs_week"].max()) if len(base) else 0,
    "max_t_final_week": int(base["t_final_week"].max()) if len(base) else 0,
}])

print("=== P4 SUMMARY (Single Event Definition Preserved) ===")
display(summary_p4)

# person_period_weekly is now your discrete-time survival dataset

=== P4 SUMMARY (Single Event Definition Preserved) ===


,N_enrollments,N_unique_students,N_events_withdrawn,Event_rate,N_person_period_rows,max_t_last_obs_week,max_t_final_week
0,32593,28785,7387,0.226644,775295,38,63


# P5 — Feature Construction: Time-Varying \(Z_i(t)\) and Static \(X_i\)

In [923]:
# ==============================================================
# P5 — Feature Construction (Zi(t), Xi)
# --------------------------------------------------------------
# Purpose:
#   Build covariates for discrete-time hazard modeling:
#   - Time-varying features Zi(t): derived from studentVle, aggregated by enrollment-week.
#   - Static features Xi: derived from studentInfo (enrollment-level attributes).
#
# Minimal design (academically defensible and easy to extend):
#   Zi(t): total_clicks_it
#   Xi:   studied_credits, num_of_prev_attempts, (optional) demographics
#
# Output:
#   - pp_features: person_period_weekly enriched with Zi(t) and Xi
# ==============================================================

# ------------------------------
# 1) Time-varying Zi(t): weekly VLE activity
# ------------------------------
# We aggregate 'sum_click' by enrollment-week (week = floor(date/7)).
vle_weekly = studentVle[KEYS + ["date", "sum_click"]].copy()
vle_weekly["week"] = np.floor(vle_weekly["date"] / 7).astype(int)
vle_weekly["week"] = vle_weekly["week"].clip(lower=0)

Z_weekly = (
    vle_weekly.groupby(KEYS + ["week"])["sum_click"]
    .sum()
    .reset_index()
    .rename(columns={"sum_click": "total_clicks"})
)

# ------------------------------
# 2) Static Xi: baseline enrollment attributes
# ------------------------------
# Keep this minimal for now; you can add demographics later.
X_cols = ["studied_credits", "num_of_prev_attempts", "highest_education", "age_band", "gender"]
X_cols = [c for c in X_cols if c in studentInfo.columns]

X_static = studentInfo[KEYS + X_cols].drop_duplicates(subset=KEYS).copy()

# ------------------------------
# 3) Merge into person-period
# ------------------------------
pp_features = person_period_weekly.merge(Z_weekly, on=KEYS + ["week"], how="left")
pp_features["total_clicks"] = pp_features["total_clicks"].fillna(0)

pp_features = pp_features.merge(X_static, on=KEYS, how="left")

print("=== P5 — Feature Merge Summary ===")
print("pp_features rows:", len(pp_features))
print("Missing total_clicks (should be 0):", int(pp_features["total_clicks"].isna().sum()))
print("Example columns:", list(pp_features.columns)[:20])

=== P5 — Feature Merge Summary ===
pp_features rows: 775295
Missing total_clicks (should be 0): 0
Example columns: ['id_student', 'code_module', 'code_presentation', 'event_withdrawn', 't_event_week', 't_last_obs_week', 't_final_week', 'final_result', 'date_unregistration', 'week', 'event', 'total_clicks', 'studied_credits', 'num_of_prev_attempts', 'highest_education', 'age_band', 'gender']


# P5.1 — Recency and Streak (Time-Varying Features, PRE-SPLIT)


In [924]:
# ==============================================================
# P5.1 — Recency and Streak (Time-Varying Features, PRE-SPLIT)
# --------------------------------------------------------------
# Purpose:
#   Materialize recency/streak on pp_features BEFORE any split/model step.
#   This avoids the "recency/streak missing in pp_train" failures in P7.2+.
#
# Requires:
#   - pp_features (from P5)
#   - KEYS
#   - total_clicks
#
# Produces:
#   - pp_features['activity_flag','recency','streak']
# ==============================================================

import numpy as np
import pandas as pd

required = ["pp_features", "KEYS"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P5.1 missing required objects: {missing}")

if "total_clicks" not in pp_features.columns:
    raise KeyError("P5.1 requires 'total_clicks' in pp_features (from P5 merge).")

pp_features = (
    pp_features.sort_values(KEYS + ["week"])
    .reset_index(drop=True)
    .copy()
)

pp_features["activity_flag"] = (pd.to_numeric(pp_features["total_clicks"], errors="coerce").fillna(0.0) > 0).astype(int)

recency_arr = np.zeros(len(pp_features), dtype=int)
streak_arr  = np.zeros(len(pp_features), dtype=int)

for _, g in pp_features.groupby(KEYS, sort=False):
    idx = g.index.to_numpy()
    a = g["activity_flag"].to_numpy(dtype=int)

    r = np.zeros(len(a), dtype=int)
    s = np.zeros(len(a), dtype=int)

    for i in range(len(a)):
        if i == 0:
            r[i] = 0 if a[i] == 1 else 1
            s[i] = 1 if a[i] == 1 else 0
        else:
            if a[i] == 1:
                r[i] = 0
                s[i] = s[i - 1] + 1
            else:
                r[i] = r[i - 1] + 1
                s[i] = 0

    recency_arr[idx] = r
    streak_arr[idx]  = s

pp_features["recency"] = recency_arr
pp_features["streak"]  = streak_arr

# hard sanity checks
bad1 = pp_features[(pp_features["activity_flag"] == 1) & (pp_features["recency"] != 0)]
bad2 = pp_features[(pp_features["activity_flag"] == 1) & (pp_features["streak"] < 1)]

print("=== P5.1 — Recency/Streak materialized (PRE-SPLIT) ===")
print("pp_features rows:", len(pp_features))
print("recency min/med/max:", int(pp_features["recency"].min()), float(pp_features["recency"].median()), int(pp_features["recency"].max()))
print("streak  min/med/max:", int(pp_features["streak"].min()), float(pp_features["streak"].median()), int(pp_features["streak"].max()))
print("Sanity: activity_flag=1 but recency!=0 rows:", len(bad1))
print("Sanity: activity_flag=1 but streak<1 rows:", len(bad2))
print("=== P5.1 DONE ===")

=== P5.1 — Recency/Streak materialized (PRE-SPLIT) ===
pp_features rows: 775295
recency min/med/max: 0 0.0 42
streak  min/med/max: 0 2.0 39
Sanity: activity_flag=1 but recency!=0 rows: 0
Sanity: activity_flag=1 but streak<1 rows: 0
=== P5.1 DONE ===


# P5.2 — Weekly Assessment Submissions (Zi(t): submitted_this_week)


In [925]:
# ==============================================================
# P5.2-v3 — Weekly Assessment Submissions (Zi(t): submitted_this_week) [ROBUST MERGE]
# --------------------------------------------------------------
# Purpose:
#   Create a weekly time-varying covariate that captures whether an enrollment
#   submitted at least one assessment in week t:
#
#       submitted_this_week_i(t) = 1 if any assessment submission exists in week t
#                                 0 otherwise
#
# Robustness fix (v3):
#   - Prevent merge suffix collisions if pp_features already has a prior
#     submitted_this_week column from previous runs.
#   - Drop any existing submitted_this_week / *_x / *_y before merge.
#
# Data sources (OULAD):
#   - studentAssessment.csv : submissions by student and assessment
#   - assessments.csv       : maps id_assessment -> (code_module, code_presentation)
#
# Requires:
#   - KEYS
#   - pp_features
#   - studentAssessment, assessments
#
# Output:
#   - Updates pp_features with column: submitted_this_week (int in {0,1})
#   - Prints diagnostics to paste back into chat
# ==============================================================

import numpy as np
import pandas as pd

print("=== P5.2-v3 — Building submitted_this_week (Zi(t)) ===")

# ------------------------------
# 0) Preconditions (fail fast)
# ------------------------------
required_objs = ["KEYS", "pp_features", "studentAssessment", "assessments"]
missing_objs = [o for o in required_objs if o not in globals()]
if missing_objs:
    raise NameError(f"P5.2-v3 missing required objects: {missing_objs}")

sa_required = ["id_student", "id_assessment", "date_submitted"]
as_required = ["id_assessment", "code_module", "code_presentation"]

sa_missing = [c for c in sa_required if c not in studentAssessment.columns]
as_missing = [c for c in as_required if c not in assessments.columns]

if sa_missing:
    raise KeyError(
        "P5.2-v3: studentAssessment missing required columns: "
        f"{sa_missing}\nAvailable: {list(studentAssessment.columns)}"
    )
if as_missing:
    raise KeyError(
        "P5.2-v3: assessments missing required columns: "
        f"{as_missing}\nAvailable: {list(assessments.columns)}"
    )

pp_missing = [c for c in (list(KEYS) + ["week"]) if c not in pp_features.columns]
if pp_missing:
    raise KeyError(
        "P5.2-v3: pp_features missing required columns: "
        f"{pp_missing}\nAvailable: {list(pp_features.columns)}"
    )

# ------------------------------
# 1) Drop any pre-existing submitted_this_week columns (avoid merge suffixes)
# ------------------------------
to_drop = [c for c in pp_features.columns if c == "submitted_this_week" or c.startswith("submitted_this_week_")]
dropped = []
if to_drop:
    pp_features = pp_features.drop(columns=to_drop).copy()
    dropped = to_drop

print("[P5.2-v3][INFO] Dropped pre-existing columns:", dropped if dropped else "NONE")

# ------------------------------
# 2) Build submission records with enrollment keys
# ------------------------------
sa = studentAssessment[["id_student", "id_assessment", "date_submitted"]].copy()
asmt = assessments[["id_assessment", "code_module", "code_presentation"]].copy()

subm = sa.merge(asmt, on="id_assessment", how="left", validate="m:1")

n_missing_keys = int(subm["code_module"].isna().sum() + subm["code_presentation"].isna().sum())
if n_missing_keys > 0:
    print(f"[P5.2-v3][WARN] Missing module/presentation after merge: {n_missing_keys}")

# ------------------------------
# 3) Compute submission week (only valid submissions)
# ------------------------------
subm["date_submitted"] = pd.to_numeric(subm["date_submitted"], errors="coerce")
valid = subm["date_submitted"].notna() & (subm["date_submitted"] >= 0)

subm_valid = subm.loc[
    valid, ["id_student", "code_module", "code_presentation", "date_submitted"]
].copy()

subm_valid["week"] = np.floor(subm_valid["date_submitted"] / 7.0).astype(int)
subm_valid.loc[subm_valid["week"] < 0, "week"] = 0

# ------------------------------
# 4) Aggregate to enrollment-week: binary submitted_this_week
# ------------------------------
weekly_subm = (
    subm_valid
    .assign(submitted_this_week=1)
    .groupby(KEYS + ["week"], as_index=False)["submitted_this_week"]
    .max()
)

if "submitted_this_week" not in weekly_subm.columns:
    raise RuntimeError("P5.2-v3: weekly_subm did not contain 'submitted_this_week' after aggregation (unexpected).")

# ------------------------------
# 5) Merge into pp_features
# ------------------------------
pp_features = pp_features.merge(
    weekly_subm,
    on=KEYS + ["week"],
    how="left",
    validate="m:1"
)

# After merge, the column MUST exist
if "submitted_this_week" not in pp_features.columns:
    raise KeyError(
        "P5.2-v3: merge did not create 'submitted_this_week'. "
        f"Columns now include: {[c for c in pp_features.columns if 'submitted_this_week' in c]}"
    )

pp_features["submitted_this_week"] = (
    pd.to_numeric(pp_features["submitted_this_week"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# ------------------------------
# 6) REQUIRED TEXT OUTPUTS (paste back)
# ------------------------------
rate = float(pp_features["submitted_this_week"].mean())
n1 = int(pp_features["submitted_this_week"].sum())
nrows = int(len(pp_features))

print("\n=== P5.2-v3 — Diagnostics ===")
print("[pp_features]")
print("rows:", nrows)
print("submitted_this_week sum:", n1)
print("submitted_this_week mean:", rate)

wk_tab = (
    pp_features
    .groupby("week", as_index=False)
    .agg(
        n_rows=("submitted_this_week", "size"),
        n_submitted=("submitted_this_week", "sum"),
        rate=("submitted_this_week", "mean"),
    )
    .sort_values("week")
)

print("\n[By-week snapshot (head)]")
print(wk_tab.head(10).to_string(index=False))

print("\n[By-week snapshot (tail)]")
print(wk_tab.tail(10).to_string(index=False))

print("\n[Column check]")
print("submitted_this_week in pp_features:", "submitted_this_week" in pp_features.columns)

print("=== P5.2-v3 DONE ===")

=== P5.2-v3 — Building submitted_this_week (Zi(t)) ===
[P5.2-v3][INFO] Dropped pre-existing columns: NONE

=== P5.2-v3 — Diagnostics ===
[pp_features]
rows: 775295
submitted_this_week sum: 148617
submitted_this_week mean: 0.19169090475238457

[By-week snapshot (head)]
 week  n_rows  n_submitted     rate
    0   32593          659 0.020219
    1   28633         2308 0.080606
    2   27396        10405 0.379800
    3   26956         7974 0.295815
    4   26363         4858 0.184273
    5   25843         3280 0.126920
    6   25459         5881 0.230999
    7   25010         8733 0.349180
    8   24554         5912 0.240775
    9   24160         5182 0.214487

[By-week snapshot (tail)]
 week  n_rows  n_submitted  rate
   54       1            0   0.0
   55       1            0   0.0
   56       1            0   0.0
   57       1            0   0.0
   58       1            0   0.0
   59       1            0   0.0
   60       1            0   0.0
   61       1            0   0.0
   62      

# P6 — Temporal Holdout by Event Time (cutoff_event_week)

In [927]:
# ==============================================================
# P7-v3 — Discrete-Time Hazard Model (NO leakage) + submitted_this_week
# --------------------------------------------------------------
# Purpose:
#   Fit the discrete-time hazard model on the person-period frame using a
#   leakage-safe feature set, now including the time-varying covariate:
#       submitted_this_week (binary Zi(t))
#
# Key constraints:
#   - DO NOT use event/t_event_week/t_final_week/t_last_obs_week/date_unregistration
#   - Keep modeling comparable to prior versions (logistic baseline)
#
# Requires:
#   - pp_train, pp_test
#   - KEYS
#   - Columns expected in pp_*: week, total_clicks, recency, streak,
#     studied_credits, num_of_prev_attempts, submitted_this_week,
#     code_module, code_presentation, highest_education, age_band, gender
#
# Outputs:
#   - hazard_model (calibrated)
#   - pp_train['hazard_hat'], pp_test['hazard_hat']
#   - Prints diagnostics for audit
# ==============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

print("=== P7-v3 — Temporal-Safe Discrete-Time Hazard Model (+ submitted_this_week) ===")

# ------------------------------
# 0) Preconditions
# ------------------------------
required_objs = ["pp_train", "pp_test", "KEYS"]
missing = [o for o in required_objs if o not in globals()]
if missing:
    raise NameError(f"P7-v3 missing required objects: {missing}")

# Explicitly require submitted_this_week now
for df_name in ["pp_train", "pp_test"]:
    df = globals()[df_name]
    if "submitted_this_week" not in df.columns:
        raise KeyError(f"P7-v3 requires 'submitted_this_week' in {df_name}. Run P5.2-v3 first.")

# ------------------------------
# 1) Define leakage-safe feature set (assert columns exist)
# ------------------------------
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration", "final_result",
    "hazard_hat", "S_hat", "S0_hat", "S_shock_hat", "S_mech_hat",
}

# Numeric (time-varying + static) — include submitted_this_week
numeric_features = [
    "total_clicks",
    "studied_credits",
    "num_of_prev_attempts",
    "recency",
    "streak",
    "submitted_this_week",
]

# Categorical (baseline attrs + course/run id + week as flexible baseline)
categorical_features = [
    "week",
    "code_module",
    "code_presentation",
    "highest_education",
    "age_band",
    "gender",
]

def _check_cols(df: pd.DataFrame, cols: list, df_name: str):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"P7-v3: {df_name} missing required columns: {miss}")

_check_cols(pp_train, numeric_features + categorical_features + ["event"], "pp_train")
_check_cols(pp_test,  numeric_features + categorical_features + ["event"], "pp_test")

# ------------------------------
# 2) Build preprocessing pipeline
# ------------------------------
pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=True, with_std=True), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

base = LogisticRegression(
    solver="liblinear",
    max_iter=4000,
    class_weight="balanced",
    random_state=42,
)

pipe = Pipeline([("pre", pre), ("lr", base)])

# ------------------------------
# 3) Fit + predict (GROUPED CV calibration at enrollment level)
# ------------------------------
X_train = pp_train[numeric_features + categorical_features]
y_train = pp_train["event"].astype(int).to_numpy()

X_test = pp_test[numeric_features + categorical_features]
y_test = pp_test["event"].astype(int).to_numpy()

# Grouped splits (enrollment-level) to avoid leakage across the same student-module-run
groups_train = (
    pp_train[KEYS]
    .astype(str)
    .agg("|".join, axis=1)
    .to_numpy()
)
gkf = GroupKFold(n_splits=5)
cv_splits = list(gkf.split(X_train, y_train, groups=groups_train))

hazard_model = CalibratedClassifierCV(
    estimator=pipe,
    method="sigmoid",
    cv=cv_splits,
    ensemble=False,
)

hazard_model.fit(X_train, y_train)

p_train = hazard_model.predict_proba(X_train)[:, 1].astype(float)
p_test  = hazard_model.predict_proba(X_test)[:, 1].astype(float)

# Clip for stability downstream
p_train = np.clip(p_train, 1e-12, 1 - 1e-12)
p_test  = np.clip(p_test, 1e-12, 1 - 1e-12)

pp_train["hazard_hat"] = p_train
pp_test["hazard_hat"] = p_test

auc_train = float(roc_auc_score(y_train, p_train))
auc_test  = float(roc_auc_score(y_test,  p_test))

# ------------------------------
# 4) REQUIRED TEXT OUTPUTS
# ------------------------------
print("[AUC row-level]")
print("AUC_train:", auc_train)
print("AUC_test :", auc_test)

print("\n[Hazard range]")
print("train min/max:", float(pp_train["hazard_hat"].min()), float(pp_train["hazard_hat"].max()))
print("test  min/max:", float(pp_test["hazard_hat"].min()),  float(pp_test["hazard_hat"].max()))

print("\n[submitted_this_week prevalence]")
print("train mean:", float(pp_train["submitted_this_week"].mean()), " sum:", int(pp_train["submitted_this_week"].sum()))
print("test  mean:", float(pp_test["submitted_this_week"].mean()),  " sum:", int(pp_test["submitted_this_week"].sum()))

print("\n[Feature lists used]")
print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)

print("=== P7-v3 DONE ===")

=== P7-v3 — Temporal-Safe Discrete-Time Hazard Model (+ submitted_this_week) ===
[AUC row-level]
AUC_train: 0.8358494389616812
AUC_test : 0.839573388360638

[Hazard range]
train min/max: 1e-12 0.9184779140515784
test  min/max: 1e-12 0.6796595123758901

[submitted_this_week prevalence]
train mean: 0.19158632326231675  sum: 104008
test  mean: 0.19193518546405813  sum: 44609

[Feature lists used]
numeric_features: ['total_clicks', 'studied_credits', 'num_of_prev_attempts', 'recency', 'streak', 'submitted_this_week']
categorical_features: ['week', 'code_module', 'code_presentation', 'highest_education', 'age_band', 'gender']
=== P7-v3 DONE ===


In [929]:
# ==============================================================
# P7.2-v3 — Conditional IPCW Discrete-Time Hazard Model (G(t|X_t))
# --------------------------------------------------------------
# Changes vs v2:
#   - Estimate censoring survival conditionally: train a discrete-time censoring model
#     on TRAIN (person-period), then compute per-row G_hat_cens(t|X_t).
#   - Attach IPCW weights: ipcw_w = 1 / max(G_hat_cens, g_min_ipcw).
#   - Refit hazard_model_ipcw using conditional weights (no change to architecture).
#   - Export per-week G summaries + figure for audit.
#
# Outputs:
#   - pp_train/pp_test columns: G_hat_cens, ipcw_w, hazard_hat_ipcw_raw
#   - hazard_model_ipcw (fitted), cens_model (fitted)
#   - Exports: outputs_v2/tables/table_ipcw_censoring_G_train.csv,
#              outputs_v2/figures/fig_ipcw_censoring_G_train.png
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# ------------------------------
# 0) Preconditions
# ------------------------------
req = ["KEYS", "enrollments_eventtime", "enroll_train_keys", "enroll_test_keys", "pp_train", "pp_test"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"P7.2-v3 missing required objects: {missing}")

for df_name in ["pp_train", "pp_test"]:
    df = globals()[df_name]
    for c in ["event", "week"]:
        if c not in df.columns:
            raise KeyError(f"P7.2-v3 requires '{c}' in {df_name}")

need_cols_enr = KEYS + ["event_withdrawn", "t_final_week"]
miss_cols = [c for c in need_cols_enr if c not in enrollments_eventtime.columns]
if miss_cols:
    raise KeyError(f"P7.2-v3 missing columns in enrollments_eventtime: {miss_cols}")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

g_min_ipcw = 0.05
EPS = 1e-12

# ------------------------------
# 1) Build censoring person-period frames (TRAIN/TEST)
# ------------------------------
use_cols = KEYS + ["event_withdrawn", "t_final_week"]

enr_train = (
    enrollments_eventtime.merge(enroll_train_keys, on=KEYS, how="inner")
    .loc[:, use_cols]
    .drop_duplicates(subset=KEYS)
    .copy()
)

enr_test = (
    enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner")
    .loc[:, use_cols]
    .drop_duplicates(subset=KEYS)
    .copy()
)

def _prep_enr(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["event_withdrawn"] = pd.to_numeric(out["event_withdrawn"], errors="raise").astype(int)
    out["t_final_week"] = pd.to_numeric(out["t_final_week"], errors="coerce").fillna(0).astype(int)
    out.loc[out["t_final_week"] < 0, "t_final_week"] = 0
    return out

enr_train = _prep_enr(enr_train)
enr_test = _prep_enr(enr_test)

def _resolve_col(df: pd.DataFrame, base: str) -> pd.Series:
    """Return column by name, tolerating merge suffixes; prefer right-merge (_y) first."""
    if base in df.columns:
        return df[base]
    yname = base + "_y"
    xname = base + "_x"
    if yname in df.columns:
        return df[yname]
    if xname in df.columns:
        return df[xname]
    cand = [c for c in df.columns if c.startswith(base + "_")]
    if len(cand) >= 1:
        return df[cand[0]]
    raise KeyError(f"P7.2-v3: column '{base}' missing after merge; columns={list(df.columns)}")

def build_cens_frame(pp: pd.DataFrame, enr: pd.DataFrame) -> pd.DataFrame:
    df = pp.merge(enr, on=KEYS, how="left")
    df["week"] = pd.to_numeric(df["week"], errors="raise").astype(int)
    df["t_final_week"] = pd.to_numeric(_resolve_col(df, "t_final_week"), errors="raise").astype(int)
    df = df.loc[df["week"] <= df["t_final_week"]].copy()
    df["event_withdrawn"] = pd.to_numeric(_resolve_col(df, "event_withdrawn"), errors="raise").astype(int)
    df["cens_event"] = ((df["event_withdrawn"] == 0) & (df["week"] == df["t_final_week"])).astype(int)
    return df

pp_cens_train = build_cens_frame(pp_train, enr_train)
pp_cens_test = build_cens_frame(pp_test, enr_test)

# ------------------------------
# 2) Censoring model G(t|X_t)
# ------------------------------
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration", "hazard_hat", "hazard_hat_ipcw_raw", "ipcw_w",
}

candidate_numeric_cens = ["total_clicks", "studied_credits", "num_of_prev_attempts", "recency", "streak", "submitted_this_week"]
candidate_categorical_cens = ["week", "code_module", "code_presentation", "highest_education", "age_band", "gender"]

num_cols_cens = [c for c in candidate_numeric_cens if c in pp_cens_train.columns and c not in FORBIDDEN]
cat_cols_cens = [c for c in candidate_categorical_cens if c in pp_cens_train.columns and c not in FORBIDDEN]

if len(num_cols_cens) + len(cat_cols_cens) == 0:
    raise ValueError("P7.2-v3: no usable censoring features after leakage filter")

num_pipe_cens = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipe_cens = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

pre_cens = ColumnTransformer(
    transformers=[
        ("num", num_pipe_cens, num_cols_cens),
        ("cat", cat_pipe_cens, cat_cols_cens),
    ],
    remainder="drop",
)

cens_lr = LogisticRegression(
    solver="saga",
    C=1.0,
    max_iter=8000,
    tol=1e-3,
    class_weight="balanced",
    random_state=42,
)

cens_model = Pipeline([("pre", pre_cens), ("lr", cens_lr)])

X_cens_tr = pp_cens_train[num_cols_cens + cat_cols_cens]
y_cens_tr = pp_cens_train["cens_event"].astype(int).to_numpy()

cens_model.fit(X_cens_tr, y_cens_tr)

pp_cens_train["cens_hazard_hat"] = np.clip(cens_model.predict_proba(X_cens_tr)[:, 1].astype(float), EPS, 1 - EPS)
pp_cens_test["cens_hazard_hat"] = np.clip(
    cens_model.predict_proba(pp_cens_test[num_cols_cens + cat_cols_cens])[:, 1].astype(float),
    EPS,
    1 - EPS,
)

# ------------------------------
# 3) Compute conditional G_hat_cens and ipcw_w
# ------------------------------

def add_G_hat(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(KEYS + ["week"]).copy()
    df["log1m_cens"] = np.log1p(-df["cens_hazard_hat"])
    df["logG_hat"] = df["log1m_cens"].groupby([df[k] for k in KEYS]).cumsum()
    df["G_hat_cens"] = np.exp(df["logG_hat"])
    df["ipcw_w"] = (1.0 / np.maximum(df["G_hat_cens"], g_min_ipcw)).astype(float)
    return df

pp_cens_train = add_G_hat(pp_cens_train)
pp_cens_test = add_G_hat(pp_cens_test)

# merge back into pp_train/pp_test
pp_train = pp_train.drop(columns=["G_hat_cens", "ipcw_w"], errors="ignore")
pp_test = pp_test.drop(columns=["G_hat_cens", "ipcw_w"], errors="ignore")

pp_train = pp_train.merge(
    pp_cens_train[KEYS + ["week", "G_hat_cens", "ipcw_w"]], on=KEYS + ["week"], how="left", validate="m:1"
)
pp_test = pp_test.merge(
    pp_cens_test[KEYS + ["week", "G_hat_cens", "ipcw_w"]], on=KEYS + ["week"], how="left", validate="m:1"
)

if pp_train[["G_hat_cens", "ipcw_w"]].isna().any().any():
    raise ValueError("P7.2-v3: missing G_hat_cens or ipcw_w after merge into pp_train")
if pp_test[["G_hat_cens", "ipcw_w"]].isna().any().any():
    raise ValueError("P7.2-v3: missing G_hat_cens or ipcw_w after merge into pp_test")

# ------------------------------
# 4) Summaries + export G table/figure (TRAIN)
# ------------------------------

dfG = (
    pp_cens_train
    .groupby("week")
    .agg(
        G_hat_mean=("G_hat_cens", "mean"),
        G_hat_median=("G_hat_cens", "median"),
        G_hat_p10=("G_hat_cens", lambda x: float(np.quantile(x, 0.10))),
        G_hat_p90=("G_hat_cens", lambda x: float(np.quantile(x, 0.90))),
    )
    .reset_index()
)

pG = os.path.join(TABLE_DIR, "table_ipcw_censoring_G_train.csv")
dfG.to_csv(pG, index=False)

pGfig = os.path.join(FIG_DIR, "fig_ipcw_censoring_G_train.png")
plt.figure()
plt.plot(dfG["week"], dfG["G_hat_mean"], marker="o", label="mean G_hat_cens")
plt.plot(dfG["week"], dfG["G_hat_median"], linestyle="--", label="median")
plt.axhline(g_min_ipcw, linestyle=":", color="red", label="g_min_ipcw")
plt.xlabel("Week")
plt.ylabel("G_hat_cens (TRAIN)")
plt.legend()
plt.tight_layout()
plt.savefig(pGfig, dpi=200)
plt.close()

# ------------------------------
# 5) Hazard model with conditional IPCW weights
# ------------------------------
FORBIDDEN_HZ = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration", "final_result",
    "hazard_hat", "gb_hazard_hat", "hazard_hat_ipcw_raw",
    "S_hat", "logS_hat", "log1m_h",
    "S0_hat", "S_shock_hat", "S_mech_hat",
    "ipcw_w", "G_hat_cens",
}

candidate_numeric = ["total_clicks", "studied_credits", "num_of_prev_attempts", "recency", "streak", "submitted_this_week"]
candidate_categorical = ["week", "code_module", "code_presentation", "highest_education", "age_band", "gender"]

num_cols = [c for c in candidate_numeric if c in pp_train.columns and c not in FORBIDDEN_HZ]
cat_cols = [c for c in candidate_categorical if c in pp_train.columns and c not in FORBIDDEN_HZ]

if len(num_cols) + len(cat_cols) == 0:
    raise ValueError("P7.2-v3: no usable hazard features after leakage filter")

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

pre = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

lr = LogisticRegression(
    solver="saga",
    C=2.0,
    max_iter=12000,
    tol=1e-3,
    random_state=42,
)

hazard_model_ipcw = Pipeline(steps=[("pre", pre), ("lr", lr)])

Xtr = pp_train[num_cols + cat_cols]
ytr = pp_train["event"].astype(int).to_numpy()
wtr = pp_train["ipcw_w"].astype(float).to_numpy()

Xte = pp_test[num_cols + cat_cols]
yte = pp_test["event"].astype(int).to_numpy()

hazard_model_ipcw.fit(Xtr, ytr, lr__sample_weight=wtr)

p_tr = np.clip(hazard_model_ipcw.predict_proba(Xtr)[:, 1].astype(float), EPS, 1 - EPS)
p_te = np.clip(hazard_model_ipcw.predict_proba(Xte)[:, 1].astype(float), EPS, 1 - EPS)

pp_train["hazard_hat_ipcw_raw"] = p_tr
pp_test["hazard_hat_ipcw_raw"] = p_te

aoc = lambda y, p: float(roc_auc_score(y, p))
auc_tr = aoc(ytr, p_tr)
auc_te = aoc(yte, p_te)

# Convergence audit
try:
    n_iter = int(hazard_model_ipcw.named_steps["lr"].n_iter_[0])
except Exception:
    n_iter = None

# ------------------------------
# 6) REQUIRED TEXT OUTPUTS
# ------------------------------
print("=== P7.2-v3 — Conditional IPCW-weighted hazard model ===")
print("[TRAIN censoring model]")
print("g_min_ipcw :", g_min_ipcw)
print("censoring features -> num:", num_cols_cens)
print("censoring features -> cat:", cat_cols_cens)
print("cens_event rate (train):", float(pp_cens_train["cens_event"].mean()))

print("\n[Row weights]")
print("ipcw_w train min/median/max:", float(np.min(wtr)), float(np.median(wtr)), float(np.max(wtr)))

print("\n[Hazard feature set]")
print("numeric:", num_cols)
print("categorical:", cat_cols)

print("\n[Convergence]")
print("n_iter_ (saga):", n_iter, "| max_iter:", lr.max_iter, "| tol:", lr.tol, "| C:", lr.C)

print("\n[AUC row-level (IPCW-weighted fit; predictions uncalibrated)]")
print("AUC_train:", auc_tr)
print("AUC_test :", auc_te)

print("\n[hazard_hat_ipcw_raw range]")
print("train min/max:", float(np.min(p_tr)), float(np.max(p_tr)))
print("test  min/max:", float(np.min(p_te)), float(np.max(p_te)))

print("\n[G_hat_cens summary export]")
print("G_table_csv:", pG)
print("G_fig_png  :", pGfig)
print("=== P7.2-v3 DONE ===")

[TABLE] saved: outputs_v2/tables/table_ipcw_censoring_G_train.csv | shape=(64, 5)
columns: ['week', 'G_hat_mean', 'G_hat_median', 'G_hat_p10', 'G_hat_p90']
sample (n=5):
   week  G_hat_mean  G_hat_median  G_hat_p10  G_hat_p90
0     0    0.215104      0.155223   0.062922   0.426483
1     1    0.189774      0.127334   0.048456   0.406639
2     2    0.163814      0.105418   0.034620   0.365206
3     3    0.142176      0.087461   0.025278   0.325399
4     4    0.124841      0.073689   0.018502   0.293219
[FIGURE] saved: outputs_v2/figures/fig_ipcw_censoring_G_train.png | size_in=(6.40, 4.80) | dpi=200 | approx_px=(1280x960)
=== P7.2-v3 — Conditional IPCW-weighted hazard model ===
[TRAIN censoring model]
g_min_ipcw : 0.05
censoring features -> num: ['total_clicks', 'studied_credits', 'num_of_prev_attempts', 'recency', 'streak', 'submitted_this_week']
censoring features -> cat: ['week', 'code_module', 'code_presentation', 'highest_education', 'age_band', 'gender']
cens_event rate (train): 0.

# P7.2.1 — Export IPCW feature lists (required by P7.3+)


In [930]:
# ==============================================================
# P7.2.1 — Export IPCW feature lists (required by P7.3+)
# --------------------------------------------------------------
# Purpose:
#   Ensure the IPCW feature lists exist as GLOBALS:
#     - num_cols_ipcw
#     - cat_cols_ipcw
#     - X_cols_ipcw
#
# This block is the "contract" between P7.2 and P7.3.
#
# Requires:
#   - pp_train (must already include the chosen feature columns)
# ==============================================================

# ---------- Preconditions ----------
required = ["pp_train"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P7.2.1 missing required objects: {missing}")

# ---------- SOURCE OF TRUTH (match your IPCW model features) ----------
# These are EXACTLY the lists you printed earlier in P7.2-v3.
# If you changed your IPCW model feature set, edit only these two lists.

num_cols_ipcw = [
    "total_clicks",
    "studied_credits",
    "num_of_prev_attempts",
    "recency",
    "streak",
    "submitted_this_week",
]

cat_cols_ipcw = [
    "week",
    "code_module",
    "code_presentation",
    "highest_education",
    "age_band",
    "gender",
]

# ---------- Validate existence in pp_train ----------
miss_num = [c for c in num_cols_ipcw if c not in pp_train.columns]
miss_cat = [c for c in cat_cols_ipcw if c not in pp_train.columns]
if miss_num or miss_cat:
    raise KeyError(
        "P7.2.1 feature mismatch vs pp_train. Missing:\n"
        f"  numeric_missing={miss_num}\n"
        f"  categorical_missing={miss_cat}\n"
        "Fix upstream feature engineering / merges before calibration."
    )

X_cols_ipcw = num_cols_ipcw + cat_cols_ipcw

print("=== P7.2.1 — IPCW feature lists exported ===")
print("num_cols_ipcw:", num_cols_ipcw)
print("cat_cols_ipcw:", cat_cols_ipcw)
print("X_cols_ipcw  :", X_cols_ipcw)
print("=== P7.2.1 DONE ===")

=== P7.2.1 — IPCW feature lists exported ===
num_cols_ipcw: ['total_clicks', 'studied_credits', 'num_of_prev_attempts', 'recency', 'streak', 'submitted_this_week']
cat_cols_ipcw: ['week', 'code_module', 'code_presentation', 'highest_education', 'age_band', 'gender']
X_cols_ipcw  : ['total_clicks', 'studied_credits', 'num_of_prev_attempts', 'recency', 'streak', 'submitted_this_week', 'week', 'code_module', 'code_presentation', 'highest_education', 'age_band', 'gender']
=== P7.2.1 DONE ===


In [933]:
# ==============================================================
# P7.4-v2 - Hazard with IPCW in training (no_ipcw vs ipcw_train)
# --------------------------------------------------------------
# Purpose:
#   - Address informative censoring during model fitting by applying IPCW on TRAIN.
#   - Keep the baseline no-IPCW track available for direct comparison.
#   - Avoid sklearn sample_weight routing warnings by using explicit weighted fit
#     + manual Platt calibration (instead of CalibratedClassifierCV).
# ============================================================== 

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("=== P7.4-v2 - IPCW-weighted hazard training (train-only, warning-free) ===")

# ------------------------------
# 0) Preconditions
# ------------------------------
req = ["pp_train", "pp_test", "KEYS", "numeric_features", "categorical_features"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"P7.4-v2 missing required objects: {missing}")

for c in ["G_hat_cens", "event"]:
    if c not in pp_train.columns:
        raise KeyError(f"P7.4-v2 requires '{c}' in pp_train. Run P7.2-v3 first.")
if "hazard_hat" not in pp_train.columns:
    raise KeyError("P7.4-v2 requires baseline hazard_hat from P7-v3.")

# ------------------------------
# 1) IPCW weight setup (train)
# ------------------------------
eps_ipcw_train = 1e-6
w_caps = [20.0, 50.0, 100.0]
w_cap_main = 50.0

G_train = pd.to_numeric(pp_train["G_hat_cens"], errors="coerce")
if G_train.isna().any():
    raise ValueError("P7.4-v2: G_hat_cens has NaNs in pp_train.")

w_raw = 1.0 / np.maximum(G_train.to_numpy(dtype=float), eps_ipcw_train)

weight_rows = []
for cap in w_caps:
    w_cap = np.minimum(w_raw, cap)
    row = {
        "w_cap": float(cap),
        "eps_ipcw": eps_ipcw_train,
        "n": int(len(w_cap)),
        "min": float(w_cap.min()),
        "median": float(np.median(w_cap)),
        "p95": float(np.quantile(w_cap, 0.95)),
        "p99": float(np.quantile(w_cap, 0.99)),
        "max": float(w_cap.max()),
        "frac_clipped": float((w_raw > cap).mean()),
        "n_clipped": int((w_raw > cap).sum()),
        "frac_G_lt_1e-3": float((G_train < 1e-3).mean()),
        "frac_G_lt_1e-2": float((G_train < 1e-2).mean()),
    }
    weight_rows.append(row)

weights_df = pd.DataFrame(weight_rows)

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

w_summary_csv = os.path.join(TABLE_DIR, "table_ipcw_weights_summary_train.csv")
weights_df.to_csv(w_summary_csv, index=False)

w_main = np.minimum(w_raw, w_cap_main)
pp_train["ipcw_w_train"] = w_main

fig_w = os.path.join(FIG_DIR, "fig_ipcw_weights_distribution_train.png")
plt.figure()
plt.hist(w_raw, bins=50, alpha=0.6, label="w_raw")
plt.hist(w_main, bins=50, alpha=0.6, label=f"w_cap_main={w_cap_main:g}")
plt.axvline(w_cap_main, color="red", linestyle=":", label="w_cap_main")
plt.xlabel("IPCW weight")
plt.ylabel("Count")
plt.title("IPCW weights - train (raw vs capped)")
plt.legend()
plt.tight_layout()
plt.savefig(fig_w, dpi=200)
plt.close()

# ------------------------------
# 2) Helpers (metrics + calibration)
# ------------------------------
EPS = 1e-12


def brier_score(y_true: np.ndarray, p: np.ndarray) -> float:
    y_true = y_true.astype(int)
    return float(np.mean((y_true - p) ** 2))


def calibration_bins(y_true: np.ndarray, p: np.ndarray, n_bins: int = 15) -> pd.DataFrame:
    y_true = y_true.astype(int)
    p = np.clip(p.astype(float), 0.0, 1.0)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.digitize(p, edges, right=False) - 1
    bin_id = np.clip(bin_id, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = bin_id == b
        n = int(m.sum())
        if n == 0:
            rows.append((b, 0, edges[b], edges[b + 1], np.nan, np.nan, np.nan))
            continue
        mean_pred = float(np.mean(p[m]))
        emp_rate = float(np.mean(y_true[m]))
        rows.append((b, n, edges[b], edges[b + 1], mean_pred, emp_rate, abs(mean_pred - emp_rate)))
    return pd.DataFrame(rows, columns=["bin", "n", "p_min", "p_max", "mean_pred", "empirical_rate", "abs_gap"])


def ece_from_bins(df_bins: pd.DataFrame) -> float:
    n_total = float(df_bins["n"].sum())
    if n_total <= 0:
        return float("nan")
    return float((df_bins["n"] / n_total * df_bins["abs_gap"].fillna(0.0)).sum())


def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p.astype(float), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def _fit_platt_from_raw(raw_p: np.ndarray, y: np.ndarray, w: np.ndarray) -> tuple[float, float]:
    z = _logit(raw_p).reshape(-1, 1)
    lr = LogisticRegression(
        solver="lbfgs",
        C=1e6,
        max_iter=4000,
        random_state=42,
    )
    lr.fit(z, y.astype(int), sample_weight=w.astype(float))
    a = float(lr.coef_[0, 0])
    b = float(lr.intercept_[0])
    return a, b


def _apply_platt_from_raw(raw_p: np.ndarray, a: float, b: float) -> np.ndarray:
    z = _logit(raw_p)
    p = 1.0 / (1.0 + np.exp(-(a * z + b)))
    return np.clip(p.astype(float), EPS, 1 - EPS)


class IPCWTrainCalibrator:
    """Wraps fitted base pipeline + Platt parameters with predict_proba(X)."""

    def __init__(self, base_model, a: float, b: float):
        self.base_model = base_model
        self.a = float(a)
        self.b = float(b)

    def predict_proba(self, X):
        raw = np.clip(self.base_model.predict_proba(X)[:, 1].astype(float), EPS, 1 - EPS)
        p1 = _apply_platt_from_raw(raw, self.a, self.b)
        p0 = 1.0 - p1
        return np.vstack([p0, p1]).T


# ------------------------------
# 3) Data and features (same columns as P7-v3)
# ------------------------------
X_cols_dt = list(numeric_features + categorical_features)

for df_name in ["pp_train", "pp_test"]:
    df = globals()[df_name]
    miss = [c for c in X_cols_dt if c not in df.columns]
    if miss:
        raise KeyError(f"P7.4-v2: {df_name} missing required columns: {miss}")

X_train = pp_train[X_cols_dt]
y_train = pp_train["event"].astype(int).to_numpy()
X_test = pp_test[X_cols_dt]
y_test = pp_test["event"].astype(int).to_numpy()

# Baseline already materialized (hazard_hat)
p_train_base = np.clip(pp_train["hazard_hat"].astype(float).to_numpy(), EPS, 1 - EPS)
p_test_base = np.clip(pp_test["hazard_hat"].astype(float).to_numpy(), EPS, 1 - EPS)


# ------------------------------
# 4) IPCW training/evaluation function (cap sensitivity)
# ------------------------------
def fit_ipcw(cap: float, save_preds: bool = False):
    w_use = np.minimum(w_raw, cap)

    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(with_mean=True, with_std=True), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ],
        remainder="drop",
    )

    base_lr = LogisticRegression(
        solver="liblinear",
        max_iter=4000,
        class_weight="balanced",
        random_state=42,
    )

    pipe = Pipeline([("pre", pre), ("lr", base_lr)])
    pipe.fit(X_train, y_train, lr__sample_weight=w_use)

    p_tr_raw = np.clip(pipe.predict_proba(X_train)[:, 1].astype(float), EPS, 1 - EPS)
    p_te_raw = np.clip(pipe.predict_proba(X_test)[:, 1].astype(float), EPS, 1 - EPS)

    try:
        a_platt, b_platt = _fit_platt_from_raw(p_tr_raw, y_train, w_use)
    except Exception as e:
        print(f"[WARN] P7.4-v2 Platt fit failed at cap={cap:g}; using identity calibration. Error: {e}")
        a_platt, b_platt = 1.0, 0.0

    p_tr = _apply_platt_from_raw(p_tr_raw, a_platt, b_platt)
    p_te = _apply_platt_from_raw(p_te_raw, a_platt, b_platt)

    auc_tr = float(roc_auc_score(y_train, p_tr))
    auc_te = float(roc_auc_score(y_test, p_te))
    brier_te = brier_score(y_test, p_te)
    bins_te = calibration_bins(y_test, p_te, n_bins=15)
    ece_te = ece_from_bins(bins_te)

    calibrator = IPCWTrainCalibrator(pipe, a_platt, b_platt)

    if save_preds:
        pp_train["hazard_hat_ipcwtrain"] = p_tr
        pp_test["hazard_hat_ipcwtrain"] = p_te
        globals()["hazard_model_ipcwtrain"] = calibrator
        globals()["hazard_calibrator_gcv_ipcwtrain"] = calibrator
        globals()["hazard_calibrator_rs_ipcwtrain"] = calibrator

        bins_path = os.path.join(TABLE_DIR, "table_calibration_ipcwtrain_bins_test.csv")
        bins_te.to_csv(bins_path, index=False)

        metrics_path = os.path.join(TABLE_DIR, "table_calibration_ipcwtrain_metrics.csv")
        pd.DataFrame(
            [
                ("AUC_train", auc_tr),
                ("AUC_test", auc_te),
                ("Brier_test", brier_te),
                ("ECE_test", ece_te),
                ("n_bins", 15.0),
                ("w_cap", cap),
                ("eps_ipcw", eps_ipcw_train),
                ("platt_a", a_platt),
                ("platt_b", b_platt),
            ],
            columns=["metric", "value"],
        ).to_csv(metrics_path, index=False)

        fig_cal = os.path.join(FIG_DIR, "fig_calibration_reliability_test_ipcwtrain.png")
        df_plot = bins_te.loc[bins_te["n"] > 0].copy()
        plt.figure()
        plt.plot(df_plot["mean_pred"], df_plot["empirical_rate"], marker="o", linestyle="-")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("Mean predicted probability (bin)")
        plt.ylabel("Empirical event rate (bin)")
        plt.title("Calibration (Reliability) - TEST (IPCW train)")
        plt.tight_layout()
        plt.savefig(fig_cal, dpi=200)
        plt.close()

    return {
        "model_label": f"ipcw_train_cap{cap:g}",
        "w_cap": float(cap),
        "eps_ipcw": eps_ipcw_train,
        "auc_train": auc_tr,
        "auc_test": auc_te,
        "brier_test": brier_te,
        "ece_test": ece_te,
        "frac_clipped": float((w_raw > cap).mean()),
    }


# ------------------------------
# 5) Main run (selected cap) + sensitivity
# ------------------------------
metrics_rows = []

# Baseline row
bins_base = calibration_bins(y_test, p_test_base, n_bins=15)
metrics_rows.append(
    {
        "model_label": "baseline_no_ipcw",
        "w_cap": np.nan,
        "eps_ipcw": np.nan,
        "auc_train": float(roc_auc_score(y_train, p_train_base)),
        "auc_test": float(roc_auc_score(y_test, p_test_base)),
        "brier_test": brier_score(y_test, p_test_base),
        "ece_test": ece_from_bins(bins_base),
        "frac_clipped": 0.0,
    }
)

# Sensitivity caps
for cap in w_caps:
    save = bool(cap == w_cap_main)
    m = fit_ipcw(cap=cap, save_preds=save)
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows)
model_cmp_csv = os.path.join(TABLE_DIR, "table_model_comparison_ipcw_vs_noipcw.csv")
metrics_df.to_csv(model_cmp_csv, index=False)

# ------------------------------
# 6) Logs
# ------------------------------
print("[Weights - cap summary]")
print(weights_df)

print("\n[Weights - main cap stats]")
print(weights_df.loc[weights_df["w_cap"] == w_cap_main].to_dict(orient="records")[0])

print("\n[Model comparison]")
print(metrics_df)

print("\nArtifacts:")
print("weights summary :", w_summary_csv)
print("weights figure  :", fig_w)
print("model cmp table :", model_cmp_csv)
print("calib bins ipcw :", os.path.join(TABLE_DIR, "table_calibration_ipcwtrain_bins_test.csv"))
print("calib fig ipcw  :", os.path.join(FIG_DIR, "fig_calibration_reliability_test_ipcwtrain.png"))
print("=== P7.4-v2 DONE ===")


=== P7.4-v2 - IPCW-weighted hazard training (train-only, warning-free) ===
[TABLE] saved: outputs_v2/tables/table_ipcw_weights_summary_train.csv | shape=(3, 12)
columns: ['w_cap', 'eps_ipcw', 'n', 'min', 'median', 'p95', 'p99', 'max', 'frac_clipped', 'n_clipped', 'frac_G_lt_1e-3', 'frac_G_lt_1e-2']
sample (n=3):
   w_cap  eps_ipcw       n  min     median    p95    p99    max  frac_clipped  \
0   20.0  0.000001  542878  1.0  20.000000   20.0   20.0   20.0      0.656426   
1   50.0  0.000001  542878  1.0  47.436463   50.0   50.0   50.0      0.491713   
2  100.0  0.000001  542878  1.0  47.436463  100.0  100.0  100.0      0.390937   

   n_clipped  frac_G_lt_1e-3  frac_G_lt_1e-2  
0     356359         0.18608        0.390937  
1     266940         0.18608        0.390937  
2     212231         0.18608        0.390937  
[FIGURE] saved: outputs_v2/figures/fig_ipcw_weights_distribution_train.png | size_in=(6.40, 4.80) | dpi=200 | approx_px=(1280x960)
[TABLE] saved: outputs_v2/tables/table_cal

In [934]:
# ==============================================================
# P7.3.1-v2 — IPCW Platt aliases created (NO DeprecationWarning)
# --------------------------------------------------------------
# Purpose:
#   Stabilize Platt parameters under consistent names so downstream
#   blocks never "guess" variable names.
#
# Source of truth in THIS notebook (after P7.3):
#   - platt_a_ipcw
#   - platt_b_ipcw
#
# Outputs:
#   - platt_a, platt_b   (float)
#   - a, b               (float)  [legacy aliases]
#   - prints types/values + finite checks
# ==============================================================

import numpy as np

def _to_float_scalar(x, name: str) -> float:
    """
    Convert x to a Python float safely.
    - If x is a numpy scalar -> float(x)
    - If x is a numpy ndarray with a single element -> float(x.item())
    - If x is already Python number -> float(x)
    Otherwise raise (no silent coercion).
    """
    if x is None:
        raise ValueError(f"P7.3.1-v2: '{name}' is None.")
    if isinstance(x, (float, int, np.floating, np.integer)):
        return float(x)
    if isinstance(x, np.ndarray):
        if x.size != 1:
            raise ValueError(f"P7.3.1-v2: '{name}' is ndarray with size={x.size}, expected size=1.")
        return float(x.item())
    # last resort: try float(), but still fail loudly if not possible
    try:
        return float(x)
    except Exception as e:
        raise ValueError(f"P7.3.1-v2: cannot convert '{name}' (type={type(x)}) to float.") from e

required = ["platt_a_ipcw", "platt_b_ipcw"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P7.3.1-v2 missing required objects: {missing} (run P7.3 first).")

# source
src_a = globals()["platt_a_ipcw"]
src_b = globals()["platt_b_ipcw"]

# normalize
platt_a = _to_float_scalar(src_a, "platt_a_ipcw")
platt_b = _to_float_scalar(src_b, "platt_b_ipcw")

# write aliases
a = platt_a
b = platt_b
globals()["platt_a"] = platt_a
globals()["platt_b"] = platt_b
globals()["a"] = a
globals()["b"] = b

print("=== P7.3.1-v2 — IPCW Platt aliases created (NO WARNING) ===")
print("[Source vars]")
print("platt_a_ipcw:", type(src_a).__name__, "| repr:", repr(src_a))
print("platt_b_ipcw:", type(src_b).__name__, "| repr:", repr(src_b))

print("\n[Aliases written as Python floats]")
print("platt_a:", platt_a, "| type:", type(platt_a).__name__)
print("platt_b:", platt_b, "| type:", type(platt_b).__name__)
print("a      :", a,       "| type:", type(a).__name__)
print("b      :", b,       "| type:", type(b).__name__)

print("\n[Sanity]")
print("np.isfinite(platt_a), np.isfinite(platt_b):", bool(np.isfinite(platt_a)), bool(np.isfinite(platt_b)))
print("=== P7.3.1-v2 DONE ===")

=== P7.3.1-v2 — IPCW Platt aliases created (NO WARNING) ===
[Source vars]
platt_a_ipcw: float | repr: 0.9624038615400873
platt_b_ipcw: float | repr: -0.16132664730068075

[Aliases written as Python floats]
platt_a: 0.9624038615400873 | type: float
platt_b: -0.16132664730068075 | type: float
a      : 0.9624038615400873 | type: float
b      : -0.16132664730068075 | type: float

[Sanity]
np.isfinite(platt_a), np.isfinite(platt_b): True True
=== P7.3.1-v2 DONE ===


In [935]:
# ==============================================================
# P7.4-v4 — Materialize hazard_hat_ipcw into pp_train/pp_test (STRICT)
# --------------------------------------------------------------
# Requires:
#   - pp_train, pp_test
#   - X_cols_ipcw
#   - hazard_model_ipcw_fitted
#   - platt_a_ipcw, platt_b_ipcw
#
# Output:
#   - pp_train['hazard_hat_ipcw']
#   - pp_test ['hazard_hat_ipcw']
# ==============================================================

import numpy as np
from sklearn.metrics import roc_auc_score

required = ["pp_train", "pp_test", "X_cols_ipcw", "hazard_model_ipcw_fitted", "platt_a_ipcw", "platt_b_ipcw"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P7.4-v4 missing required objects: {missing}")

EPS = 1e-12
X_cols = list(X_cols_ipcw)

def _logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def _platt_apply(p):
    z = _logit(p)
    s = 1.0 / (1.0 + np.exp(-(float(platt_a_ipcw) * z + float(platt_b_ipcw))))
    return np.clip(s, EPS, 1 - EPS)

p_tr_raw = hazard_model_ipcw_fitted.predict_proba(pp_train[X_cols])[:, 1].astype(float)
p_te_raw = hazard_model_ipcw_fitted.predict_proba(pp_test[X_cols])[:, 1].astype(float)

pp_train["hazard_hat_ipcw"] = _platt_apply(p_tr_raw)
pp_test["hazard_hat_ipcw"] = _platt_apply(p_te_raw)

auc_train = float(roc_auc_score(pp_train["event"].astype(int), pp_train["hazard_hat_ipcw"]))
auc_test  = float(roc_auc_score(pp_test["event"].astype(int),  pp_test["hazard_hat_ipcw"]))

print("=== P7.4-v4 — hazard_hat_ipcw materialized (calibrated) ===")
print("AUC_train:", auc_train)
print("AUC_test :", auc_test)
print("train min/max:", float(pp_train["hazard_hat_ipcw"].min()), float(pp_train["hazard_hat_ipcw"].max()))
print("test  min/max:", float(pp_test["hazard_hat_ipcw"].min()), float(pp_test["hazard_hat_ipcw"].max()))
print("=== P7.4-v4 DONE ===")

=== P7.4-v4 — hazard_hat_ipcw materialized (calibrated) ===
AUC_train: 0.8354830684290305
AUC_test : 0.8394664715157028
train min/max: 2.4048632671391433e-12 0.7873502092871676
test  min/max: 2.4048632671391433e-12 0.45478795405535916
=== P7.4-v4 DONE ===


In [938]:
# ==============================================================
# P9.1-v1 — Non-Temporal Enrollment Baseline (RSF-first)
# --------------------------------------------------------------
# Purpose:
#   Add a strong non-temporal benchmark at enrollment level to complement
#   temporal discrete-time and Cox baselines.
#
# Strategy:
#   1) Build one row per enrollment with static/baseline attributes.
#   2) Train Random Survival Forest (RSF) if scikit-survival is available.
#   3) If RSF is unavailable, fallback to RandomForestClassifier and report it
#      explicitly as a fallback benchmark.
#
# Outputs (source of truth):
#   - outputs_v2/tables/table_non_temporal_rsf_metrics.csv
#   - outputs_v2/tables/table_non_temporal_rsf_predictions_test.csv
#   - outputs_v2/tables/table_non_temporal_rsf_calibration_teval.csv
#   - outputs_v2/figures/fig_non_temporal_rsf_calibration_teval.png
#   - outputs_v2/figures/fig_non_temporal_rsf_feature_importance_top15.png (if available)
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ------------------------------
# 0) Preconditions + constants
# ------------------------------
required = ["KEYS", "enrollments_eventtime", "studentInfo"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P9.1-v1 missing required objects: {missing}")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

T_policy_local = int(globals().get("T_policy", 18))
T_eval_local = int(globals().get("T_eval_metrics", max(T_policy_local, 30)))

# ------------------------------
# 1) Build enrollment-level table
# ------------------------------
base_cols = KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]
missing_base = [c for c in base_cols if c not in enrollments_eventtime.columns]
if missing_base:
    raise KeyError(f"P9.1-v1 missing columns in enrollments_eventtime: {missing_base}")

enr = (
    enrollments_eventtime[base_cols]
    .drop_duplicates(subset=KEYS)
    .copy()
)

enr["event_withdrawn"] = pd.to_numeric(enr["event_withdrawn"], errors="coerce").fillna(0).astype(int)
enr["t_event_week"] = pd.to_numeric(enr["t_event_week"], errors="coerce")
enr["t_final_week"] = pd.to_numeric(enr["t_final_week"], errors="coerce").fillna(0).astype(int)

static_candidates = [
    "studied_credits", "num_of_prev_attempts", "highest_education",
    "age_band", "gender", "disability", "imd_band", "region"
]
static_cols = [c for c in static_candidates if c in studentInfo.columns]

static_df = (
    studentInfo[KEYS + static_cols]
    .drop_duplicates(subset=KEYS)
    .copy()
)

enr = enr.merge(static_df, on=KEYS, how="left", validate="1:1")

# Add course/run identifiers as non-temporal features
for c in ["code_module", "code_presentation"]:
    if c not in enr.columns and c in KEYS:
        # Already present via KEYS in enrollments_eventtime
        pass

num_candidates = ["studied_credits", "num_of_prev_attempts"]
cat_candidates = ["code_module", "code_presentation", "highest_education", "age_band", "gender", "disability", "imd_band", "region"]

num_cols = [c for c in num_candidates if c in enr.columns]
cat_cols = [c for c in cat_candidates if c in enr.columns]
feature_cols = num_cols + cat_cols

if len(feature_cols) == 0:
    raise ValueError("P9.1-v1: no non-temporal features available for training.")

# ------------------------------
# 2) Train/test enrollment split
# ------------------------------
if "enroll_train_keys" in globals() and "enroll_test_keys" in globals():
    train_keys = enroll_train_keys[KEYS].drop_duplicates().copy()
    test_keys = enroll_test_keys[KEYS].drop_duplicates().copy()
elif "pp_train" in globals() and "pp_test" in globals():
    train_keys = pp_train[KEYS].drop_duplicates().copy()
    test_keys = pp_test[KEYS].drop_duplicates().copy()
else:
    raise NameError(
        "P9.1-v1 requires enroll_train_keys/enroll_test_keys or pp_train/pp_test to define split."
    )

train_df = enr.merge(train_keys, on=KEYS, how="inner", validate="1:1").copy()
test_df = enr.merge(test_keys, on=KEYS, how="inner", validate="1:1").copy()

if train_df.empty or test_df.empty:
    raise ValueError("P9.1-v1: empty train/test enrollment split after merge.")

# Event-by-horizon labels (enrollment-level)
def event_by_horizon(df: pd.DataFrame, T: int) -> np.ndarray:
    t_event = pd.to_numeric(df["t_event_week"], errors="coerce").astype(float).fillna(np.inf).to_numpy(dtype=float)
    e = df["event_withdrawn"].astype(int).to_numpy()
    return ((e == 1) & (t_event <= T)).astype(int)

y_test_policy = event_by_horizon(test_df, T_policy_local)
y_test_eval = event_by_horizon(test_df, T_eval_local)

# ------------------------------
# 3) Preprocess
# ------------------------------
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", ohe),
])

pre = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

X_train = pre.fit_transform(train_df[feature_cols])
X_test = pre.transform(test_df[feature_cols])

# Observed time for survival setup: event time if event, otherwise censoring time
time_train = np.where(
    train_df["event_withdrawn"].astype(int).to_numpy() == 1,
    pd.to_numeric(train_df["t_event_week"], errors="coerce").fillna(train_df["t_final_week"]).to_numpy(dtype=float),
    pd.to_numeric(train_df["t_final_week"], errors="coerce").to_numpy(dtype=float),
)
time_train = np.maximum(time_train, 1.0)

event_train = train_df["event_withdrawn"].astype(int).to_numpy().astype(bool)

# ------------------------------
# 4) RSF-first fit (with fallback)
# ------------------------------
model_impl = None
model_name = None
feature_importances = None
risk_test = None
p_test_policy = None
p_test_eval = None

try:
    from sksurv.ensemble import RandomSurvivalForest
    from sksurv.util import Surv

    y_train_surv = Surv.from_arrays(event=event_train, time=time_train)

    rsf = RandomSurvivalForest(
        n_estimators=300,
        min_samples_split=20,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42,
    )
    rsf.fit(X_train, y_train_surv)

    sf_test = rsf.predict_survival_function(X_test, return_array=False)

    def event_prob_at_t(sf_list, T: int) -> np.ndarray:
        vals = []
        for sf in sf_list:
            s = float(sf(T))
            vals.append(np.clip(1.0 - s, 1e-12, 1 - 1e-12))
        return np.asarray(vals, dtype=float)

    p_test_policy = event_prob_at_t(sf_test, T_policy_local)
    p_test_eval = event_prob_at_t(sf_test, T_eval_local)
    risk_test = rsf.predict(X_test).astype(float)

    feature_importances = getattr(rsf, "feature_importances_", None)
    model_name = "random_survival_forest"
    model_impl = "sksurv"

except Exception as ex:
    # Fallback keeps notebook runnable when scikit-survival is not available.
    rf = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, train_df["event_withdrawn"].astype(int).to_numpy())

    p_any = rf.predict_proba(X_test)[:, 1].astype(float)
    p_test_policy = np.clip(p_any, 1e-12, 1 - 1e-12)
    p_test_eval = np.clip(p_any, 1e-12, 1 - 1e-12)
    risk_test = p_any.copy()

    feature_importances = getattr(rf, "feature_importances_", None)
    model_name = "rf_classifier_fallback"
    model_impl = f"sklearn_fallback_no_sksurv ({type(ex).__name__})"

# ------------------------------
# 5) Metrics + evidence tables
# ------------------------------
def safe_auc(y_true: np.ndarray, p: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, p))

auc_policy = safe_auc(y_test_policy, p_test_policy)
auc_eval = safe_auc(y_test_eval, p_test_eval)
brier_policy = float(np.mean((y_test_policy - p_test_policy) ** 2))
brier_eval = float(np.mean((y_test_eval - p_test_eval) ** 2))

metrics_df = pd.DataFrame([
    ("model_name", model_name),
    ("implementation", model_impl),
    ("T_policy", float(T_policy_local)),
    ("T_eval_metrics", float(T_eval_local)),
    ("n_train_enrollments", float(len(train_df))),
    ("n_test_enrollments", float(len(test_df))),
    ("n_features_numeric", float(len(num_cols))),
    ("n_features_categorical", float(len(cat_cols))),
    ("AUC_event_by_T_policy", auc_policy),
    ("AUC_event_by_T_eval", auc_eval),
    ("Brier_event_by_T_policy", brier_policy),
    ("Brier_event_by_T_eval", brier_eval),
], columns=["metric", "value"])

pred_df = test_df[KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]].copy()
pred_df["y_event_by_T_policy"] = y_test_policy
pred_df["y_event_by_T_eval"] = y_test_eval
pred_df["p_event_by_T_policy"] = p_test_policy
pred_df["p_event_by_T_eval"] = p_test_eval
pred_df["risk_score"] = risk_test

# Calibration-by-decile at T_eval
ranked = pred_df[["y_event_by_T_eval", "p_event_by_T_eval"]].copy()
ranked["rank"] = ranked["p_event_by_T_eval"].rank(method="first")
ranked["decile"] = pd.qcut(ranked["rank"], q=10, labels=False)
calib_df = (
    ranked.groupby("decile", as_index=False)
    .agg(
        n=("y_event_by_T_eval", "size"),
        mean_pred=("p_event_by_T_eval", "mean"),
        empirical_rate=("y_event_by_T_eval", "mean"),
    )
)
calib_df["abs_gap"] = (calib_df["mean_pred"] - calib_df["empirical_rate"]).abs()

p_metrics = os.path.join(TABLE_DIR, "table_non_temporal_rsf_metrics.csv")
p_preds = os.path.join(TABLE_DIR, "table_non_temporal_rsf_predictions_test.csv")
p_calib = os.path.join(TABLE_DIR, "table_non_temporal_rsf_calibration_teval.csv")
metrics_df.to_csv(p_metrics, index=False)
pred_df.to_csv(p_preds, index=False)
calib_df.to_csv(p_calib, index=False)

# ------------------------------
# 6) Figures for paper/evidence
# ------------------------------
fig_cal = os.path.join(FIG_DIR, "fig_non_temporal_rsf_calibration_teval.png")
plt.figure()
plt.plot(calib_df["mean_pred"], calib_df["empirical_rate"], marker="o", linestyle="-")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean predicted probability (decile)")
plt.ylabel("Empirical event rate by T_eval")
plt.title("Non-temporal baseline calibration at T_eval")
plt.tight_layout()
plt.savefig(fig_cal, dpi=200)
plt.close()

p_feat = os.path.join(FIG_DIR, "fig_non_temporal_rsf_feature_importance_top15.png")
if feature_importances is not None:
    try:
        feat_names = pre.get_feature_names_out()
    except Exception:
        feat_names = np.array([f"f{i}" for i in range(len(feature_importances))])

    k = min(15, len(feature_importances))
    top_idx = np.argsort(feature_importances)[-k:][::-1]

    fi_df = pd.DataFrame({
        "feature": np.asarray(feat_names)[top_idx],
        "importance": np.asarray(feature_importances)[top_idx],
    })

    plt.figure(figsize=(8, 5))
    plt.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    plt.xlabel("Importance")
    plt.title("Top non-temporal baseline feature importances")
    plt.tight_layout()
    plt.savefig(p_feat, dpi=200)
    plt.close()
else:
    p_feat = "not_available"

# ------------------------------
# 7) Required console outputs (source of truth)
# ------------------------------
print("=== P9.1-v1 — Non-Temporal Enrollment Baseline (RSF-first) ===")
print("[Model]")
print("model_name     :", model_name)
print("implementation :", model_impl)
print("\n[Horizon setup]")
print("T_policy       :", T_policy_local)
print("T_eval_metrics :", T_eval_local)
print("\n[Data split]")
print("n_train_enrollments:", len(train_df))
print("n_test_enrollments :", len(test_df))
print("n_features_numeric :", len(num_cols), "->", num_cols)
print("n_features_categorical:", len(cat_cols), "->", cat_cols)
print("\n[Metrics]")
print(metrics_df.to_string(index=False))
print("\n[Calibration deciles @ T_eval]")
print(calib_df.to_string(index=False))
print("\n[Artifacts]")
print("metrics_table      :", p_metrics)
print("predictions_table  :", p_preds)
print("calibration_table  :", p_calib)
print("calibration_figure :", fig_cal)
print("feature_importance :", p_feat)
print("=== P9.1-v1 DONE ===")

[TABLE] saved: outputs_v2/tables/table_non_temporal_rsf_metrics.csv | shape=(12, 2)
columns: ['metric', 'value']
sample (n=5):
                metric                                             value
0           model_name                            rf_classifier_fallback
1       implementation  sklearn_fallback_no_sksurv (ModuleNotFoundError)
2             T_policy                                              18.0
3       T_eval_metrics                                              37.0
4  n_train_enrollments                                           22815.0
[TABLE] saved: outputs_v2/tables/table_non_temporal_rsf_predictions_test.csv | shape=(9778, 11)
columns: ['id_student', 'code_module', 'code_presentation', 'event_withdrawn', 't_event_week', 't_final_week', 'y_event_by_T_policy', 'y_event_by_T_eval', 'p_event_by_T_policy', 'p_event_by_T_eval', 'risk_score']
sample (n=5):
   id_student code_module code_presentation  event_withdrawn  t_event_week  \
0       32885         AAA         

In [939]:
# ==============================================================
# P10-v2 — Recency/Streak Audit (NO-OP; already materialized in P5.1)
# --------------------------------------------------------------
# Purpose:
#   Keep P10 numbering for the notebook narrative, but do NOT recompute.
#   Just verify pp_train/pp_test contain the required columns.
#
# Requires:
#   - pp_train, pp_test (from P6)
#   - recency, streak (should exist via P5.1)
# ==============================================================

import pandas as pd

required = ["pp_train", "pp_test"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P10-v2 missing required objects: {missing}")

need_cols = ["recency", "streak", "activity_flag"]
miss_train = [c for c in need_cols if c not in pp_train.columns]
miss_test  = [c for c in need_cols if c not in pp_test.columns]
if miss_train or miss_test:
    raise KeyError(
        "P10-v2 expected recency/streak/activity_flag to already exist.\n"
        f"Missing in pp_train: {miss_train}\n"
        f"Missing in pp_test : {miss_test}\n"
        "Fix: ensure P5.1 runs before P6."
    )

print("=== P10-v2 — Recency/Streak Audit ===")
print("Train rows:", len(pp_train), " Test rows:", len(pp_test))

print("Recency min/median/max (train):",
      int(pp_train["recency"].min()),
      float(pp_train["recency"].median()),
      int(pp_train["recency"].max()))
print("Streak  min/median/max (train):",
      int(pp_train["streak"].min()),
      float(pp_train["streak"].median()),
      int(pp_train["streak"].max()))

bad1 = pp_train[(pp_train["activity_flag"] == 1) & (pp_train["recency"] != 0)]
bad2 = pp_train[(pp_train["activity_flag"] == 1) & (pp_train["streak"] < 1)]
print("Sanity (train): activity_flag=1 but recency!=0 rows:", len(bad1))
print("Sanity (train): activity_flag=1 but streak<1 rows:", len(bad2))
print("=== P10-v2 DONE ===")

=== P10-v2 — Recency/Streak Audit ===
Train rows: 542878  Test rows: 232417
Recency min/median/max (train): 0 0.0 42
Streak  min/median/max (train): 0 2.0 39
Sanity (train): activity_flag=1 but recency!=0 rows: 0
Sanity (train): activity_flag=1 but streak<1 rows: 0
=== P10-v2 DONE ===


In [940]:
# ==============================================================
# P11-v2 — Calibration Report (REPORTABLE; GROUPED CV by enrollment)
# --------------------------------------------------------------
# Fix:
#   - Precompute GroupKFold splits (train/test indices) using enrollment groups
#   - Pass the split list directly to CalibratedClassifierCV(cv=split_list)
#   - Avoids the sklearn routing issue where groups are not forwarded to cv.split()
#
# Expected notebook objects from earlier steps (P1–P10):
#   - KEYS = ["id_student","code_module","code_presentation"]
#   - pp_train, pp_test (person-period DataFrames)
#   - numeric_features, categorical_features (feature lists)
#
# Outputs (exports):
#   - outputs_v2/tables/table_calibration_bins_test.csv
#   - outputs_v2/tables/table_calibration_metrics.csv
#   - outputs_v2/figures/fig_calibration_reliability_test.png
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# ------------------------------
# 0) Preconditions
# ------------------------------
required = ["KEYS", "pp_train", "pp_test", "numeric_features", "categorical_features"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P11-v2 missing required objects: {missing}")

for df_name in ["pp_train", "pp_test"]:
    df = globals()[df_name]
    for col in ["event", "week"]:
        if col not in df.columns:
            raise KeyError(f"P11-v2 requires '{df_name}' to contain column '{col}'.")

# Make sure features exist
num_cols = [c for c in list(numeric_features) if c in pp_train.columns]
cat_cols = [c for c in list(categorical_features) if c in pp_train.columns]

if len(num_cols) == 0 and len(cat_cols) == 0:
    raise ValueError("P11-v2: No features found from numeric_features/categorical_features in pp_train.")

# Defensive anti-leak filter (should already be clean, but we enforce)
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration",
    "hazard_hat", "S_hat", "S0_hat", "S_shock_hat", "S_mech_hat",
}
num_cols = [c for c in num_cols if c not in FORBIDDEN]
cat_cols = [c for c in cat_cols if c not in FORBIDDEN]

if len(num_cols) == 0 and len(cat_cols) == 0:
    raise ValueError("P11-v2: All features removed by FORBIDDEN filter. Check feature lists.")

# Ensure week_sq if expected
if "week_sq" in numeric_features and "week_sq" not in pp_train.columns:
    pp_train = pp_train.copy()
    pp_test = pp_test.copy()
    pp_train["week_sq"] = pp_train["week"] ** 2
    pp_test["week_sq"] = pp_test["week"] ** 2
    if "week_sq" in numeric_features and "week_sq" not in num_cols:
        num_cols.append("week_sq")

X_cols = num_cols + cat_cols

# ------------------------------
# 1) Output dirs
# ------------------------------
OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ------------------------------
# 2) Enrollment-level groups (KEYS concat)
# ------------------------------
def _make_groups(df: pd.DataFrame) -> np.ndarray:
    return (
        df[KEYS]
        .astype(str)
        .agg("|".join, axis=1)
        .to_numpy()
    )

groups_train = _make_groups(pp_train)

# ------------------------------
# 3) Preprocess + base estimator
# ------------------------------
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

pre = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

base_lr = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

base_model = Pipeline([("pre", pre), ("lr", base_lr)])

# ------------------------------
# 4) GROUPED CV splits (FIX)
# ------------------------------
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

X_train = pp_train[X_cols]
y_train = pp_train["event"].astype(int).to_numpy()

# Precompute splits so CalibratedClassifierCV doesn't need groups routing
cv_splits = list(gkf.split(X_train, y_train, groups_train))
if len(cv_splits) != n_splits:
    raise ValueError(f"P11-v2: unexpected number of CV splits. got={len(cv_splits)} expected={n_splits}")

# ------------------------------
# 5) Calibrated model (sigmoid) with grouped CV splits
# ------------------------------
hazard_calibrator_gcv = CalibratedClassifierCV(
    estimator=base_model,
    method="sigmoid",
    cv=cv_splits,
)

hazard_calibrator_gcv.fit(X_train, y_train)

# Store for downstream (compatibility)
hazard_calibrator_rs = hazard_calibrator_gcv

# ------------------------------
# 6) Predictions + AUC diagnostics
# ------------------------------
p_train = hazard_calibrator_gcv.predict_proba(pp_train[X_cols])[:, 1].astype(float)
p_test = hazard_calibrator_gcv.predict_proba(pp_test[X_cols])[:, 1].astype(float)

auc_train = float(roc_auc_score(pp_train["event"].astype(int), p_train))
auc_test = float(roc_auc_score(pp_test["event"].astype(int), p_test))

# ------------------------------
# 7) Calibration metrics on TEST: Brier + ECE + bins
# ------------------------------
def brier_score(y_true: np.ndarray, p: np.ndarray) -> float:
    y_true = y_true.astype(int)
    return float(np.mean((y_true - p) ** 2))

def calibration_bins(y_true: np.ndarray, p: np.ndarray, n_bins: int = 15) -> pd.DataFrame:
    y_true = y_true.astype(int)
    p = np.clip(p.astype(float), 0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.digitize(p, edges, right=False) - 1
    bin_id = np.clip(bin_id, 0, n_bins - 1)

    rows = []
    for b in range(n_bins):
        mask = (bin_id == b)
        n = int(mask.sum())
        if n == 0:
            rows.append((b, 0, edges[b], edges[b+1], np.nan, np.nan, np.nan))
            continue
        mean_pred = float(np.mean(p[mask]))
        emp_rate = float(np.mean(y_true[mask]))
        abs_gap = float(abs(mean_pred - emp_rate))
        rows.append((b, n, edges[b], edges[b+1], mean_pred, emp_rate, abs_gap))

    return pd.DataFrame(rows, columns=["bin", "n", "p_min", "p_max", "mean_pred", "empirical_rate", "abs_gap"])

def ece_from_bins(df_bins: pd.DataFrame) -> float:
    n_total = float(df_bins["n"].sum())
    if n_total <= 0:
        return np.nan
    return float((df_bins["n"] / n_total * df_bins["abs_gap"]).sum())

y_test = pp_test["event"].astype(int).to_numpy()
brier_test = brier_score(y_test, p_test)

n_bins = 15
bins_df = calibration_bins(y_test, p_test, n_bins=n_bins)
ece_test = ece_from_bins(bins_df)

# ------------------------------
# 8) Reliability diagram (TEST)
# ------------------------------
fig_path = os.path.join(FIG_DIR, "fig_calibration_reliability_test.png")

df_plot = bins_df.loc[bins_df["n"] > 0].copy()

plt.figure(figsize=(6.4, 4.8))
plt.plot(df_plot["mean_pred"], df_plot["empirical_rate"], marker="o", linestyle="-", linewidth=1.8, markersize=4, color="#1f77b4", label="Empirical reliability (test bins)")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, color="#ff7f0e", label="Ideal calibration (y = x)")
plt.xlabel("Mean predicted probability (bin)")
plt.ylabel("Empirical event rate (bin)")
plt.title("Calibration (Reliability) — TEST (Grouped CV: GroupKFold splits)")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.gca().set_aspect("equal", adjustable="box")
plt.grid(True, linestyle=":", linewidth=0.6, alpha=0.35)
plt.legend(loc="best", frameon=True, fontsize=8)
plt.tight_layout()
plt.savefig(fig_path, dpi=200)
plt.close()

# ------------------------------
# 9) Exports
# ------------------------------
bins_csv = os.path.join(TABLE_DIR, "table_calibration_bins_test.csv")
metrics_csv = os.path.join(TABLE_DIR, "table_calibration_metrics.csv")

bins_df.to_csv(bins_csv, index=False)

metrics_df = pd.DataFrame(
    [
        ("AUC_train", auc_train),
        ("AUC_test", auc_test),
        ("Brier_test", brier_test),
        ("ECE_test", ece_test),
        ("n_bins", float(n_bins)),
        ("calibration_cv_splits", float(n_splits)),
        ("calibration_cv_type", "GroupKFold(enrollment) via precomputed splits"),
        ("n_features_numeric", float(len(num_cols))),
        ("n_features_categorical", float(len(cat_cols))),
    ],
    columns=["metric", "value"],
)
metrics_df.to_csv(metrics_csv, index=False)

# ------------------------------
# 10) REQUIRED TEXT OUTPUTS
# ------------------------------
print("=== P11-v2 — Calibration Report (REPORTABLE; GROUPED CV) ===")
print("[AUC row-level]")
print("AUC_train:", auc_train)
print("AUC_test :", auc_test)

print("\n[Calibration metrics — TEST]")
print("Brier_test:", brier_test)
print("ECE_test  :", ece_test)
print("n_bins    :", n_bins)

print("\n[Grouped CV calibration]")
print("cv_type   : GroupKFold (enrollment-level)")
print("n_splits  :", n_splits)
print("cv_impl   : precomputed splits (fix for sklearn groups routing)")
print("group_key : KEYS concatenation")

print("\n[Bins head]")
print(bins_df.head(10).to_string(index=False))

print("\n[Exports]")
print("bins_csv   :", bins_csv)
print("metrics_csv:", metrics_csv)
print("fig_png    :", fig_path)
print("=== P11-v2 DONE ===")

[FIGURE] saved: outputs_v2/figures/fig_calibration_reliability_test.png | size_in=(6.40, 4.80) | dpi=200 | approx_px=(1280x960)
[TABLE] saved: outputs_v2/tables/table_calibration_bins_test.csv | shape=(15, 7)
columns: ['bin', 'n', 'p_min', 'p_max', 'mean_pred', 'empirical_rate', 'abs_gap']
sample (n=5):
   bin       n     p_min     p_max  mean_pred  empirical_rate   abs_gap
0    0  230157  0.000000  0.066667   0.008142        0.008724  0.000582
1    1    1499  0.066667  0.133333   0.087664        0.096731  0.009068
2    2     304  0.133333  0.200000   0.163386        0.062500  0.100886
3    3     180  0.200000  0.266667   0.232119        0.055556  0.176564
4    4     104  0.266667  0.333333   0.296277        0.076923  0.219354
[TABLE] saved: outputs_v2/tables/table_calibration_metrics.csv | shape=(9, 2)
columns: ['metric', 'value']
sample (n=5):
       metric     value
0   AUC_train   0.83585
1    AUC_test  0.839577
2  Brier_test  0.009323
3    ECE_test  0.001221
4      n_bins      15.

# P11.1 — Materialize calibrated hazard_hat into pp_train / pp_test

In [941]:
# ==============================================================
# P11.1 — Materialize calibrated hazard_hat into pp_train / pp_test
# --------------------------------------------------------------
# Purpose:
#   P12+ expects pp_train/pp_test to contain 'hazard_hat'.
#   After P11-v2 we have hazard_calibrator_rs; this block writes:
#     - pp_train['hazard_hat']
#     - pp_test ['hazard_hat']
#
# Requires:
#   - hazard_calibrator_rs (from P11-v2)
#   - pp_train, pp_test
#   - numeric_features, categorical_features, KEYS (for feature columns)
#
# Outputs:
#   - Prints ranges + AUC check (should match P11-v2 closely)
# ==============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

required = ["hazard_calibrator_rs", "pp_train", "pp_test", "numeric_features", "categorical_features", "KEYS"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P11.1 missing required objects: {missing}")

# Rebuild feature column list exactly like P11-v2 used (no leakage)
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration",
    "hazard_hat", "S_hat", "S0_hat", "S_shock_hat", "S_mech_hat",
}

num_cols = [c for c in list(numeric_features) if c in pp_train.columns and c not in FORBIDDEN]
cat_cols = [c for c in list(categorical_features) if c in pp_train.columns and c not in FORBIDDEN]

# Ensure week_sq if expected and exists in the feature list
if "week_sq" in numeric_features and "week_sq" not in pp_train.columns:
    pp_train = pp_train.copy()
    pp_test = pp_test.copy()
    pp_train["week_sq"] = pp_train["week"] ** 2
    pp_test["week_sq"] = pp_test["week"] ** 2
    if "week_sq" not in num_cols:
        num_cols.append("week_sq")

X_cols = num_cols + cat_cols
if len(X_cols) == 0:
    raise ValueError("P11.1: no feature columns available to compute hazard_hat.")

# Predict calibrated hazards
p_tr = hazard_calibrator_rs.predict_proba(pp_train[X_cols])[:, 1].astype(float)
p_te = hazard_calibrator_rs.predict_proba(pp_test[X_cols])[:, 1].astype(float)

# Clip to avoid 0/1 (stability for cumprod later)
p_tr = np.clip(p_tr, 1e-12, 1 - 1e-12)
p_te = np.clip(p_te, 1e-12, 1 - 1e-12)

pp_train["hazard_hat"] = p_tr
pp_test["hazard_hat"] = p_te

# Diagnostics
auc_train = float(roc_auc_score(pp_train["event"].astype(int), pp_train["hazard_hat"]))
auc_test = float(roc_auc_score(pp_test["event"].astype(int), pp_test["hazard_hat"]))

print("=== P11.1 — hazard_hat materialized ===")
print("[Shapes]")
print("pp_train:", pp_train.shape, "| pp_test:", pp_test.shape)

print("\n[hazard_hat range]")
print("train min/max:", float(pp_train["hazard_hat"].min()), float(pp_train["hazard_hat"].max()))
print("test  min/max:", float(pp_test["hazard_hat"].min()), float(pp_test["hazard_hat"].max()))

print("\n[AUC check (should be close to P11-v2)]")
print("AUC_train:", auc_train)
print("AUC_test :", auc_test)

print("=== P11.1 DONE ===")

=== P11.1 — hazard_hat materialized ===
[Shapes]
pp_train: (542878, 28) | pp_test: (232417, 27)

[hazard_hat range]
train min/max: 1e-12 0.9248880777547503
test  min/max: 1e-12 0.6794415803018788

[AUC check (should be close to P11-v2)]
AUC_train: 0.8358500884891436
AUC_test : 0.8395774755908261
=== P11.1 DONE ===


In [942]:
# ==============================================================
# P11.2 - Active hazard selection (baseline vs ipcw_train)
# --------------------------------------------------------------
# Selects which calibrated hazard feeds P12+: baseline (no IPCW) or ipcw_train.
# Preserves hazard_hat_noipcw and sets hazard_calibrator_rs to the active model.
# ==============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

hazard_source = "ipcw_train"  # options: "ipcw_train", "baseline"

required = ["pp_train", "pp_test", "numeric_features", "categorical_features"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P11.2 missing required objects: {missing}")

X_cols = list(numeric_features + categorical_features)


def _ensure_cols(df: pd.DataFrame, cols):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"P11.2: missing columns in df: {miss}")


for df_name in ["pp_train", "pp_test"]:
    df = globals()[df_name]
    _ensure_cols(df, ["hazard_hat"])
    if "hazard_hat_noipcw" not in df.columns:
        df["hazard_hat_noipcw"] = df["hazard_hat"]


def _materialize(col_name: str, calibrator):
    _ensure_cols(pp_train, X_cols)
    _ensure_cols(pp_test, X_cols)
    if (col_name not in pp_train.columns) or (col_name not in pp_test.columns):
        p_tr = calibrator.predict_proba(pp_train[X_cols])[:, 1].astype(float)
        p_te = calibrator.predict_proba(pp_test[X_cols])[:, 1].astype(float)
        pp_train[col_name] = p_tr
        pp_test[col_name] = p_te
    pp_train[col_name] = np.clip(pp_train[col_name].astype(float), 1e-12, 1 - 1e-12)
    pp_test[col_name] = np.clip(pp_test[col_name].astype(float), 1e-12, 1 - 1e-12)


if hazard_source == "ipcw_train":
    if "hazard_calibrator_gcv_ipcwtrain" not in globals():
        raise NameError("P11.2: missing hazard_calibrator_gcv_ipcwtrain (run P7.4-v2).")
    calibrator_sel = hazard_calibrator_gcv_ipcwtrain
    pred_col = "hazard_hat_ipcwtrain"
    _materialize(pred_col, calibrator_sel)
elif hazard_source == "baseline":
    if "hazard_calibrator_gcv" not in globals():
        raise NameError("P11.2: missing hazard_calibrator_gcv (run P11-v2).")
    calibrator_sel = hazard_calibrator_gcv
    pred_col = "hazard_hat_noipcw"
    _materialize(pred_col, calibrator_sel)
else:
    raise ValueError("P11.2: hazard_source must be 'ipcw_train' or 'baseline'.")

pp_train["hazard_hat"] = pp_train[pred_col].astype(float)
pp_test["hazard_hat"] = pp_test[pred_col].astype(float)

hazard_calibrator_active = calibrator_sel
hazard_calibrator_rs = calibrator_sel

auc_train = float(roc_auc_score(pp_train["event"].astype(int), pp_train["hazard_hat"]))
auc_test = float(roc_auc_score(pp_test["event"].astype(int), pp_test["hazard_hat"]))

print(f"=== P11.2 - active hazard: {hazard_source} ===")
print("hazard_hat column :", pred_col)
print("AUC_train (active):", auc_train)
print("AUC_test  (active):", auc_test)
print("pp_train shape    :", pp_train.shape)
print("pp_test shape     :", pp_test.shape)


=== P11.2 - active hazard: ipcw_train ===
hazard_hat column : hazard_hat_ipcwtrain
AUC_train (active): 0.8350153849193052
AUC_test  (active): 0.8404845466472504
pp_train shape    : (542878, 29)
pp_test shape     : (232417, 28)


# P12 — Deterministic Policy Definition (Anchored Recency Trigger)

In [943]:
# ==============================================================
# P12 — Deterministic Policy Definition (Anchored Recency Trigger)
# --------------------------------------------------------------
# Anchored trigger (Kay & Bostock, 2023 — "The Power of the Nudge"):
#   - Flag if no LMS engagement in the last 7 days.
#   - In a weekly person-period frame, this maps to: recency >= 1 week.
#
# Scenario contract introduced here:
#   - The trigger remains anchored and fixed.
#   - Shock intensity is no longer a single scalar; it is a catalog of scenarios.
#   - A reference scenario is kept only for backward compatibility with legacy cells.
#
# Outputs:
#   - pp_test_surv (with logS_hat and S_hat)
#   - policy_trigger (enrollment-level t_star)
#   - pp_policy_base (TEST backbone for policy simulation)
#   - pp_policy (pp_policy_base merged with t_star)
#   - policy_params (trigger metadata + scenario catalog)
#   - shock_scenarios / policy_scenario_catalog / reference_scenario_id
# ==============================================================

import numpy as np
import pandas as pd

# ------------------------------
# 0) Anchored policy threshold
# ------------------------------
r_star = 1  # anchored: 7 days without engagement ≈ recency >= 1 week

DEFAULT_SHOCK_SCENARIOS = [
    {
        "scenario_id": "anchored_conservative",
        "scenario_label": "Conservative anchored",
        "delta_shock_ablation": 0.08,
        "scenario_status": "anchored",
        "scenario_source": "SneyersDeWitte2018_anchored",
        "scenario_note": "Conservative anchored intensity",
    },
    {
        "scenario_id": "hypothetical_A",
        "scenario_label": "Hypothetical A",
        "delta_shock_ablation": 0.20,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_legacy_baseline",
        "scenario_note": "Legacy notebook baseline, now treated as hypothetical",
    },
    {
        "scenario_id": "hypothetical_B",
        "scenario_label": "Hypothetical B",
        "delta_shock_ablation": 0.60,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_extreme_stress_test",
        "scenario_note": "Extreme stress-test intensity",
    },
]
DEFAULT_REFERENCE_SCENARIO_ID = "hypothetical_A"

# ------------------------------
# 1) Defensive checks (match notebook variables)
# ------------------------------
try:
    _ = pp_test
except NameError as e:
    raise NameError("P12 expects `pp_test` to exist (created upstream).") from e

required_cols = KEYS + ["week", "hazard_hat", "total_clicks"]
missing = [c for c in required_cols if c not in pp_test.columns]
if missing:
    raise KeyError(f"P12 requires columns missing from pp_test: {missing}")

pp_test_local = pp_test.copy()
pp_test_local["week"] = pd.to_numeric(pp_test_local["week"], errors="coerce").fillna(0).astype(int)
pp_test_local["total_clicks"] = pd.to_numeric(pp_test_local["total_clicks"], errors="coerce").fillna(0.0).astype(float)

# ------------------------------
# 2) Ensure recency/streak exist (compute here if missing)
# ------------------------------
if ("recency" not in pp_test_local.columns) or ("streak" not in pp_test_local.columns):
    pp_tmp = (
        pp_test_local.sort_values(KEYS + ["week"])
        .reset_index(drop=True)
        .copy()
    )
    pp_tmp["activity_flag"] = (pp_tmp["total_clicks"] > 0).astype(int)

    recency_arr = np.zeros(len(pp_tmp), dtype=int)
    streak_arr = np.zeros(len(pp_tmp), dtype=int)

    for _, g in pp_tmp.groupby(KEYS, sort=False):
        idx = g.index.to_numpy()
        a = g["activity_flag"].to_numpy()

        r = np.zeros(len(a), dtype=int)
        s = np.zeros(len(a), dtype=int)

        for i in range(len(a)):
            if i == 0:
                r[i] = 0 if a[i] == 1 else 1
                s[i] = 1 if a[i] == 1 else 0
            else:
                if a[i] == 1:
                    r[i] = 0
                    s[i] = s[i - 1] + 1
                else:
                    r[i] = r[i - 1] + 1
                    s[i] = 0

        recency_arr[idx] = r
        streak_arr[idx] = s

    pp_tmp["recency"] = recency_arr
    pp_tmp["streak"] = streak_arr
    pp_test_local = pp_tmp.drop(columns=["activity_flag"], errors="ignore")
else:
    pp_test_local["recency"] = pd.to_numeric(pp_test_local["recency"], errors="coerce").fillna(0).astype(int)
    pp_test_local["streak"] = pd.to_numeric(pp_test_local["streak"], errors="coerce").fillna(0).astype(int)

# ------------------------------
# 3) Recompute survival on pp_test (log-space) so it contains recency/streak
# ------------------------------
EPS = 1e-12

def add_log_survival(pp: pd.DataFrame, keys=KEYS, week_col="week", hazard_col="hazard_hat") -> pd.DataFrame:
    pp = pp.sort_values(keys + [week_col]).copy()
    h = pp[hazard_col].astype(float).clip(EPS, 1.0 - EPS)
    pp["log1m_h"] = np.log1p(-h)
    pp["logS_hat"] = pp["log1m_h"].groupby([pp[k] for k in keys]).cumsum()
    pp["S_hat"] = np.exp(pp["logS_hat"])
    return pp

pp_test_surv = add_log_survival(pp_test_local)
pp_policy_base = pp_test_surv.copy()

# ------------------------------
# 4) Compute trigger time t_star per enrollment (anchored r_star)
# ------------------------------
trigger_rows = pp_policy_base.loc[pp_policy_base["recency"] >= r_star].copy()

policy_trigger = (
    trigger_rows.sort_values(KEYS + ["week"])
    .groupby(KEYS, as_index=False)
    .first()[KEYS + ["week"]]
    .rename(columns={"week": "t_star"})
)

pp_policy = pp_policy_base.merge(policy_trigger, on=KEYS, how="left")

# ------------------------------
# 5) Resolve scenario catalog (single source created here)
# ------------------------------
def _normalize_scenario(row: dict) -> dict:
    out = dict(row)
    out["scenario_id"] = str(out["scenario_id"])
    out["scenario_label"] = str(out.get("scenario_label", out["scenario_id"]))
    out["delta_shock_ablation"] = float(out["delta_shock_ablation"])
    out["scenario_status"] = str(out.get("scenario_status", "hypothetical"))
    out["scenario_source"] = str(out.get("scenario_source", "unspecified"))
    out["scenario_note"] = str(out.get("scenario_note", ""))
    return out

shock_scenarios = [_normalize_scenario(row) for row in DEFAULT_SHOCK_SCENARIOS]
scenario_ids = [row["scenario_id"] for row in shock_scenarios]
if len(scenario_ids) != len(set(scenario_ids)):
    raise ValueError(f"P12: scenario_id values must be unique. Got: {scenario_ids}")

reference_scenario_id = DEFAULT_REFERENCE_SCENARIO_ID
if reference_scenario_id not in set(scenario_ids):
    raise ValueError(f"P12: reference_scenario_id={reference_scenario_id} not found in shock_scenarios")

reference_scenario = next(row for row in shock_scenarios if row["scenario_id"] == reference_scenario_id)
policy_scenario_catalog = pd.DataFrame(shock_scenarios)

# ------------------------------
# 6) Define policy_params (created here in this notebook)
# ------------------------------
policy_params = {
    "r_star": int(r_star),
    "trigger_anchor_study": "Kay & Bostock (2023) — The Power of the Nudge: Technology Driving Persistence",
    "trigger_anchor_definition": "flag if no LMS engagement in last 7 days (≈ recency>=1 week)",
    "trigger_frequency": "weekly",
    "recency_definition": "weeks since last active week (activity_flag = 1 if total_clicks>0)",
    "recency_proxy_col": "total_clicks",
    "anchor_study": "Kay & Bostock (2023)",
    "anchor_window_days": 14,
    "anchor_metric_week0": "time_on_task_uplift_day7_35pct",
    "anchor_metric_week1": "uplift_day14_lt_10pct",
    "anchor_trigger_note": "weekly digital re-engagement nudge after 7 days without LMS activity",
    "intervention_class": "short_term_digital_nudge",
    "shock_intensity_anchor_note": "delta=0.08 is an anchored conservative scenario",
    "shock_scenarios": shock_scenarios,
    "reference_scenario_id": reference_scenario_id,
    "reference_delta_shock_ablation": float(reference_scenario["delta_shock_ablation"]),
    "scenario_count": int(len(shock_scenarios)),
}

# ------------------------------
# 7) REQUIRED TEXT OUTPUTS
# ------------------------------
n_triggered = int(len(policy_trigger))
n_total = int(pp_policy_base[KEYS].drop_duplicates().shape[0])

print("=== P12 — Policy Trigger Summary (Anchored to Kay & Bostock, 2023) ===")
print("[Anchors]")
print("trigger_anchor_study:", policy_params["trigger_anchor_study"])
print("trigger rule:", policy_params["trigger_anchor_definition"])
print("r_star:", r_star)

print("\n[Coverage]")
print("Enrollments (test):", n_total)
print("Triggered enrollments:", n_triggered)
print("Trigger rate:", round(n_triggered / max(n_total, 1), 6))

rec = pp_policy_base["recency"].astype(int)
print("\n[Recency distribution (rows)]")
print("min/median/max:", int(rec.min()), int(rec.median()), int(rec.max()))

if n_triggered > 0:
    print("\n[t_star distribution among triggered enrollments]")
    print(
        "min/median/max:",
        int(policy_trigger["t_star"].min()),
        float(policy_trigger["t_star"].median()),
        int(policy_trigger["t_star"].max()),
    )
else:
    print("\nNo enrollments triggered the policy under the current r_star.")

sample = (
    pp_policy.loc[pp_policy["t_star"].notna(), KEYS + ["week", "total_clicks", "recency", "streak", "t_star"]]
    .sort_values(KEYS + ["week"])
)
sample = sample[sample["week"] == sample["t_star"]].head(10)
print("\n[Sample enrollments at trigger week (first 10 rows)]")
print(sample.to_string(index=False))

print("\n[Shock scenario catalog]")
print(policy_scenario_catalog[["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status", "scenario_source"]].to_string(index=False))
print("reference_scenario_id:", reference_scenario_id)
print("reference_delta_shock_ablation:", float(reference_scenario["delta_shock_ablation"]))

=== P12 — Policy Trigger Summary (Anchored to Kay & Bostock, 2023) ===
[Anchors]
trigger_anchor_study: Kay & Bostock (2023) — The Power of the Nudge: Technology Driving Persistence
trigger rule: flag if no LMS engagement in last 7 days (≈ recency>=1 week)
r_star: 1

[Coverage]
Enrollments (test): 9778
Triggered enrollments: 8387
Trigger rate: 0.857742

[Recency distribution (rows)]
min/median/max: 0 0 32

[t_star distribution among triggered enrollments]
min/median/max: 0 4.0 37

[Sample enrollments at trigger week (first 10 rows)]
 id_student code_module code_presentation  week  total_clicks  recency  streak  t_star
       8462         DDD             2013J     9           0.0        1       0     9.0
       8462         DDD             2014J     0           0.0        1       0     0.0
      23629         BBB             2013B     4           0.0        1       0     4.0
      23798         BBB             2013J     7           0.0        1       0     7.0
      24391         GGG    

In [944]:
# ==============================================================
# P13-v7 — Policy Impact (stateful covariate propagation + shock ablation)
# --------------------------------------------------------------
# Purpose:
#   - Recompute counterfactual hazards with stateful covariate propagation
#     for the mechanism-aware policy.
#   - Keep one reference shock scenario for backward compatibility.
#   - Materialize a formal catalog of shock scenarios for downstream exports.
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=== P13-v7 — START (stateful propagation) ===")

# ------------------------------
# 0) Preconditions (fail fast, no guessing)
# ------------------------------
REQ_OBJS = ["pp_policy_base", "policy_trigger", "policy_params", "X_cols", "KEYS"]
missing = [k for k in REQ_OBJS if k not in globals()]
if missing:
    raise NameError(f"P13-v7 missing required objects: {missing}")

pp0 = pp_policy_base.copy()
REQ_COLS = list(KEYS) + ["week", "hazard_hat", "total_clicks", "recency", "streak"]
miss_cols = [c for c in REQ_COLS if c not in pp0.columns]
if miss_cols:
    raise KeyError(f"P13-v7 requires columns missing from pp_policy_base: {miss_cols}")
if "t_star" not in policy_trigger.columns:
    raise KeyError("P13-v7 requires policy_trigger['t_star'] (run P12 trigger first).")

# ------------------------------
# 1) Resolve effect parameters + scenario catalog
# ------------------------------
DEFAULT_EFFECT_SHARED = {
    "W": 2,
    "alpha_week0": 0.35,
    "alpha_week1": 0.10,
    "decay_type": "kb2023_step_2w",
    "window_exclusive_upper": True,
}
DEFAULT_SHOCK_SCENARIOS = [
    {
        "scenario_id": "anchored_conservative",
        "scenario_label": "Conservative anchored",
        "delta_shock_ablation": 0.08,
        "scenario_status": "anchored",
        "scenario_source": "SneyersDeWitte2018_anchored",
        "scenario_note": "Conservative anchored intensity",
    },
    {
        "scenario_id": "hypothetical_A",
        "scenario_label": "Hypothetical A",
        "delta_shock_ablation": 0.20,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_legacy_baseline",
        "scenario_note": "Legacy notebook baseline, now treated as hypothetical",
    },
    {
        "scenario_id": "hypothetical_B",
        "scenario_label": "Hypothetical B",
        "delta_shock_ablation": 0.60,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_extreme_stress_test",
        "scenario_note": "Extreme stress-test intensity",
    },
]

if "policy_spec" in globals() and isinstance(globals().get("policy_spec"), dict):
    eff_shared = dict(globals()["policy_spec"].get("effect_shared", globals()["policy_spec"].get("effect", {})))
    resolved_scenarios = list(globals()["policy_spec"].get("shock_scenarios", []))
    reference_scenario_id = str(globals()["policy_spec"].get("reference_scenario_id", policy_params.get("reference_scenario_id", "hypothetical_A")))
    src_effect = "policy_spec"
elif "policy_effect_params" in globals() and isinstance(globals().get("policy_effect_params"), dict):
    eff_shared = dict(globals()["policy_effect_params"])
    resolved_scenarios = list(policy_params.get("shock_scenarios", globals().get("shock_scenarios", [])))
    reference_scenario_id = str(policy_params.get("reference_scenario_id", globals().get("reference_scenario_id", "hypothetical_A")))
    src_effect = "policy_effect_params + policy_params.shock_scenarios"
else:
    eff_shared = DEFAULT_EFFECT_SHARED.copy()
    resolved_scenarios = list(policy_params.get("shock_scenarios", DEFAULT_SHOCK_SCENARIOS))
    reference_scenario_id = str(policy_params.get("reference_scenario_id", "hypothetical_A"))
    src_effect = "anchored_defaults + policy_params.shock_scenarios"

for key, value in DEFAULT_EFFECT_SHARED.items():
    eff_shared.setdefault(key, value)

def _normalize_scenario(row: dict) -> dict:
    out = dict(row)
    out["scenario_id"] = str(out["scenario_id"])
    out["scenario_label"] = str(out.get("scenario_label", out["scenario_id"]))
    out["delta_shock_ablation"] = float(out["delta_shock_ablation"])
    out["scenario_status"] = str(out.get("scenario_status", "hypothetical"))
    out["scenario_source"] = str(out.get("scenario_source", "unspecified"))
    out["scenario_note"] = str(out.get("scenario_note", ""))
    return out

shock_scenarios = [_normalize_scenario(row) for row in (resolved_scenarios if len(resolved_scenarios) > 0 else DEFAULT_SHOCK_SCENARIOS)]
scenario_ids = [row["scenario_id"] for row in shock_scenarios]
if len(scenario_ids) != len(set(scenario_ids)):
    raise ValueError(f"P13-v7: scenario_id values must be unique. Got: {scenario_ids}")
if reference_scenario_id not in set(scenario_ids):
    raise ValueError(f"P13-v7: reference_scenario_id={reference_scenario_id} not found in shock_scenarios")

reference_scenario = next(row for row in shock_scenarios if row["scenario_id"] == reference_scenario_id)
policy_scenario_catalog = pd.DataFrame(shock_scenarios)

W = int(eff_shared["W"])
alpha_week0 = float(eff_shared["alpha_week0"])
alpha_week1 = float(eff_shared["alpha_week1"])
decay_type = str(eff_shared["decay_type"])
USE_EXCLUSIVE_UPPER = bool(eff_shared["window_exclusive_upper"])
DELTA_SHOCK = float(reference_scenario["delta_shock_ablation"])
EPS = 1e-12

# ------------------------------
# 2) Merge trigger and enforce typing
# ------------------------------
pp0["week"] = pd.to_numeric(pp0["week"], errors="raise").astype(int)
pp0["hazard_hat"] = pd.to_numeric(pp0["hazard_hat"], errors="raise").astype(float).clip(EPS, 1 - EPS)
pp0["total_clicks"] = pd.to_numeric(pp0["total_clicks"], errors="coerce").fillna(0.0).astype(float)
pp0["recency"] = pd.to_numeric(pp0["recency"], errors="coerce").fillna(0).astype(int)
pp0["streak"] = pd.to_numeric(pp0["streak"], errors="coerce").fillna(0).astype(int)

pp = pp0.merge(policy_trigger[list(KEYS) + ["t_star"]], on=KEYS, how="left", validate="m:1")
n_total_enr = int(pp[list(KEYS)].drop_duplicates().shape[0])
n_trig_enr = int(pp.loc[pp["t_star"].notna(), list(KEYS)].drop_duplicates().shape[0])
print(f"[P13-v7][OK] Injected t_star into pp_policy_base. Enrollments with a trigger: {n_trig_enr} / {n_total_enr}")

# ------------------------------
# 3) Identify the hazard model that reproduces hazard_hat (NO GUESS)
# ------------------------------
CAND_NAMES = [
    "hazard_model",
    "hazard_calibrator_rs",
    "hazard_calibrator_gcv",
    "hazard_scorer",
    "gb_model_cal",
    "hazard_model_ipcw_fitted",
    "hazard_model_ipcw",
]

cand = []
for name in CAND_NAMES:
    if name in globals():
        obj = globals()[name]
        if hasattr(obj, "predict_proba") and callable(getattr(obj, "predict_proba")):
            cand.append((name, obj))
for key in list(globals().keys()):
    obj = globals()[key]
    if key in [x[0] for x in cand]:
        continue
    if hasattr(obj, "predict_proba") and callable(getattr(obj, "predict_proba")):
        cand.append((key, obj))
if len(cand) == 0:
    raise NameError("P13-v7 could not find any fitted model with predict_proba() in globals().")

pp_sample = pp.sample(n=min(20000, len(pp)), random_state=42).copy()
X_cols_local = list(X_cols)
miss_X = [c for c in X_cols_local if c not in pp_sample.columns]
if miss_X:
    raise KeyError(f"P13-v7: X_cols missing from pp_policy_base: {miss_X}")

y_ref = pd.to_numeric(pp_sample["hazard_hat"], errors="coerce").astype(float).to_numpy()
if np.isnan(y_ref).any():
    raise ValueError("P13-v7: hazard_hat has NaNs in sample; cannot match models.")

def _prepare_X(model, Xdf, cols):
    base = Xdf.loc[:, cols]
    if hasattr(model, "feature_names_in_"):
        try:
            return base.loc[:, model.feature_names_in_]
        except Exception:
            return base
    return base.to_numpy()

def _pred(model, Xdf):
    X_use = _prepare_X(model, Xdf, X_cols_local)
    p = model.predict_proba(X_use)[:, 1].astype(float)
    return np.clip(p, EPS, 1 - EPS)

scores = []
for name, model in cand:
    try:
        p = _pred(model, pp_sample[X_cols_local])
        mae = float(np.mean(np.abs(p - y_ref)))
        mx = float(np.max(np.abs(p - y_ref)))
        scores.append((mae, mx, name, model))
    except Exception:
        scores.append((np.inf, np.inf, name, None))

scores_sorted = sorted(scores, key=lambda t: (t[0], t[1]))
best_mae, best_mx, best_name, best_model = scores_sorted[0]
if best_model is None or not np.isfinite(best_mae):
    bad = [(mae, mx, name) for (mae, mx, name, model) in scores_sorted[:10]]
    raise RuntimeError(f"P13-v7: No candidate model could score predictions. Top attempts: {bad}")

print("\n[P13-v7][MODEL MATCH]")
print("Selected model name:", best_name)
print("MAE(pred, hazard_hat) on sample:", best_mae)
print("MAX|pred-hazard_hat| on sample:", best_mx)

# ------------------------------
# 4) Compute policy_active + boost_factor
# ------------------------------
pp["hazard_0"] = pd.to_numeric(pp["hazard_hat"], errors="raise").astype(float).clip(EPS, 1 - EPS)

if USE_EXCLUSIVE_UPPER:
    pp["policy_active"] = (
        pp["t_star"].notna()
        & (pp["week"] >= pp["t_star"])
        & (pp["week"] < (pp["t_star"] + W))
    ).astype(int)
else:
    pp["policy_active"] = (
        pp["t_star"].notna()
        & (pp["week"] >= pp["t_star"])
        & (pp["week"] <= (pp["t_star"] + W))
    ).astype(int)

if "event" in pp.columns:
    pp.loc[pd.to_numeric(pp["event"], errors="coerce").fillna(0).astype(int) == 1, "policy_active"] = 0

active_mask = pp["policy_active"].astype(int) == 1
w_int = pd.to_numeric(pp["week"] - pp["t_star"], errors="coerce").fillna(-999).astype(int)

bf = pd.Series(1.0, index=pp.index, dtype=float)
if decay_type == "kb2023_step_2w":
    bf.loc[active_mask & (w_int == 0)] = 1.0 + alpha_week0
    bf.loc[active_mask & (w_int == 1)] = 1.0 + alpha_week1
elif decay_type == "flat":
    bf.loc[active_mask] = 1.0 + alpha_week0
else:
    raise ValueError(f"P13-v7: unsupported decay_type='{decay_type}'")
pp["boost_factor"] = bf

# ------------------------------
# 5) Stateful counterfactual covariates
# ------------------------------
pp["total_clicks_base"] = pp["total_clicks"].astype(float)
pp["recency_base"] = pp["recency"].astype(int)
pp["streak_base"] = pp["streak"].astype(int)

pp["total_clicks_cf"] = pp["total_clicks_base"]
pp.loc[active_mask, "total_clicks_cf"] = (
    pp.loc[active_mask, "total_clicks_base"] * pp.loc[active_mask, "boost_factor"]
).astype(float)

pp["activity_cf"] = (pp["total_clicks_cf"] > 0).astype(int)
pp = pp.sort_values(list(KEYS) + ["week"]).reset_index(drop=True)
rec_cf = np.zeros(len(pp), dtype=int)
str_cf = np.zeros(len(pp), dtype=int)

for _, g in pp.groupby(KEYS, sort=False):
    idx = g.index.to_numpy()
    a = g["activity_cf"].to_numpy(dtype=int)
    r = np.zeros(len(a), dtype=int)
    s = np.zeros(len(a), dtype=int)
    for i in range(len(a)):
        if i == 0:
            r[i] = 0 if a[i] == 1 else 1
            s[i] = 1 if a[i] == 1 else 0
        else:
            if a[i] == 1:
                r[i] = 0
                s[i] = s[i - 1] + 1
            else:
                r[i] = r[i - 1] + 1
                s[i] = 0
    rec_cf[idx] = r
    str_cf[idx] = s

pp["recency_cf"] = rec_cf
pp["streak_cf"] = str_cf

# ------------------------------
# 6) Plug-in recomputation of hazard under counterfactual covariates
# ------------------------------
Xcf = pp[X_cols_local].copy()
if "total_clicks" in X_cols_local:
    Xcf["total_clicks"] = pp["total_clicks_cf"].astype(float)
if "recency" in X_cols_local:
    Xcf["recency"] = pp["recency_cf"].astype(int)
if "streak" in X_cols_local:
    Xcf["streak"] = pp["streak_cf"].astype(int)
if "week_sq" in X_cols_local and "week_sq" not in Xcf.columns:
    Xcf["week_sq"] = pp["week"].astype(int) ** 2

pp["hazard_1_mech"] = _pred(best_model, Xcf)
pp["hazard_1_shock"] = (pp["hazard_0"] * (1.0 - DELTA_SHOCK)).astype(float).clip(EPS, 1 - EPS)
pp["reference_scenario_id"] = reference_scenario_id
pp["reference_delta_shock_ablation"] = DELTA_SHOCK

# ------------------------------
# 7) Covariate deltas/overwrites + exports
# ------------------------------
OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

pp["delta_total_clicks"] = pp["total_clicks_cf"] - pp["total_clicks_base"]
pp["delta_recency"] = pp["recency_cf"] - pp["recency_base"]
pp["delta_streak"] = pp["streak_cf"] - pp["streak_base"]

is_changed_clicks = np.abs(pp["delta_total_clicks"]) > 1e-9
is_changed_rec = pp["delta_recency"] != 0
is_changed_str = pp["delta_streak"] != 0

agg = pp.groupby("week", as_index=False).agg({
    "policy_active": "mean",
    "delta_total_clicks": "mean",
    "delta_recency": "mean",
    "delta_streak": "mean",
    "hazard_0": "size",
    "t_star": lambda x: x.notna().mean(),
})
agg = agg.rename(columns={"policy_active": "policy_active_rate", "hazard_0": "n_rows", "t_star": "trigger_rate"})
p_deltas = os.path.join(TABLE_DIR, "table_policy_covariate_deltas_by_week.csv")
agg.to_csv(p_deltas, index=False)

propagation_mask = (~active_mask) & (is_changed_rec | is_changed_str | is_changed_clicks)
row_over = {
    "rows_total": int(len(pp)),
    "rows_active": int(active_mask.sum()),
    "rows_changed_total_clicks": int(is_changed_clicks.sum()),
    "rows_changed_recency": int(is_changed_rec.sum()),
    "rows_changed_streak": int(is_changed_str.sum()),
    "rows_changed_nonactive": int(propagation_mask.sum()),
    "frac_changed_total_clicks": float(is_changed_clicks.mean()),
    "frac_changed_recency": float(is_changed_rec.mean()),
    "frac_changed_streak": float(is_changed_str.mean()),
    "frac_changed_nonactive": float(propagation_mask.mean()),
}
df_over = pd.DataFrame([row_over])
p_over = os.path.join(TABLE_DIR, "table_policy_covariate_overwrites.csv")
df_over.to_csv(p_over, index=False)

row_act = {
    "effect_params_source": src_effect,
    "W": W,
    "alpha_week0": alpha_week0,
    "alpha_week1": alpha_week1,
    "decay_type": decay_type,
    "window_exclusive_upper": int(USE_EXCLUSIVE_UPPER),
    "delta_shock_ablation": DELTA_SHOCK,
    "reference_scenario_id": reference_scenario_id,
    "scenario_count": int(len(shock_scenarios)),
    "n_enrollments_total": n_total_enr,
    "n_enrollments_triggered": n_trig_enr,
    "trigger_rate_enrollment": float(n_trig_enr / max(n_total_enr, 1)),
    "n_rows_total": int(len(pp)),
    "n_rows_active": int(active_mask.sum()),
    "active_rate_rows": float(active_mask.mean()),
}
df_act = pd.DataFrame([row_act])
p_act = os.path.join(TABLE_DIR, "table_policy_activation_summary.csv")
df_act.to_csv(p_act, index=False)

# Keep legacy catalog path and create canonical scenario main table.
p_catalog = os.path.join(TABLE_DIR, "table_policy_shock_scenarios.csv")
p_catalog_main = os.path.join(TABLE_DIR, "table_policy_scenarios_main.csv")
policy_scenario_catalog.to_csv(p_catalog, index=False)
policy_scenario_catalog.to_csv(p_catalog_main, index=False)

p_fig_cov = os.path.join(FIG_DIR, "fig_policy_covariate_deltas_by_week.png")
plt.figure()
plt.plot(agg["week"], agg["delta_total_clicks"], marker="o", label="Δ total_clicks")
plt.plot(agg["week"], agg["delta_recency"], marker="o", label="Δ recency")
plt.plot(agg["week"], agg["delta_streak"], marker="o", label="Δ streak")
plt.axvline(W, linestyle=":", label="W")
plt.xlabel("Week")
plt.ylabel("Mean Δ (cf - base)")
plt.title("Policy-induced covariate deltas by week (stateful)")
plt.legend()
plt.tight_layout()
plt.savefig(p_fig_cov, dpi=200)
plt.close()

# Export pp_policy for downstream (P14 expects pp_policy with hazards)
pp_policy = pp.copy()

# ------------------------------
# 8) REQUIRED TEXT OUTPUTS (paste back)
# ------------------------------
n_active_rows = int(active_mask.sum())
n_rows = int(len(pp))
n_active_enr = int(pp.loc[active_mask, list(KEYS)].drop_duplicates().shape[0])

print("\n=== P13-v7 — Policy Impact Summary ===")
print("[Trigger anchor]")
print("anchor_study:", policy_params.get("trigger_anchor_study"))
print("trigger_rule:", policy_params.get("trigger_anchor_definition"))
print("r_star:", policy_params.get("r_star"))

print("\n[Effect params — shared]")
print("source:", src_effect)
print("W (weeks):", W)
print("alpha_week0:", alpha_week0, "| alpha_week1:", alpha_week1)
print("decay_type:", decay_type)
print("window_exclusive_upper:", USE_EXCLUSIVE_UPPER)
print("reference_scenario_id:", reference_scenario_id)
print("reference_delta_shock_ablation:", DELTA_SHOCK)

print("\n[Shock scenarios]")
print(policy_scenario_catalog[["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status", "scenario_source"]].to_string(index=False))

print("\n[Activation]")
print(f"Rows with policy_active=1: {n_active_rows} / {n_rows} ({n_active_rows/max(n_rows,1):.6f})")
print(f"Enrollments affected at least once: {n_active_enr} / {n_total_enr} ({n_active_enr/max(n_total_enr,1):.6f})")

if n_active_rows > 0:
    bf_act = pp.loc[active_mask, "boost_factor"].astype(float)
    print("\n[Boost factor — active rows]")
    print("boost_factor mean:", float(bf_act.mean()))
    print("boost_factor min :", float(bf_act.min()))
    print("boost_factor max :", float(bf_act.max()))

print("\n[Hazard ranges]")
print("hazard_0: min={:.12f} max={:.12f}".format(float(pp["hazard_0"].min()), float(pp["hazard_0"].max())))
print("hazard_1_shock: min={:.12f} max={:.12f}".format(float(pp["hazard_1_shock"].min()), float(pp["hazard_1_shock"].max())))
print("hazard_1_mech:  min={:.12f} max={:.12f}".format(float(pp["hazard_1_mech"].min()), float(pp["hazard_1_mech"].max())))

if n_active_rows > 0:
    h0 = pp.loc[active_mask, "hazard_0"].astype(float)
    hs = pp.loc[active_mask, "hazard_1_shock"].astype(float)
    hm = pp.loc[active_mask, "hazard_1_mech"].astype(float)
    print("\n[Active-only mean hazard]")
    print("mean(hazard_0)      :", float(h0.mean()))
    print("mean(hazard_1_shock):", float(hs.mean()), "| mean ratio shock:", float((hs / h0).mean()))
    print("mean(hazard_1_mech) :", float(hm.mean()), "| mean ratio mech :", float((hm / h0).mean()))

print("\n[Propagation checks]")
print("rows changed (total_clicks):", int(is_changed_clicks.sum()))
print("rows changed (recency)     :", int(is_changed_rec.sum()))
print("rows changed (streak)      :", int(is_changed_str.sum()))
print("rows changed outside active:", int(propagation_mask.sum()))

print("\n[Exports]")
print("activation summary :", p_act)
print("scenario catalog   :", p_catalog)
print("scenario main table:", p_catalog_main)
print("overwrites summary :", p_over)
print("deltas by week     :", p_deltas)
print("figure deltas      :", p_fig_cov)
print("\n=== P13-v7 DONE ===")

=== P13-v7 — START (stateful propagation) ===
[P13-v7][OK] Injected t_star into pp_policy_base. Enrollments with a trigger: 8387 / 9778

[P13-v7][MODEL MATCH]
Selected model name: best_model
MAE(pred, hazard_hat) on sample: 0.0009553749332388387
MAX|pred-hazard_hat| on sample: 0.09761850302771424
[TABLE] saved: outputs_v2/tables/table_policy_covariate_deltas_by_week.csv | shape=(39, 7)
columns: ['week', 'policy_active_rate', 'delta_total_clicks', 'delta_recency', 'delta_streak', 'n_rows', 'trigger_rate']
sample (n=5):
   week  policy_active_rate  delta_total_clicks  delta_recency  delta_streak  \
0     0            0.165883            0.000000            0.0           0.0   
1     1            0.205872            0.119247            0.0           0.0   
2     2            0.189219            0.481188            0.0           0.0   
3     3            0.130413            0.165244            0.0           0.0   
4     4            0.116641            0.155180            0.0           0.0

In [945]:
# ==============================================================
# P11.2 - Robust calibration tail check (eligibility-rule reporting)
# --------------------------------------------------------------
# Goal:
#   - Keep the original P11 figure untouched.
#   - Add an auditable robust view for high-risk tail instability with
#     explicit bin eligibility criteria (no cosmetic smoothing).
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

required = ["y_test", "p_test", "TABLE_DIR", "FIG_DIR"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P11.2 missing required objects: {missing}. Run P11-v2 first.")

def _wilson_ci(k: int, n: int, z: float = 1.96):
    if n <= 0:
        return np.nan, np.nan
    phat = k / n
    denom = 1.0 + (z ** 2) / n
    center = (phat + (z ** 2) / (2.0 * n)) / denom
    half = (z / denom) * np.sqrt((phat * (1.0 - phat) / n) + ((z ** 2) / (4.0 * n * n)))
    lo = max(0.0, center - half)
    hi = min(1.0, center + half)
    return float(lo), float(hi)

def _eligible(n: int, positives: int, min_n: int, min_pos: int, min_neg: int) -> bool:
    negatives = int(n - positives)
    return (int(n) >= int(min_n)) and (int(positives) >= int(min_pos)) and (negatives >= int(min_neg))

def calibration_bins_quantile(y_true: np.ndarray, p: np.ndarray, n_bins: int = 12) -> pd.DataFrame:
    y = y_true.astype(int)
    p = np.clip(p.astype(float), 0.0, 1.0)

    q = pd.qcut(p, q=n_bins, labels=False, duplicates="drop")
    if pd.isna(q).all():
        edges = np.linspace(0.0, 1.0, n_bins + 1)
        q = np.digitize(p, edges, right=False) - 1
        q = np.clip(q, 0, n_bins - 1)
        q = pd.Series(q)
    else:
        q = pd.Series(q.astype(int))

    rows = []
    for b in sorted(q.dropna().unique()):
        mask = (q == b).to_numpy()
        n = int(mask.sum())
        if n <= 0:
            continue
        positives = int(y[mask].sum())
        negatives = int(n - positives)
        mean_pred = float(np.mean(p[mask]))
        emp_rate = float(positives / n)
        abs_gap = float(abs(mean_pred - emp_rate))
        p_min = float(np.min(p[mask]))
        p_max = float(np.max(p[mask]))
        ci_low, ci_high = _wilson_ci(positives, n)
        rows.append({
            "bin": int(b),
            "n": n,
            "positives": positives,
            "negatives": negatives,
            "p_min": p_min,
            "p_max": p_max,
            "mean_pred": mean_pred,
            "empirical_rate": emp_rate,
            "abs_gap": abs_gap,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
        })

    out = pd.DataFrame(rows).sort_values("mean_pred").reset_index(drop=True)
    out["bin"] = np.arange(len(out), dtype=int)
    return out

def merge_sparse_tail_bins_by_eligibility(
    df_bins: pd.DataFrame,
    min_n: int = 80,
    min_pos: int = 5,
    min_neg: int = 5,
 ) -> pd.DataFrame:
    if df_bins.empty:
        return df_bins.copy()

    work = df_bins.copy().sort_values("mean_pred").reset_index(drop=True)

    # Merge only from the high-risk tail until the tail bin is inferentially reportable.
    while len(work) > 1:
        last = work.iloc[-1]
        if _eligible(last["n"], last["positives"], min_n=min_n, min_pos=min_pos, min_neg=min_neg):
            break

        a = work.iloc[-2].copy()
        b = work.iloc[-1].copy()
        n_new = int(a["n"] + b["n"])
        pos_new = int(a["positives"] + b["positives"])
        neg_new = int(n_new - pos_new)

        mean_pred_new = float((a["mean_pred"] * a["n"] + b["mean_pred"] * b["n"]) / n_new)
        emp_rate_new = float(pos_new / n_new)
        abs_gap_new = float(abs(mean_pred_new - emp_rate_new))
        ci_low_new, ci_high_new = _wilson_ci(pos_new, n_new)

        merged = {
            "bin": int(a["bin"]),
            "n": n_new,
            "positives": pos_new,
            "negatives": neg_new,
            "p_min": float(min(a["p_min"], b["p_min"])),
            "p_max": float(max(a["p_max"], b["p_max"])),
            "mean_pred": mean_pred_new,
            "empirical_rate": emp_rate_new,
            "abs_gap": abs_gap_new,
            "ci95_low": ci_low_new,
            "ci95_high": ci_high_new,
        }

        work = pd.concat([work.iloc[:-2], pd.DataFrame([merged])], ignore_index=True)
        work = work.sort_values("mean_pred").reset_index(drop=True)

    work["bin"] = np.arange(len(work), dtype=int)
    return work

def ece_from_bins(df_bins: pd.DataFrame) -> float:
    n_total = float(df_bins["n"].sum())
    if n_total <= 0:
        return np.nan
    return float(((df_bins["n"] / n_total) * df_bins["abs_gap"]).sum())

# Eligibility rule (explicit and auditable).
n_bins_robust = 12
min_n_bin = 80
min_positives = 5
min_negatives = 5

bins_q_raw = calibration_bins_quantile(y_test, p_test, n_bins=n_bins_robust)
bins_q_merged = merge_sparse_tail_bins_by_eligibility(
    bins_q_raw,
    min_n=min_n_bin,
    min_pos=min_positives,
    min_neg=min_negatives,
)

bins_q_merged["eligible"] = bins_q_merged.apply(
    lambda r: _eligible(
        int(r["n"]),
        int(r["positives"]),
        min_n=min_n_bin,
        min_pos=min_positives,
        min_neg=min_negatives,
    ),
    axis=1,
 )
bins_q_merged["support_status"] = bins_q_merged["eligible"].map({True: "eligible", False: "insufficient_support"})

# Only eligible bins are used for rate-reporting and reliability visualization.
bins_q_report = bins_q_merged.loc[bins_q_merged["eligible"]].copy().reset_index(drop=True)
ece_eligible = ece_from_bins(bins_q_report)

bins_q_raw_csv = os.path.join(TABLE_DIR, "table_calibration_bins_test_quantile_raw.csv")
bins_q_robust_csv = os.path.join(TABLE_DIR, "table_calibration_bins_test_robust.csv")
bins_q_report_csv = os.path.join(TABLE_DIR, "table_calibration_bins_test_robust_reportable.csv")
metrics_robust_csv = os.path.join(TABLE_DIR, "table_calibration_metrics_robust.csv")

bins_q_raw.to_csv(bins_q_raw_csv, index=False)
bins_q_merged.to_csv(bins_q_robust_csv, index=False)
bins_q_report.to_csv(bins_q_report_csv, index=False)

pd.DataFrame(
    [
        ("n_bins_quantile_raw", float(n_bins_robust)),
        ("min_n_bin", float(min_n_bin)),
        ("min_positives", float(min_positives)),
        ("min_negatives", float(min_negatives)),
        ("n_bins_after_tail_merge", float(len(bins_q_merged))),
        ("n_bins_reportable", float(len(bins_q_report))),
        ("n_bins_insufficient_support", float((~bins_q_merged["eligible"]).sum())),
        ("ECE_test_reportable_bins", ece_eligible),
    ],
    columns=["metric", "value"],
).to_csv(metrics_robust_csv, index=False)

plt.figure(figsize=(7.0, 5.2))
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, color="#ff7f0e", label="Ideal calibration (y = x)")

if len(bins_q_report) > 0:
    x = bins_q_report["mean_pred"].to_numpy(dtype=float)
    y = bins_q_report["empirical_rate"].to_numpy(dtype=float)
    yerr = np.vstack([
        y - bins_q_report["ci95_low"].to_numpy(dtype=float),
        bins_q_report["ci95_high"].to_numpy(dtype=float) - y,
    ])
    sizes = 18.0 + 1.6 * np.sqrt(bins_q_report["n"].to_numpy(dtype=float))

    plt.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="none",
        ecolor="#1f77b4",
        elinewidth=1.2,
        capsize=3,
        alpha=0.9,
        label="Wilson CI95 (reportable bins)",
    )
    plt.scatter(
        x,
        y,
        s=sizes,
        color="#1f77b4",
        alpha=0.9,
        label="Reportable bins (size ~ sqrt(n))",
    )

    for _, r in bins_q_report.iterrows():
        plt.annotate(
            f"n={int(r['n'])}",
            (r["mean_pred"], r["empirical_rate"]),
            textcoords="offset points",
            xytext=(4, 4),
            fontsize=7,
            alpha=0.85,
        )

plt.xlabel("Mean predicted probability (bin)")
plt.ylabel("Empirical event rate (bin)")
plt.title("Calibration (Reliability) - TEST (reportable bins only)")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.gca().set_aspect("equal", adjustable="box")
plt.grid(True, linestyle=":", linewidth=0.6, alpha=0.35)
plt.legend(loc="best", frameon=True, fontsize=8)
plt.tight_layout()
plt.close()

print("=== P11.2 - Robust calibration tail check ===")
print("Rule: reportable bin iff n>=80, positives>=5, negatives>=5.")
print("Raw quantile bins      :", len(bins_q_raw))
print("After tail merge       :", len(bins_q_merged))
print("Reportable bins        :", len(bins_q_report))
print("Insufficient-support   :", int((~bins_q_merged["eligible"]).sum()))
print("ECE (reportable bins)  :", ece_eligible)
print("bins_raw_csv           :", bins_q_raw_csv)
print("bins_robust_csv        :", bins_q_robust_csv)
print("bins_reportable_csv    :", bins_q_report_csv)
print("metrics_robust_csv     :", metrics_robust_csv)
print("=== P11.2 DONE ===")

[TABLE] saved: outputs_v2/tables/table_calibration_bins_test_quantile_raw.csv | shape=(12, 11)
columns: ['bin', 'n', 'positives', 'negatives', 'p_min', 'p_max', 'mean_pred', 'empirical_rate', 'abs_gap', 'ci95_low', 'ci95_high']
sample (n=5):
   bin      n  positives  negatives         p_min     p_max  mean_pred  \
0    0  19369          6      19363  6.840772e-13  0.000571   0.000248   
1    1  19367         11      19356  5.710470e-04  0.001301   0.000924   
2    2  19369         19      19350  1.301139e-03  0.002097   0.001700   
3    3  19367         34      19333  2.097342e-03  0.002935   0.002505   
4    4  19368         34      19334  2.935197e-03  0.003900   0.003410   

   empirical_rate   abs_gap  ci95_low  ci95_high  
0        0.000310  0.000061  0.000142   0.000676  
1        0.000568  0.000356  0.000317   0.001017  
2        0.000981  0.000719  0.000628   0.001532  
3        0.001756  0.000750  0.001257   0.002452  
4        0.001755  0.001654  0.001257   0.002452  
[TABLE]

In [946]:
# ==============================================================
# P11.3 - Calibration support table for paper (simple, no new figure)
# --------------------------------------------------------------
# Goal:
#   Export an auditable support table for the same bins used by the
#   main reliability figure in P11-v2, with a compact high-risk tail view.
# Outputs:
#   - outputs_v2/tables/table_calibration_bin_support_test.csv
#   - outputs_v2/tables/table_calibration_bin_support_tail3_test.csv
# ==============================================================

import os
import numpy as np
import pandas as pd

required = ["TABLE_DIR"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(
        f"P11.3 missing required objects: {missing}. "
        "Run P11-v2 first."
)

if "bins_df" in globals() and isinstance(bins_df, pd.DataFrame):
    calib_bins = bins_df.copy()
    source_name = "in-memory bins_df (P11-v2)"
else:
    bins_csv = os.path.join(TABLE_DIR, "table_calibration_bins_test.csv")
    if not os.path.exists(bins_csv):
        raise FileNotFoundError(
            "P11.3 could not find calibration bins table. "
            "Run P11-v2 or provide outputs_v2/tables/table_calibration_bins_test.csv."
)
    calib_bins = pd.read_csv(bins_csv)
    source_name = bins_csv

required_cols = ["bin", "n", "p_min", "p_max", "mean_pred", "empirical_rate"]
missing_cols = [c for c in required_cols if c not in calib_bins.columns]
if missing_cols:
    raise KeyError(f"P11.3 requires columns missing from calibration bins table: {missing_cols}")

support = calib_bins.loc[calib_bins["n"].fillna(0) > 0, required_cols].copy()
if support.empty:
    raise ValueError("P11.3 found no non-empty calibration bins to report.")

support["bin"] = pd.to_numeric(support["bin"], errors="raise").astype(int)
support["n"] = pd.to_numeric(support["n"], errors="raise").astype(int)
support["mean_pred"] = pd.to_numeric(support["mean_pred"], errors="raise").astype(float)
support["empirical_rate"] = pd.to_numeric(support["empirical_rate"], errors="raise").astype(float)
support["p_min"] = pd.to_numeric(support["p_min"], errors="raise").astype(float)
support["p_max"] = pd.to_numeric(support["p_max"], errors="raise").astype(float)

support["events"] = np.rint(support["empirical_rate"] * support["n"]).astype(int)
support["events"] = np.minimum(np.maximum(support["events"], 0), support["n"])
support["non_events"] = support["n"] - support["events"]
support["abs_gap"] = (support["mean_pred"] - support["empirical_rate"]).abs()

support["risk_range"] = support.apply(
    lambda r: f"[{r['p_min']:.3f}, {r['p_max']:.3f})",
    axis=1,
)

support = support.sort_values("bin").reset_index(drop=True)
support_simple = support[[
    "bin",
    "risk_range",
    "n",
    "events",
    "non_events",
    "mean_pred",
    "empirical_rate",
    "abs_gap",
]].copy()

for c in ["mean_pred", "empirical_rate", "abs_gap"]:
    support_simple[c] = support_simple[c].round(4)

tail_k = 3
support_tail = support_simple.tail(tail_k).reset_index(drop=True)

support_csv = os.path.join(TABLE_DIR, "table_calibration_bin_support_test.csv")
support_tail_csv = os.path.join(TABLE_DIR, "table_calibration_bin_support_tail3_test.csv")

support_simple.to_csv(support_csv, index=False)
support_tail.to_csv(support_tail_csv, index=False)

last = support_simple.iloc[-1]

print("=== P11.3 - Calibration support table for paper ===")
print("source_table        :", source_name)
print("full_support_csv    :", support_csv)
print("tail_support_csv    :", support_tail_csv)
print("tail_bins_reported  :", int(len(support_tail)))
print(
    "last_bin_support    :",
    f"bin={int(last['bin'])}, n={int(last['n'])}, events={int(last['events'])},",
    f"non_events={int(last['non_events'])}, mean_pred={float(last['mean_pred']):.4f},",
    f"empirical_rate={float(last['empirical_rate']):.4f}"
)
print("=== P11.3 DONE ===")

[TABLE] saved: outputs_v2/tables/table_calibration_bin_support_test.csv | shape=(11, 8)
columns: ['bin', 'risk_range', 'n', 'events', 'non_events', 'mean_pred', 'empirical_rate', 'abs_gap']
sample (n=5):
   bin      risk_range       n  events  non_events  mean_pred  empirical_rate  \
0    0  [0.000, 0.067)  230157    2008      228149     0.0081          0.0087   
1    1  [0.067, 0.133)    1499     145        1354     0.0877          0.0967   
2    2  [0.133, 0.200)     304      19         285     0.1634          0.0625   
3    3  [0.200, 0.267)     180      10         170     0.2321          0.0556   
4    4  [0.267, 0.333)     104       8          96     0.2963          0.0769   

   abs_gap  
0   0.0006  
1   0.0091  
2   0.1009  
3   0.1766  
4   0.2194  
[TABLE] saved: outputs_v2/tables/table_calibration_bin_support_tail3_test.csv | shape=(3, 8)
columns: ['bin', 'risk_range', 'n', 'events', 'non_events', 'mean_pred', 'empirical_rate', 'abs_gap']
sample (n=3):
   bin      risk_range

# P14 — Counterfactual Trajectories and Structural Contrast 

In [948]:
# ==============================================================
# P14-v4.0 — Trajectories Builder (SOURCE: pp_policy; NO pph dependency)
# --------------------------------------------------------------
# Purpose:
#   Materialize canonical policy trajectories while promoting shock intensity
#   from a single scalar to a formal scenario catalog.
#
# Outputs:
#   - pp_traj_multi  : row-level trajectories by scenario
#   - weekly_multi   : week-level mean survival and deltaS by scenario
#   - horizons_multi : horizon table by scenario
#   - pp_traj / weekly / horizons : reference-scenario aliases for backward compatibility
# ==============================================================

import numpy as np
import pandas as pd

print("=== P14-v4.0 — START (build pp_traj/weekly/horizons) ===")

required_globals = ["KEYS", "pp_policy", "enrollments_eventtime", "enroll_test_keys"]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise NameError(f"P14-v4.0 missing required globals: {missing}")

req_pp_cols = list(KEYS) + ["week", "hazard_0", "hazard_1_mech"]
miss_pp = [c for c in req_pp_cols if c not in pp_policy.columns]
if miss_pp:
    raise KeyError(f"P14-v4.0 requires columns missing from pp_policy: {miss_pp}")

req_enr_cols = list(KEYS) + ["t_final_week"]
miss_enr = [c for c in req_enr_cols if c not in enrollments_eventtime.columns]
if miss_enr:
    raise KeyError(f"P14-v4.0 requires columns missing from enrollments_eventtime: {miss_enr}")

if "policy_scenario_catalog" in globals() and isinstance(globals().get("policy_scenario_catalog"), pd.DataFrame):
    scenario_catalog = globals()["policy_scenario_catalog"].copy()
else:
    scenario_catalog = pd.DataFrame(globals().get("shock_scenarios", policy_params.get("shock_scenarios", [])))

if scenario_catalog.empty:
    raise ValueError("P14-v4.0 requires a non-empty shock scenario catalog.")

required_scen_cols = ["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status", "scenario_source"]
miss_scen = [c for c in required_scen_cols if c not in scenario_catalog.columns]
if miss_scen:
    raise KeyError(f"P14-v4.0 scenario catalog missing columns: {miss_scen}")

reference_scenario_id = str(globals().get("reference_scenario_id", policy_params.get("reference_scenario_id", scenario_catalog.iloc[0]["scenario_id"])))
if reference_scenario_id not in set(scenario_catalog["scenario_id"].astype(str)):
    raise ValueError(f"P14-v4.0: reference_scenario_id={reference_scenario_id} not present in scenario catalog")

T_policy = 18

test_enr = (
    enrollments_eventtime
    .merge(enroll_test_keys, on=KEYS, how="inner")
    .loc[:, list(KEYS) + ["t_final_week"]]
    .drop_duplicates(subset=KEYS)
    .copy()
)

test_enr["t_final_week"] = pd.to_numeric(test_enr["t_final_week"], errors="raise").astype(int)
T_eval = int(test_enr["t_final_week"].max())
if T_policy > T_eval:
    raise ValueError(f"P14-v4.0: T_policy={T_policy} exceeds TEST support T_eval={T_eval}")

EPS = 1e-12
df = pp_policy.loc[:, list(KEYS) + ["week", "hazard_0", "hazard_1_mech"]].copy()
df["week"] = pd.to_numeric(df["week"], errors="raise").astype(int)
for c in ["hazard_0", "hazard_1_mech"]:
    df[c] = pd.to_numeric(df[c], errors="raise").astype(float).clip(EPS, 1 - EPS)

df = df.merge(test_enr, on=KEYS, how="inner", validate="m:1")
df = df.loc[df["week"] <= df["t_final_week"]].copy()
df = df.sort_values(list(KEYS) + ["week"]).reset_index(drop=True)
gkey = df[list(KEYS)].apply(tuple, axis=1)

df["S0_hat"] = (1.0 - df["hazard_0"]).groupby(gkey).cumprod().astype(float).clip(EPS, 1.0)
df["S_mech_hat"] = (1.0 - df["hazard_1_mech"]).groupby(gkey).cumprod().astype(float).clip(EPS, 1.0)

traj_parts = []
for row in scenario_catalog.to_dict(orient="records"):
    delta = float(row["delta_shock_ablation"])
    tmp = df.copy()
    tmp["scenario_id"] = str(row["scenario_id"])
    tmp["scenario_label"] = str(row["scenario_label"])
    tmp["scenario_status"] = str(row.get("scenario_status", "hypothetical"))
    tmp["scenario_source"] = str(row.get("scenario_source", "unspecified"))
    tmp["delta_shock_ablation"] = delta
    tmp["hazard_1_shock"] = (tmp["hazard_0"] * (1.0 - delta)).astype(float).clip(EPS, 1 - EPS)
    tmp["S_shock_hat"] = (1.0 - tmp["hazard_1_shock"]).groupby(gkey).cumprod().astype(float).clip(EPS, 1.0)
    traj_parts.append(tmp)

pp_traj_multi = pd.concat(traj_parts, axis=0, ignore_index=True)

weekly_multi = (
    pp_traj_multi.groupby([
        "scenario_id",
        "scenario_label",
        "scenario_status",
        "scenario_source",
        "delta_shock_ablation",
        "week",
    ], as_index=False)[["S0_hat", "S_shock_hat", "S_mech_hat"]]
    .mean()
    .rename(columns={
        "S0_hat": "S0_bar",
        "S_shock_hat": "S_shock_bar",
        "S_mech_hat": "S_mech_bar",
    })
)
weekly_multi["deltaS_shock"] = weekly_multi["S_shock_bar"] - weekly_multi["S0_bar"]
weekly_multi["deltaS_mech"] = weekly_multi["S_mech_bar"] - weekly_multi["S0_bar"]
weekly_multi["deltaDeltaS_mech_minus_shock"] = weekly_multi["deltaS_mech"] - weekly_multi["deltaS_shock"]

if int(weekly_multi["week"].max()) < T_eval:
    raise ValueError(
        f"P14-v4.0: weekly_multi week_max={int(weekly_multi['week'].max())} < T_eval={T_eval}. "
        "This indicates pp_policy does not contain full TEST support weeks."
    )

horizon_parts = []
for row in scenario_catalog.to_dict(orient="records"):
    sid = str(row["scenario_id"])
    wk = weekly_multi.loc[weekly_multi["scenario_id"] == sid].copy()
    for horizon_label, t in [("T_policy", T_policy), ("T_eval", T_eval)]:
        r = wk.loc[wk["week"] == int(t)]
        if r.empty:
            raise ValueError(f"P14-v4.0: week={t} not present for scenario_id={sid}")
        rr = r.iloc[0].to_dict()
        rr["horizon_label"] = horizon_label
        rr["week"] = int(t)
        horizon_parts.append(rr)

horizons_multi = pd.DataFrame(horizon_parts)
horizons_multi = horizons_multi[[
    "scenario_id",
    "scenario_label",
    "scenario_status",
    "scenario_source",
    "delta_shock_ablation",
    "horizon_label",
    "week",
    "S0_bar",
    "S_shock_bar",
    "S_mech_bar",
    "deltaS_shock",
    "deltaS_mech",
    "deltaDeltaS_mech_minus_shock",
]].copy()

pp_traj = pp_traj_multi.loc[pp_traj_multi["scenario_id"] == reference_scenario_id].copy().reset_index(drop=True)
weekly = weekly_multi.loc[weekly_multi["scenario_id"] == reference_scenario_id].copy().reset_index(drop=True)
horizons = horizons_multi.loc[horizons_multi["scenario_id"] == reference_scenario_id].copy().reset_index(drop=True)

print("\n=== P14-v4.0 — Trajectory Objects READY ===")
print("[Horizons]")
print("T_policy:", T_policy)
print("T_eval  :", T_eval)
print("reference_scenario_id:", reference_scenario_id)

print("\n[Shapes]")
print("pp_traj_multi rows:", int(pp_traj_multi.shape[0]), "| cols:", int(pp_traj_multi.shape[1]))
print("weekly_multi rows :", int(weekly_multi.shape[0]), "| week min/max:", int(weekly_multi["week"].min()), int(weekly_multi["week"].max()))
print("horizons_multi rows:", int(horizons_multi.shape[0]))
print("reference weekly rows:", int(weekly.shape[0]))

print("\n[Scenario horizons]")
print(horizons_multi[["scenario_id", "horizon_label", "week", "deltaS_shock", "deltaS_mech"]].to_string(index=False))

print("\n=== P14-v4.0 DONE ===")

=== P14-v4.0 — START (build pp_traj/weekly/horizons) ===

=== P14-v4.0 — Trajectory Objects READY ===
[Horizons]
T_policy: 18
T_eval  : 38
reference_scenario_id: hypothetical_A

[Shapes]
pp_traj_multi rows: 697251 | cols: 16
weekly_multi rows : 117 | week min/max: 0 38
horizons_multi rows: 6
reference weekly rows: 39

[Scenario horizons]
          scenario_id horizon_label  week  deltaS_shock  deltaS_mech
anchored_conservative      T_policy    18      0.010236    -0.000604
anchored_conservative        T_eval    38      0.009589    -0.006237
       hypothetical_A      T_policy    18      0.025960    -0.000604
       hypothetical_A        T_eval    38      0.024273    -0.006237
       hypothetical_B      T_policy    18      0.081946    -0.000604
       hypothetical_B        T_eval    38      0.075976    -0.006237

=== P14-v4.0 DONE ===


In [949]:
# ==============================================================
# P14-v4.1-v2 — Contract Guard: Canonical pp_policy (NO pph dependency)
# --------------------------------------------------------------
# Purpose:
#   Enforce a stable, reportable contract for `pp_policy` BEFORE any exports.
#
#   In this notebook, counterfactual hazards are produced upstream (P13-v6),
#   so this block must NOT depend on intermediate frames such as `pph`.
#
# Contract enforced:
#   - `pp_policy` exists
#   - `pp_policy` contains:
#       KEYS + ['week', 'hazard_0', 'hazard_1_mech', 'hazard_1_shock']
#   - hazards are numeric floats clipped to [1e-12, 1-1e-12]
#   - week is int
#
# If the contract is not met:
#   - FAIL FAST with an explicit instruction to run the correct upstream P-step.
#
# Outputs (for feedback):
#   - shape
#   - column presence check
#   - min/max of each hazard
# ==============================================================

import pandas as pd
import numpy as np

print("=== P14-v4.1-v2 — START (pp_policy contract) ===")

# ------------------------------
# 0) Preconditions (assertive)
# ------------------------------
required_globals = ["pp_policy", "KEYS"]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise NameError(
        f"P14-v4.1-v2 missing required globals: {missing}. "
        "This notebook expects `pp_policy` to be created upstream (P13-v6)."
    )

required_cols = list(KEYS) + ["week", "hazard_0", "hazard_1_mech", "hazard_1_shock"]
missing_cols = [c for c in required_cols if c not in pp_policy.columns]
if missing_cols:
    raise KeyError(
        f"P14-v4.1-v2 requires columns missing from pp_policy: {missing_cols}. "
        "Run P13-v6 (policy impact; it materializes hazard_0/hazard_1_*)."
    )

# ------------------------------
# 1) Canonical typing + clipping (NO recomputation)
# ------------------------------
EPS = 1e-12

pp_policy = pp_policy.copy()
pp_policy["week"] = pd.to_numeric(pp_policy["week"], errors="raise").astype(int)

for c in ["hazard_0", "hazard_1_mech", "hazard_1_shock"]:
    pp_policy[c] = (
        pd.to_numeric(pp_policy[c], errors="raise")
        .astype(float)
        .clip(EPS, 1.0 - EPS)
    )

# Optional: enforce deterministic sort (helps later merges/exports)
pp_policy = pp_policy.sort_values(list(KEYS) + ["week"]).reset_index(drop=True)

# ------------------------------
# 2) REQUIRED TEXT OUTPUTS
# ------------------------------
print("[P14-v4.1-v2][OK] pp_policy contract satisfied.")
print("[P14-v4.1-v2][SHAPE]", pp_policy.shape)
print("[P14-v4.1-v2][COL CHECK] required cols present:",
      all(col in pp_policy.columns for col in required_cols))

print("[P14-v4.1-v2][RANGE]")
print("hazard_0       min/max:", float(pp_policy["hazard_0"].min()), float(pp_policy["hazard_0"].max()))
print("hazard_1_mech  min/max:", float(pp_policy["hazard_1_mech"].min()), float(pp_policy["hazard_1_mech"].max()))
print("hazard_1_shock min/max:", float(pp_policy["hazard_1_shock"].min()), float(pp_policy["hazard_1_shock"].max()))

print("=== P14-v4.1-v2 DONE ===")

=== P14-v4.1-v2 — START (pp_policy contract) ===
[P14-v4.1-v2][OK] pp_policy contract satisfied.
[P14-v4.1-v2][SHAPE] (232417, 49)
[P14-v4.1-v2][COL CHECK] required cols present: True
[P14-v4.1-v2][RANGE]
hazard_0       min/max: 1e-12 0.7577341266149553
hazard_1_mech  min/max: 1e-12 0.6693421470112713
hazard_1_shock min/max: 1e-12 0.6061873012919643
=== P14-v4.1-v2 DONE ===


In [950]:
# ==============================================================
# P14-v4.2.5 — Contract Guard: P14 trajectory objects
# --------------------------------------------------------------
# Purpose:
#   Enforce that the P14 trajectory step has been executed in the CURRENT kernel.
#   This block does NOT recompute anything. It only validates required objects
#   for P14 exports (P14-v4.3).
#
# Required objects produced by P14-v4 trajectory step:
#   - weekly   : DataFrame
#   - horizons : DataFrame
#   - pp_traj  : DataFrame
#   - T_policy : int
#   - T_eval   : int
#
# If missing:
#   - FAIL FAST and point to the correct upstream step to run.
# ==============================================================

import pandas as pd

print("=== P14-v4.2.5 — START (trajectory object contract) ===")

required = ["weekly", "horizons", "pp_traj", "T_policy", "T_eval"]
missing = [k for k in required if k not in globals()]

if missing:
    raise NameError(
        "P14-v4.2.5: Missing objects required for P14 exports: "
        f"{missing}. "
        "You MUST run the P14 trajectory step (the cell that builds pp_traj + weekly + horizons "
        "and sets T_policy/T_eval) in this kernel BEFORE running P14-v4.3."
    )

# Type checks (strict)
if not isinstance(weekly, pd.DataFrame):
    raise TypeError("P14-v4.2.5: `weekly` must be a pandas DataFrame.")
if not isinstance(horizons, pd.DataFrame):
    raise TypeError("P14-v4.2.5: `horizons` must be a pandas DataFrame.")
if not isinstance(pp_traj, pd.DataFrame):
    raise TypeError("P14-v4.2.5: `pp_traj` must be a pandas DataFrame.")

# Scalar checks
T_policy = int(T_policy)
T_eval = int(T_eval)

print("[P14-v4.2.5][OK] weekly/horizons/pp_traj/T_policy/T_eval are present in-memory.")
print("[P14-v4.2.5][SHAPES] weekly:", weekly.shape, "| horizons:", horizons.shape, "| pp_traj:", pp_traj.shape)
print("[P14-v4.2.5][HORIZONS] T_policy:", T_policy, "| T_eval:", T_eval)
print("=== P14-v4.2.5 DONE ===")

=== P14-v4.2.5 — START (trajectory object contract) ===
[P14-v4.2.5][OK] weekly/horizons/pp_traj/T_policy/T_eval are present in-memory.
[P14-v4.2.5][SHAPES] weekly: (39, 12) | horizons: (2, 13) | pp_traj: (232417, 16)
[P14-v4.2.5][HORIZONS] T_policy: 18 | T_eval: 38
=== P14-v4.2.5 DONE ===


In [ ]:
# ==============================================================
# P14-v4.3 — Export policy deltaS artifacts (paper-only pack)
# --------------------------------------------------------------
# Purpose:
#   Export only the policy tables and figures cited in the paper.
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=== P14-v4.3 — START ===")

required_globals = ["weekly", "horizons", "pp_traj", "T_policy", "T_eval"]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise NameError(f"P14-v4.3 missing required objects: {missing}")
if not isinstance(weekly, pd.DataFrame) or not isinstance(horizons, pd.DataFrame) or not isinstance(pp_traj, pd.DataFrame):
    raise TypeError("P14-v4.3 expects weekly/horizons/pp_traj to be pandas DataFrames.")

if "weekly_multi" in globals() and isinstance(globals().get("weekly_multi"), pd.DataFrame):
    weekly_multi_local = globals()["weekly_multi"].copy()
else:
    weekly_multi_local = weekly.copy()
    if "scenario_id" not in weekly_multi_local.columns:
        weekly_multi_local["scenario_id"] = str(globals().get("reference_scenario_id", policy_params.get("reference_scenario_id", "reference")))
        weekly_multi_local["scenario_label"] = weekly_multi_local["scenario_id"]
        weekly_multi_local["scenario_status"] = "reference"
        weekly_multi_local["scenario_source"] = "legacy_single_scenario"
        weekly_multi_local["delta_shock_ablation"] = float(globals().get("reference_delta_shock_ablation", policy_params.get("reference_delta_shock_ablation", np.nan)))

if "horizons_multi" in globals() and isinstance(globals().get("horizons_multi"), pd.DataFrame):
    horizons_multi_local = globals()["horizons_multi"].copy()
else:
    horizons_multi_local = horizons.copy()
    if "scenario_id" not in horizons_multi_local.columns:
        horizons_multi_local["scenario_id"] = str(globals().get("reference_scenario_id", policy_params.get("reference_scenario_id", "reference")))
        horizons_multi_local["scenario_label"] = horizons_multi_local["scenario_id"]
        horizons_multi_local["scenario_status"] = "reference"
        horizons_multi_local["scenario_source"] = "legacy_single_scenario"
        horizons_multi_local["delta_shock_ablation"] = float(globals().get("reference_delta_shock_ablation", policy_params.get("reference_delta_shock_ablation", np.nan)))

T_policy_local = int(T_policy)
T_eval_local = int(T_eval)
reference_scenario_id = str(globals().get("reference_scenario_id", policy_params.get("reference_scenario_id", weekly_multi_local["scenario_id"].iloc[0])))

req_weekly_cols = ["week", "deltaS_shock", "deltaS_mech", "S0_bar", "S_shock_bar", "S_mech_bar"]
miss_weekly = [c for c in req_weekly_cols if c not in weekly.columns]
if miss_weekly:
    raise KeyError(f"P14-v4.3 requires columns missing from `weekly`: {miss_weekly}")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ------------------------------
# 1) Paper-used tables
# ------------------------------
p_week_scen = os.path.join(TABLE_DIR, "table_policy_deltaS_by_week_by_scenario.csv")
p_hz_scen = os.path.join(TABLE_DIR, "table_policy_deltaS_at_horizons_by_scenario.csv")

weekly_multi_local.to_csv(p_week_scen, index=False)
horizons_multi_local.to_csv(p_hz_scen, index=False)

# ------------------------------
# 2) Paper-used figures
# ------------------------------
p_fig_surv = os.path.join(FIG_DIR, "fig_policy_survival_by_week_baseline_shock_mech.png")
p_fig_delta_scen = os.path.join(FIG_DIR, "fig_policy_deltaS_by_week_shock_scenarios.png")

wk_ref = weekly.copy()
wk_ref["week"] = pd.to_numeric(wk_ref["week"], errors="raise").astype(int)
wk_ref = wk_ref.sort_values("week")

plt.figure()
plt.plot(wk_ref["week"], wk_ref["S0_bar"], marker="o", linestyle="-", label="Baseline")
plt.plot(wk_ref["week"], wk_ref["S_shock_bar"], marker="o", linestyle="-", label=f"Shock ({reference_scenario_id})")
plt.plot(wk_ref["week"], wk_ref["S_mech_bar"], marker="o", linestyle="-", label="Mechanism-aware")
plt.axvline(T_policy_local, linestyle=":", label="T_policy")
plt.axvline(T_eval_local, linestyle=":", label="T_eval")
plt.xlabel("Week")
plt.ylabel("Mean survival S(t)")
plt.title("Survival trajectories on TEST grid")
plt.legend()
plt.tight_layout()
plt.savefig(p_fig_surv, dpi=200)
plt.close()

wk_multi = weekly_multi_local.copy()
wk_multi["week"] = pd.to_numeric(wk_multi["week"], errors="raise").astype(int)
wk_multi = wk_multi.sort_values(["scenario_id", "week"]).reset_index(drop=True)

plt.figure()
mech_ref = wk_multi.drop_duplicates(subset=["week"])[["week", "deltaS_mech"]].sort_values("week")
plt.plot(mech_ref["week"], mech_ref["deltaS_mech"], color="black", linestyle="--", linewidth=2.0, label="Mechanism-aware")
for _, sub in wk_multi.groupby("scenario_id", sort=False):
    label = f"{sub['scenario_label'].iloc[0]} (δ={float(sub['delta_shock_ablation'].iloc[0]):.2f})"
    plt.plot(sub["week"], sub["deltaS_shock"], marker="o", linestyle="-", label=label)
plt.axvline(T_policy_local, linestyle=":", label="T_policy")
plt.axvline(T_eval_local, linestyle=":", label="T_eval")
plt.xlabel("Week")
plt.ylabel("ΔS(t)")
plt.title("Policy effect on survival across shock scenarios")
plt.legend()
plt.tight_layout()
plt.savefig(p_fig_delta_scen, dpi=200)
plt.close()

print("=== P14-v4.3 — Export policy deltaS artifacts ===")
print("[Saved paper pack]")
print("weekly by scenario   :", p_week_scen)
print("horizons by scenario :", p_hz_scen)
print("fig survival         :", p_fig_surv)
print("fig deltaS scenarios :", p_fig_delta_scen)
print("\n[Sanity]")
print("reference_scenario_id:", reference_scenario_id)
print("weekly rows:", int(len(weekly)), "| scenario rows:", int(len(weekly_multi_local)))
print("horizons rows:", int(len(horizons)), "| scenario rows:", int(len(horizons_multi_local)))
print("=== P14-v4.3 DONE ===")

=== P14-v4.3 — START ===
[TABLE] saved: outputs_v2/tables/table_policy_deltaS_by_week_all.csv | shape=(39, 12)
columns: ['scenario_id', 'scenario_label', 'scenario_status', 'scenario_source', 'delta_shock_ablation', 'week', 'S0_bar', 'S_shock_bar', 'S_mech_bar', 'deltaS_shock', 'deltaS_mech', 'deltaDeltaS_mech_minus_shock']
sample (n=5):
      scenario_id  scenario_label scenario_status  \
0  hypothetical_A  Hypothetical A    hypothetical   
1  hypothetical_A  Hypothetical A    hypothetical   
2  hypothetical_A  Hypothetical A    hypothetical   
3  hypothetical_A  Hypothetical A    hypothetical   
4  hypothetical_A  Hypothetical A    hypothetical   

                scenario_source  delta_shock_ablation  week    S0_bar  \
0  hypothetical_legacy_baseline                   0.2     0  0.980959   
1  hypothetical_legacy_baseline                   0.2     1  0.961694   
2  hypothetical_legacy_baseline                   0.2     2  0.955457   
3  hypothetical_legacy_baseline                  

# P14.1 — Temporal Support Diagnostics (Events ≤ t and At-Risk at t)

In [952]:
# ==============================================================
# P14.1-v3 — Temporal Support Diagnostics (In-memory first, CSV fallback)
# --------------------------------------------------------------
# Purpose:
#   Provide reportable, horizon-aligned support diagnostics for the policy
#   horizons used in P14 (T_policy, T_eval), WITHOUT breaking when exports
#   are not yet written to disk.
#
# Key change vs P14.1-v2:
#   - Prefer in-memory objects created by P14 (weekly, horizons) when available.
#   - Fallback to reading CSVs only if those objects do not exist.
#
# Computes for each week t:
#   - n_events_at_week(t)
#   - n_events_cum_leq_week(t)
#   - n_at_risk_week(t) where t_final_week >= t
#
# Requires:
#   - KEYS
#   - enrollments_eventtime with: event_withdrawn, t_event_week, t_final_week
#   - Either:
#       (A) in-memory: `weekly` and `horizons` DataFrames from P14, OR
#       (B) exported CSVs in outputs_v2/tables/
#
# Outputs:
#   - outputs_v2/tables/table_support_by_week.csv
#   - outputs_v2/tables/table_policy_support_at_horizons.csv
#   - outputs_v2/tables/table_policy_deltaS_at_horizons.csv (OVERWRITTEN/CREATED)
# ==============================================================

import os
import numpy as np
import pandas as pd

print("=== P14.1-v3 — START ===")

# ------------------------------
# 0) Preconditions (fail fast)
# ------------------------------
required_globals = ["KEYS", "enrollments_eventtime"]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise NameError(f"P14.1-v3 missing required globals: {missing}")

required_cols_enr = KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]
miss_cols = [c for c in required_cols_enr if c not in enrollments_eventtime.columns]
if miss_cols:
    raise KeyError(f"P14.1-v3 requires columns missing from enrollments_eventtime: {miss_cols}")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

p_weekly = os.path.join(TABLE_DIR, "table_policy_deltaS_by_week_all.csv")
p_hz = os.path.join(TABLE_DIR, "table_policy_deltaS_at_horizons.csv")

# ------------------------------
# 1) Load weekly/horizons (in-memory first)
# ------------------------------
weekly_df = None
horizons_df = None

if "weekly" in globals() and isinstance(globals()["weekly"], pd.DataFrame):
    weekly_df = globals()["weekly"].copy()
    print("[P14.1-v3][INFO] Using in-memory `weekly` from P14.")
elif os.path.exists(p_weekly):
    weekly_df = pd.read_csv(p_weekly)
    print("[P14.1-v3][INFO] Using CSV weekly table:", p_weekly)
else:
    raise FileNotFoundError(
        "P14.1-v3: weekly table not found in memory nor on disk.\n"
        "Run P14 first (it must create a DataFrame named `weekly` or export CSV)."
    )

if "horizons" in globals() and isinstance(globals()["horizons"], pd.DataFrame):
    horizons_df = globals()["horizons"].copy()
    print("[P14.1-v3][INFO] Using in-memory `horizons` from P14.")
elif os.path.exists(p_hz):
    horizons_df = pd.read_csv(p_hz)
    print("[P14.1-v3][INFO] Using CSV horizons table:", p_hz)
else:
    raise FileNotFoundError(
        "P14.1-v3: horizons table not found in memory nor on disk.\n"
        "Run P14 first (it must create a DataFrame named `horizons` or export CSV)."
    )

# Validate week columns
if "week" not in weekly_df.columns:
    raise KeyError("P14.1-v3: 'week' not found in weekly table.")
if "week" not in horizons_df.columns:
    raise KeyError("P14.1-v3: 'week' not found in horizons table.")

week_min = int(pd.to_numeric(weekly_df["week"], errors="raise").min())
week_max = int(pd.to_numeric(weekly_df["week"], errors="raise").max())

# ------------------------------
# 2) Enrollment backbone (one row per enrollment)
# ------------------------------
enr = (
    enrollments_eventtime[required_cols_enr]
    .drop_duplicates(subset=KEYS)
    .copy()
)

enr["event_withdrawn"] = pd.to_numeric(enr["event_withdrawn"], errors="raise").astype(int)
enr["t_final_week"] = pd.to_numeric(enr["t_final_week"], errors="raise").astype(int)
enr["t_event_week_num"] = pd.to_numeric(enr["t_event_week"], errors="coerce")

# Integrity checks (strict)
bad_event_missing = enr[(enr["event_withdrawn"] == 1) & (enr["t_event_week_num"].isna())]
if len(bad_event_missing) > 0:
    raise ValueError(f"P14.1-v3 integrity fail: event_withdrawn==1 but t_event_week is NA (n={len(bad_event_missing)})")

bad_none_has_tevent = enr[(enr["event_withdrawn"] == 0) & (enr["t_event_week_num"].notna())]
if len(bad_none_has_tevent) > 0:
    raise ValueError(f"P14.1-v3 integrity fail: t_event_week present but event_withdrawn==0 (n={len(bad_none_has_tevent)})")

# ------------------------------
# 3) Support-by-week table
# ------------------------------
weeks = np.arange(week_min, week_max + 1, dtype=int)
df = pd.DataFrame({"week": weeks})

events_only = enr.loc[enr["event_withdrawn"] == 1, ["t_event_week_num"]].copy()
events_only["t_event_week_int"] = events_only["t_event_week_num"].astype(int)

events_at = (
    events_only.groupby("t_event_week_int", as_index=False)
    .size()
    .rename(columns={"t_event_week_int": "week", "size": "n_events_at_week"})
)

df = df.merge(events_at, on="week", how="left")
df["n_events_at_week"] = df["n_events_at_week"].fillna(0).astype(int)

# At-risk via searchsorted
tfinal = np.sort(enr["t_final_week"].to_numpy(dtype=int))
idx = np.searchsorted(tfinal, weeks, side="left")
df["n_at_risk_week"] = (len(tfinal) - idx).astype(int)

df["n_events_cum_leq_week"] = df["n_events_at_week"].cumsum().astype(int)

# ------------------------------
# 4) Attach support to horizons (and write back)
# ------------------------------
support_at_horizons = horizons_df.merge(
    df[["week", "n_events_at_week", "n_events_cum_leq_week", "n_at_risk_week"]],
    on="week",
    how="left",
    validate="m:1",
)

if "horizon_label" in support_at_horizons.columns:
    support_at_horizons["horizon_label_explicit"] = support_at_horizons["horizon_label"].astype(str).replace({"T_eval": "T_eval_policy"})

miss_support = support_at_horizons[["n_events_at_week", "n_events_cum_leq_week", "n_at_risk_week"]].isna().any(axis=1)
if miss_support.any():
    bad = support_at_horizons.loc[miss_support, ["horizon_label", "week"]] if "horizon_label" in support_at_horizons.columns else support_at_horizons.loc[miss_support, ["week"]]
    raise ValueError(f"P14.1-v3: missing support rows for some horizons:\n{bad}")

# ------------------------------
# 5) Exports (always write, to enforce single source)
# ------------------------------
p_support_by_week = os.path.join(TABLE_DIR, "table_support_by_week.csv")
p_support_at_hz = os.path.join(TABLE_DIR, "table_policy_support_at_horizons.csv")

df.to_csv(p_support_by_week, index=False)
support_at_horizons.to_csv(p_support_at_hz, index=False)
support_at_horizons.to_csv(p_hz, index=False)  # overwrite/create canonical horizons file

# ------------------------------
# 6) REQUIRED TEXT OUTPUTS
# ------------------------------
print("\n=== P14.1-v3 — Temporal Support Diagnostics Summary ===")
print("[Week range]")
print("week_min:", week_min, "| week_max:", week_max)

print("\n[Global counts]")
print("N_enrollments:", int(enr.shape[0]))
print("N_events_total:", int(enr["event_withdrawn"].sum()))
print("Event week min (events only):", float(enr.loc[enr["event_withdrawn"] == 1, "t_event_week_num"].min()))
print("Event week max (events only):", float(enr.loc[enr["event_withdrawn"] == 1, "t_event_week_num"].max()))
print("Final week min:", int(enr["t_final_week"].min()))
print("Final week max:", int(enr["t_final_week"].max()))

print("\n[Support at horizons]")
cols_show = ["horizon_label","horizon_label_explicit","week","n_events_at_week","n_events_cum_leq_week","n_at_risk_week"]
cols_show = [c for c in cols_show if c in support_at_horizons.columns]
print(support_at_horizons[cols_show].to_string(index=False))

print("\n[Exports]")
print("support_by_week      :", p_support_by_week)
print("support_at_horizons  :", p_support_at_hz)
print("deltaS_at_horizons++ :", p_hz)
print("=== P14.1-v3 DONE ===")

=== P14.1-v3 — START ===
[P14.1-v3][INFO] Using in-memory `weekly` from P14.
[P14.1-v3][INFO] Using in-memory `horizons` from P14.
[TABLE] saved: outputs_v2/tables/table_support_by_week.csv | shape=(39, 4)
columns: ['week', 'n_events_at_week', 'n_at_risk_week', 'n_events_cum_leq_week']
sample (n=5):
   week  n_events_at_week  n_at_risk_week  n_events_cum_leq_week
0     0               713           32593                    713
1     1              1068           28633                   1781
2     2               215           27396                   1996
3     3               360           26956                   2356
4     4               303           26363                   2659
[TABLE] saved: outputs_v2/tables/table_policy_support_at_horizons.csv | shape=(2, 17)
columns: ['scenario_id', 'scenario_label', 'scenario_status', 'scenario_source', 'delta_shock_ablation', 'horizon_label', 'week', 'S0_bar', 'S_shock_bar', 'S_mech_bar', 'deltaS_shock', 'deltaS_mech', 'deltaDeltaS_mech_minus

# P14.2 — Horizon Alignment Table (Policy vs Metrics Horizons)


In [953]:
# ==============================================================
# P14.2-v1 — Dual Horizon Reconciliation (Policy vs Metrics) [NO P17 dependency]
# --------------------------------------------------------------
# Purpose:
#   The notebook runs sequentially, so P14.2 cannot depend on P17.
#   This block creates a "dual horizon" contract in a self-contained way:
#
#     - T_eval_policy : evaluation horizon used for policy trajectories (RQ2),
#                      aligned to TEST raw support (from P14, typically T_max_test)
#     - T_eval_metrics: a *stable* horizon for IPCW-style metrics,
#                      chosen such that censoring survival G(t) >= g_min
#
#   We compute censoring survival G(t) via a discrete-time Kaplan–Meier on TEST:
#     - Censoring "event" for G(t): enrollment ends without dropout
#       i.e., event_censor = 1 if event_withdrawn == 0
#     - Dropout (event_withdrawn==1) is treated as "censored" in the censoring KM
#
# Requires (produced earlier in the sequential notebook):
#   - KEYS
#   - enrollments_eventtime with: event_withdrawn, t_final_week
#   - enroll_test_keys
#   - (from P14) global T_eval (or you can read max week from policy weekly export)
#   - (from P14.1) support_by_week export (optional; we read if available)
#
# Outputs:
#   - globals:
#       * T_eval_policy
#       * T_eval_metrics
#       * G_test_by_week (np.ndarray)
#   - Exports:
#       * outputs_v2/tables/table_policy_horizons_dual.csv
#       * outputs_v2/tables/table_censoring_survival_G_test_by_week.csv
#       * outputs_v2/figures/fig_censoring_survival_G_test.png
#
# REQUIRED TEXT OUTPUTS:
#   Paste printed outputs back into chat.
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# 0) Preconditions (fail fast, no guessing)
# ------------------------------
required_objs = ["KEYS", "enrollments_eventtime", "enroll_test_keys"]
missing_objs = [k for k in required_objs if k not in globals()]
if missing_objs:
    raise NameError(f"P14.2-v1 missing required objects: {missing_objs}")

required_cols = KEYS + ["event_withdrawn", "t_final_week"]
miss_cols = [c for c in required_cols if c not in enrollments_eventtime.columns]
if miss_cols:
    raise KeyError(f"P14.2-v1 requires columns missing from enrollments_eventtime: {miss_cols}")

# ------------------------------
# 1) Determine T_eval_policy (raw TEST support)
# ------------------------------
# Prefer the T_eval from P14 if it exists; otherwise infer from TEST enrollments' t_final_week
if "T_eval" in globals():
    T_eval_policy = int(globals()["T_eval"])
else:
    test_enr_tmp = (
        enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner")
        .loc[:, KEYS + ["t_final_week"]]
        .drop_duplicates(subset=KEYS)
        .copy()
    )
    test_enr_tmp["t_final_week"] = pd.to_numeric(test_enr_tmp["t_final_week"], errors="raise").astype(int)
    T_eval_policy = int(test_enr_tmp["t_final_week"].max())

# ------------------------------
# 2) Build TEST enrollment backbone for censoring KM
# ------------------------------
test_enr = (
    enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner")
    .loc[:, KEYS + ["event_withdrawn", "t_final_week"]]
    .drop_duplicates(subset=KEYS)
    .copy()
)

test_enr["event_withdrawn"] = pd.to_numeric(test_enr["event_withdrawn"], errors="raise").astype(int)
test_enr["t_final_week"] = pd.to_numeric(test_enr["t_final_week"], errors="raise").astype(int)

if (test_enr["t_final_week"] < 0).any():
    raise ValueError("P14.2-v1: negative t_final_week found in TEST enrollments (should not happen).")

T_max_test = int(test_enr["t_final_week"].max())
if T_eval_policy > T_max_test:
    # Safety: keep policy horizon within actual TEST support
    T_eval_policy = T_max_test

# ------------------------------
# 3) Discrete Kaplan–Meier for censoring survival G(t)
# ------------------------------
# Define censoring event for G(t):
#   event_censor = 1 if enrollment is censored (no dropout) and ends at t_final_week
#   event_censor = 0 if dropout occurred (dropout acts as censoring for the censoring-KM)
times = test_enr["t_final_week"].to_numpy(dtype=int)
event_censor = (test_enr["event_withdrawn"].to_numpy(dtype=int) == 0).astype(int)

def km_survival_discrete(times_int: np.ndarray, event_int: np.ndarray, T_max: int) -> np.ndarray:
    """
    Discrete-time KM survival curve S(t) for 'event_int' over integer times in [0..T_max].
    Here, S(t) = P(T_event > t).
    """
    times_int = times_int.astype(int)
    event_int = event_int.astype(int)

    S = np.ones(T_max + 1, dtype=float)
    # Risk set at time t: those with time >= t
    for t in range(0, T_max + 1):
        at_risk = np.sum(times_int >= t)
        if at_risk <= 0:
            S[t] = S[t - 1] if t > 0 else 1.0
            continue
        d_t = np.sum((times_int == t) & (event_int == 1))
        # hazard of censoring at t
        h_t = d_t / at_risk
        if t == 0:
            S[t] = (1.0 - h_t)
        else:
            S[t] = S[t - 1] * (1.0 - h_t)
        # numerical floor (avoid exact zero)
        if S[t] < 1e-12:
            S[t] = 1e-12
    return S

G_test_by_week = km_survival_discrete(times, event_censor, T_max_test)

# ------------------------------
# 4) Choose a stable metrics horizon T_eval_metrics
# ------------------------------
g_min = 0.05  # reportable stability threshold
cands = np.where(G_test_by_week[:T_eval_policy + 1] >= g_min)[0]
if len(cands) == 0:
    # No stable horizon exists under g_min; fall back to T_policy if present, else 0
    T_eval_metrics = int(globals().get("T_policy", 0))
else:
    T_eval_metrics = int(cands.max())

# ------------------------------
# 5) Optional: read at-risk support (if P14.1 exported it)
# ------------------------------
OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

support_path = os.path.join(TABLE_DIR, "table_support_by_week.csv")
n_at_risk_Tpolicy = np.nan
n_at_risk_Teval_policy = np.nan
n_at_risk_Teval_metrics = np.nan

if os.path.exists(support_path):
    sup = pd.read_csv(support_path)
    if "week" in sup.columns and "n_at_risk_week" in sup.columns:
        def _get_at_risk(t):
            s = sup.loc[sup["week"] == t, "n_at_risk_week"]
            return float(s.iloc[0]) if len(s) > 0 else np.nan
        n_at_risk_Tpolicy = _get_at_risk(int(globals().get("T_policy", 18)))
        n_at_risk_Teval_policy = _get_at_risk(T_eval_policy)
        n_at_risk_Teval_metrics = _get_at_risk(T_eval_metrics)

# ------------------------------
# 6) Exports: table + G(t) plot
# ------------------------------
dfG = pd.DataFrame({
    "week": np.arange(0, T_max_test + 1, dtype=int),
    "G_t": G_test_by_week.astype(float),
})

pG = os.path.join(TABLE_DIR, "table_censoring_survival_G_test_by_week.csv")
dfG.to_csv(pG, index=False)

# Plot
pfig = os.path.join(FIG_DIR, "fig_censoring_survival_G_test.png")
plt.figure()
plt.plot(dfG["week"], dfG["G_t"], marker="o", linestyle="-")
plt.axhline(g_min, linestyle="--")
plt.axvline(int(globals().get("T_policy", 18)), linestyle=":")
plt.axvline(T_eval_policy, linestyle=":")
plt.axvline(T_eval_metrics, linestyle=":")
plt.xlabel("Week")
plt.ylabel("Censoring survival G(t) on TEST")
plt.title("TEST Censoring Survival G(t) with Dual Horizon Selection")
plt.tight_layout()
plt.savefig(pfig, dpi=200)
plt.close()

# Dual horizon table (paper-facing)
dual = pd.DataFrame([
    {
        "horizon_label": "T_policy",
        "week": int(globals().get("T_policy", 18)),
        "G_t": float(G_test_by_week[int(globals().get("T_policy", 18))]) if int(globals().get("T_policy", 18)) <= T_max_test else np.nan,
        "n_at_risk_week": n_at_risk_Tpolicy,
        "role": "policy_trigger_and_contrast",
    },
    {
        "horizon_label": "T_eval_policy",
        "week": int(T_eval_policy),
        "G_t": float(G_test_by_week[int(T_eval_policy)]) if int(T_eval_policy) <= T_max_test else np.nan,
        "n_at_risk_week": n_at_risk_Teval_policy,
        "role": "trajectory_reporting_and_deltaS_tables",
    },
    {
        "horizon_label": "T_eval_metrics",
        "week": int(T_eval_metrics),
        "G_t": float(G_test_by_week[int(T_eval_metrics)]) if int(T_eval_metrics) <= T_max_test else np.nan,
        "n_at_risk_week": n_at_risk_Teval_metrics,
        "role": "stable_IPCW_metrics_only",
    },
])

pdual = os.path.join(TABLE_DIR, "table_policy_horizons_dual.csv")
dual.to_csv(pdual, index=False)

# ------------------------------
# 7) REQUIRED TEXT OUTPUTS
# ------------------------------
print("=== P14.2-v1 — Dual Horizons (NO P17 dependency) ===")
print("[TEST support]")
print("T_max_test:", T_max_test)
print("T_eval_policy:", int(T_eval_policy))
print("T_eval_metrics:", int(T_eval_metrics))
print("g_min:", g_min)
print("G(T_eval_policy):", float(G_test_by_week[int(T_eval_policy)]))
print("G(T_eval_metrics):", float(G_test_by_week[int(T_eval_metrics)]))

print("\n[Dual horizon table]")
print(dual.to_string(index=False))

print("\n[Exports]")
print("G_table_csv :", pG)
print("G_fig_png   :", pfig)
print("dual_csv    :", pdual)
print("=== P14.2-v1 DONE ===")

[TABLE] saved: outputs_v2/tables/table_censoring_survival_G_test_by_week.csv | shape=(39, 2)
columns: ['week', 'G_t']
sample (n=5):
   week       G_t
0     0  0.898957
1     1  0.894034
2     2  0.886963
3     3  0.879281
4     4  0.871820
[FIGURE] saved: outputs_v2/figures/fig_censoring_survival_G_test.png | size_in=(6.40, 4.80) | dpi=200 | approx_px=(1280x960)
[TABLE] saved: outputs_v2/tables/table_policy_horizons_dual.csv | shape=(3, 5)
columns: ['horizon_label', 'week', 'G_t', 'n_at_risk_week', 'role']
sample (n=3):
    horizon_label  week           G_t  n_at_risk_week  \
0        T_policy    18  7.957980e-01         21255.0   
1   T_eval_policy    38  1.000000e-12          1575.0   
2  T_eval_metrics    37  6.007189e-02          3782.0   

                                     role  
0             policy_trigger_and_contrast  
1  trajectory_reporting_and_deltaS_tables  
2                stable_IPCW_metrics_only  
=== P14.2-v1 — Dual Horizons (NO P17 dependency) ===
[TEST support]
T

In [ ]:
# ==============================================================
# P15 — Export Parameters Table + Update Metadata (paper-only)
# --------------------------------------------------------------
# Purpose:
#   Export the paper-used scenario parameter table and keep metadata aligned
#   with the reduced policy artifact pack.
# ==============================================================

import json
import os
import pandas as pd

out_root = "outputs_v2"
tables_dir = os.path.join(out_root, "tables")
figs_dir = os.path.join(out_root, "figures")
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)

p_week_scen = os.path.join(tables_dir, "table_policy_deltaS_by_week_by_scenario.csv")
p_hz_scen = os.path.join(tables_dir, "table_policy_deltaS_at_horizons_by_scenario.csv")
p_fig_surv = os.path.join(figs_dir, "fig_policy_survival_by_week_baseline_shock_mech.png")
p_fig_scen = os.path.join(figs_dir, "fig_policy_deltaS_by_week_shock_scenarios.png")
p_catalog_main = os.path.join(tables_dir, "table_policy_scenarios_main.csv")

required_artifacts = [
    p_week_scen,
    p_hz_scen,
    p_fig_surv,
    p_fig_scen,
    p_catalog_main,
]
missing = [p for p in required_artifacts if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"P15 expected paper artifacts missing: {missing}")

if "policy_params" not in globals() or not isinstance(globals().get("policy_params"), dict):
    raise NameError("P15 expects `policy_params` dict to exist from earlier steps.")

if "policy_effect_shared" in globals() and isinstance(globals().get("policy_effect_shared"), dict):
    effect_shared = dict(globals()["policy_effect_shared"])
elif "policy_spec" in globals() and isinstance(globals().get("policy_spec"), dict):
    effect_shared = dict(globals()["policy_spec"].get("effect_shared", globals()["policy_spec"].get("effect", {})))
elif "policy_effect_params" in globals() and isinstance(globals().get("policy_effect_params"), dict):
    effect_shared = dict(globals()["policy_effect_params"])
else:
    effect_shared = {
        "W": 2,
        "alpha_week0": 0.35,
        "alpha_week1": 0.10,
        "decay_type": "kb2023_step_2w",
        "window_exclusive_upper": True,
    }

scenario_rows = list(policy_params.get("shock_scenarios", globals().get("shock_scenarios", [])))
if len(scenario_rows) == 0:
    raise ValueError("P15 requires a non-empty shock scenario catalog in policy_params or globals().")
reference_scenario_id = str(policy_params.get("reference_scenario_id", globals().get("reference_scenario_id", scenario_rows[0]["scenario_id"])))

rows = []
for scen in scenario_rows:
    rows.append({
        "scenario_id": str(scen["scenario_id"]),
        "scenario_label": str(scen.get("scenario_label", scen["scenario_id"])),
        "scenario_status": str(scen.get("scenario_status", "hypothetical")),
        "scenario_source": str(scen.get("scenario_source", "unspecified")),
        "scenario_note": str(scen.get("scenario_note", "")),
        "is_reference_scenario": int(str(scen["scenario_id"]) == reference_scenario_id),
        "anchor_study": str(policy_params.get("anchor_study", policy_params.get("trigger_anchor_study", ""))),
        "anchor_window_days": int(policy_params.get("anchor_window_days", 14)),
        "anchor_metric_week0": str(policy_params.get("anchor_metric_week0", "")),
        "anchor_metric_week1": str(policy_params.get("anchor_metric_week1", "")),
        "anchor_trigger_note": str(policy_params.get("anchor_trigger_note", policy_params.get("trigger_anchor_definition", ""))),
        "W_weeks": int(effect_shared.get("W", policy_params.get("W", 2))),
        "alpha_week0": float(effect_shared.get("alpha_week0", policy_params.get("alpha_week0", 0.35))),
        "alpha_week1": float(effect_shared.get("alpha_week1", policy_params.get("alpha_week1", 0.10))),
        "decay_type": str(effect_shared.get("decay_type", policy_params.get("decay_type", "kb2023_step_2w"))),
        "window_exclusive_upper": bool(effect_shared.get("window_exclusive_upper", policy_params.get("window_exclusive_upper", True))),
        "delta_shock_ablation": float(scen["delta_shock_ablation"]),
    })

df_params = pd.DataFrame(rows)
p_params = os.path.join(tables_dir, "table_policy_scenario_params.csv")
df_params.to_csv(p_params, index=False)

meta_path = os.path.join(out_root, "metadata.json")
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

metadata.setdefault("tables", [])
metadata.setdefault("figures", [])
metadata.setdefault("artifacts", {})
metadata.setdefault("policy", {})

def _add_unique(lst, rel_path):
    if rel_path not in lst:
        lst.append(rel_path)

paper_tables = [
    os.path.relpath(p_week_scen, out_root),
    os.path.relpath(p_hz_scen, out_root),
    os.path.relpath(p_catalog_main, out_root),
    os.path.relpath(p_params, out_root),
]
paper_figures = [
    os.path.relpath(p_fig_surv, out_root),
    os.path.relpath(p_fig_scen, out_root),
]
for rel_path in paper_tables:
    _add_unique(metadata["tables"], rel_path)
for rel_path in paper_figures:
    _add_unique(metadata["figures"], rel_path)

anchored_ids = [str(r["scenario_id"]) for r in rows if str(r.get("scenario_status", "")).lower() == "anchored"]
hypothetical_ids = [str(r["scenario_id"]) for r in rows if str(r.get("scenario_status", "")).lower() == "hypothetical"]

metadata["policy"].update({
    "reference_scenario_id": reference_scenario_id,
    "scenario_catalog_table": os.path.relpath(p_params, out_root),
    "scenario_catalog_main_table": os.path.relpath(p_catalog_main, out_root),
    "scenario_count": int(df_params.shape[0]),
    "scenario_ids": [str(r["scenario_id"]) for r in rows],
    "horizons": {
        "T_policy": int(T_policy),
        "T_eval_policy": int(T_eval),
        "T_eval_metrics": int(globals().get("T_eval_metrics", T_eval)),
    },
    "operational_anchor": {
        "study": str(policy_params.get("trigger_anchor_study", policy_params.get("anchor_study", ""))),
        "trigger_rule": str(policy_params.get("trigger_anchor_definition", policy_params.get("anchor_trigger_note", ""))),
        "trigger_frequency": str(policy_params.get("trigger_frequency", "weekly")),
        "active_window_weeks": int(effect_shared.get("W", policy_params.get("W", 2))),
    },
    "shock_intensity_framing": {
        "anchored_scenarios": anchored_ids,
        "hypothetical_scenarios": hypothetical_ids,
        "anchored_note": "delta=0.08 is the anchored conservative scenario; higher deltas are hypothetical stress tests",
    },
    "deltaS_tables": {
        "weekly_by_scenario": os.path.relpath(p_week_scen, out_root),
        "horizons_by_scenario": os.path.relpath(p_hz_scen, out_root),
    },
    "figures": {
        "survival": os.path.relpath(p_fig_surv, out_root),
        "scenario_deltaS": os.path.relpath(p_fig_scen, out_root),
    },
    "scenario_params": rows,
})

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, sort_keys=True)

print("=== P15 — Export Params + Metadata Update Summary (Paper-only) ===")
print("\n[Created/Updated]")
print("params table:", p_params)
print("scenario main:", p_catalog_main)
print("metadata    :", meta_path)

print("\n[Scenario params]")
print(df_params.to_string(index=False))

print("\n[Registered paper policy artifacts]")
print("weekly_by_scenario :", metadata["policy"]["deltaS_tables"].get("weekly_by_scenario"))
print("horizons_by_scenario:", metadata["policy"]["deltaS_tables"].get("horizons_by_scenario"))
print("survival_figure    :", metadata["policy"]["figures"].get("survival"))
print("scenario_deltaS_fig:", metadata["policy"]["figures"].get("scenario_deltaS"))
print("=== P15 DONE ===")

[TABLE] saved: outputs_v2/tables/table_policy_scenario_params.csv | shape=(3, 17)
columns: ['scenario_id', 'scenario_label', 'scenario_status', 'scenario_source', 'scenario_note', 'is_reference_scenario', 'anchor_study', 'anchor_window_days', 'anchor_metric_week0', 'anchor_metric_week1', 'anchor_trigger_note', 'W_weeks', 'alpha_week0', 'alpha_week1', 'decay_type', 'window_exclusive_upper', 'delta_shock_ablation']
sample (n=3):
             scenario_id         scenario_label scenario_status  \
0  anchored_conservative  Conservative anchored        anchored   
1         hypothetical_A         Hypothetical A    hypothetical   
2         hypothetical_B         Hypothetical B    hypothetical   

                    scenario_source  \
0       SneyersDeWitte2018_anchored   
1      hypothetical_legacy_baseline   
2  hypothetical_extreme_stress_test   

                                       scenario_note  is_reference_scenario  \
0                    Conservative anchored intensity            

# P15.2 — Unified Policy Spec Contract (Trigger + Effect) [REPORTABLE]

In [957]:
# P15.1.9 — Ensure policy_effect_params in memory
# Purpose: guarantee policy_effect_params and policy_effect_shared exist before P15.2
import numpy as np
import pandas as pd

DEFAULT_EFFECT_SHARED = {
    "W": 2,
    "alpha_week0": 0.35,
    "alpha_week1": 0.10,
    "decay_type": "kb2023_step_2w",
    "window_exclusive_upper": True,
}
DEFAULT_SHOCK_SCENARIOS = [
    {
        "scenario_id": "anchored_conservative",
        "scenario_label": "Conservative anchored",
        "delta_shock_ablation": 0.08,
        "scenario_status": "anchored",
        "scenario_source": "SneyersDeWitte2018_anchored",
        "scenario_note": "Conservative anchored intensity",
    },
    {
        "scenario_id": "hypothetical_A",
        "scenario_label": "Hypothetical A",
        "delta_shock_ablation": 0.20,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_legacy_baseline",
        "scenario_note": "Legacy notebook baseline, now treated as hypothetical",
    },
    {
        "scenario_id": "hypothetical_B",
        "scenario_label": "Hypothetical B",
        "delta_shock_ablation": 0.60,
        "scenario_status": "hypothetical",
        "scenario_source": "hypothetical_extreme_stress_test",
        "scenario_note": "Extreme stress-test intensity",
    },
]

def _as_float(x, fallback):
    try:
        v = float(x)
        if np.isnan(v):
            return fallback
        return v
    except Exception:
        return fallback

def _as_bool(x, fallback):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)):
        return bool(int(x))
    if isinstance(x, str) and x.strip().lower() in ("true", "false"):
        return x.strip().lower() == "true"
    return fallback

if "policy_spec" in globals() and isinstance(globals().get("policy_spec"), dict):
    shared_src = dict(globals()["policy_spec"].get("effect_shared", globals()["policy_spec"].get("effect", {})))
    scen_src = list(globals()["policy_spec"].get("shock_scenarios", []))
    ref_src = str(globals()["policy_spec"].get("reference_scenario_id", policy_params.get("reference_scenario_id", "hypothetical_A")))
    src = "policy_spec"
elif "policy_effect_params" in globals() and isinstance(globals().get("policy_effect_params"), dict):
    shared_src = dict(globals()["policy_effect_params"])
    scen_src = list(policy_params.get("shock_scenarios", globals().get("shock_scenarios", [])))
    ref_src = str(policy_params.get("reference_scenario_id", globals().get("reference_scenario_id", "hypothetical_A")))
    src = "existing policy_effect_params + policy_params.shock_scenarios"
else:
    shared_src = DEFAULT_EFFECT_SHARED.copy()
    scen_src = list(policy_params.get("shock_scenarios", DEFAULT_SHOCK_SCENARIOS)) if "policy_params" in globals() else list(DEFAULT_SHOCK_SCENARIOS)
    ref_src = str(policy_params.get("reference_scenario_id", "hypothetical_A")) if "policy_params" in globals() else "hypothetical_A"
    src = "anchored defaults"

for key, value in DEFAULT_EFFECT_SHARED.items():
    shared_src.setdefault(key, value)

shock_scenarios = [dict(row) for row in (scen_src if len(scen_src) > 0 else DEFAULT_SHOCK_SCENARIOS)]
scenario_ids = [str(row["scenario_id"]) for row in shock_scenarios]
if ref_src not in set(scenario_ids):
    ref_src = "hypothetical_A" if "hypothetical_A" in set(scenario_ids) else scenario_ids[0]
reference_scenario_id = ref_src
reference_scenario = next(row for row in shock_scenarios if str(row["scenario_id"]) == reference_scenario_id)
reference_delta = float(reference_scenario["delta_shock_ablation"])

policy_effect_shared = {
    "W": int(shared_src.get("W", DEFAULT_EFFECT_SHARED["W"])),
    "alpha_week0": _as_float(shared_src.get("alpha_week0", DEFAULT_EFFECT_SHARED["alpha_week0"]), DEFAULT_EFFECT_SHARED["alpha_week0"]),
    "alpha_week1": _as_float(shared_src.get("alpha_week1", DEFAULT_EFFECT_SHARED["alpha_week1"]), DEFAULT_EFFECT_SHARED["alpha_week1"]),
    "decay_type": str(shared_src.get("decay_type", DEFAULT_EFFECT_SHARED["decay_type"])),
    "window_exclusive_upper": _as_bool(shared_src.get("window_exclusive_upper", DEFAULT_EFFECT_SHARED["window_exclusive_upper"]), True),
}

policy_effect_params = {
    **policy_effect_shared,
    "delta_shock_ablation": reference_delta,
    "reference_scenario_id": reference_scenario_id,
}
policy_scenario_catalog = pd.DataFrame(shock_scenarios)

print("=== P15.1.9 — policy_effect_params ready ===")
print("source:", src)
print("reference_scenario_id:", reference_scenario_id)
print("reference_delta_shock_ablation:", reference_delta)
print("policy_effect_shared:", policy_effect_shared)
print(policy_scenario_catalog[["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status"]].to_string(index=False))

=== P15.1.9 — policy_effect_params ready ===
source: policy_spec
reference_scenario_id: hypothetical_A
reference_delta_shock_ablation: 0.2
policy_effect_shared: {'W': 2, 'alpha_week0': 0.35, 'alpha_week1': 0.1, 'decay_type': 'kb2023_step_2w', 'window_exclusive_upper': True}
          scenario_id        scenario_label  delta_shock_ablation scenario_status
anchored_conservative Conservative anchored                  0.08        anchored
       hypothetical_A        Hypothetical A                  0.20    hypothetical
       hypothetical_B        Hypothetical B                  0.60    hypothetical


In [958]:
# ==============================================================
# P15.2 — Unified Policy Spec Contract (Trigger + Shared Effect + Scenarios) [REPORTABLE]
# --------------------------------------------------------------
# Purpose:
#   Make the policy definition single-source-of-truth while preserving a
#   compatibility bridge for cells that still expect policy_spec['effect'].
# ==============================================================

import os
import json
import numpy as np
import pandas as pd
from hashlib import sha256

if "policy_params" not in globals() or not isinstance(globals().get("policy_params"), dict):
    raise NameError("P15.2 requires `policy_params` as a dict (trigger params).")
if "policy_effect_shared" not in globals() or not isinstance(globals().get("policy_effect_shared"), dict):
    raise NameError("P15.2 requires `policy_effect_shared` as a dict.")
if "policy_effect_params" not in globals() or not isinstance(globals().get("policy_effect_params"), dict):
    raise NameError("P15.2 requires `policy_effect_params` as a dict (reference effect params).")

policy_params = globals()["policy_params"]
policy_effect_shared = globals()["policy_effect_shared"]
policy_effect_params = globals()["policy_effect_params"]

shock_scenarios = list(policy_params.get("shock_scenarios", globals().get("shock_scenarios", [])))
if len(shock_scenarios) == 0:
    raise ValueError("P15.2 requires a non-empty shock scenario catalog.")
reference_scenario_id = str(policy_params.get("reference_scenario_id", policy_effect_params.get("reference_scenario_id", shock_scenarios[0]["scenario_id"])))

REQ_TRIGGER = ["r_star", "trigger_anchor_definition", "trigger_anchor_study"]
REQ_EFFECT_SHARED = ["W", "alpha_week0", "alpha_week1", "decay_type", "window_exclusive_upper"]
miss_tr = [k for k in REQ_TRIGGER if k not in policy_params]
miss_eff = [k for k in REQ_EFFECT_SHARED if k not in policy_effect_shared]
if miss_tr or miss_eff:
    raise KeyError(
        "P15.2: Missing required keys.\n"
        f"  policy_params missing: {miss_tr}\n"
        f"  policy_effect_shared missing: {miss_eff}"
    )

if reference_scenario_id not in {str(row["scenario_id"]) for row in shock_scenarios}:
    raise ValueError(f"P15.2: reference_scenario_id={reference_scenario_id} not found in shock_scenarios")
reference_scenario = next(row for row in shock_scenarios if str(row["scenario_id"]) == reference_scenario_id)
reference_delta = float(reference_scenario["delta_shock_ablation"])

policy_spec = {
    "trigger": {
        "r_star": int(policy_params["r_star"]),
        "trigger_anchor_definition": str(policy_params["trigger_anchor_definition"]),
        "trigger_anchor_study": str(policy_params["trigger_anchor_study"]),
        "recency_definition": str(policy_params.get("recency_definition", "")),
        "recency_proxy_col": str(policy_params.get("recency_proxy_col", "")),
        "trigger_frequency": str(policy_params.get("trigger_frequency", "")),
    },
    "effect_shared": {
        "W": int(policy_effect_shared["W"]),
        "alpha_week0": float(policy_effect_shared["alpha_week0"]),
        "alpha_week1": float(policy_effect_shared["alpha_week1"]),
        "decay_type": str(policy_effect_shared["decay_type"]),
        "window_exclusive_upper": bool(policy_effect_shared["window_exclusive_upper"]),
    },
    "shock_scenarios": [
        {
            "scenario_id": str(row["scenario_id"]),
            "scenario_label": str(row.get("scenario_label", row["scenario_id"])),
            "delta_shock_ablation": float(row["delta_shock_ablation"]),
            "scenario_status": str(row.get("scenario_status", "hypothetical")),
            "scenario_source": str(row.get("scenario_source", "unspecified")),
            "scenario_note": str(row.get("scenario_note", "")),
        }
        for row in shock_scenarios
    ],
    "reference_scenario_id": reference_scenario_id,
    "effect": {
        "W": int(policy_effect_shared["W"]),
        "alpha_week0": float(policy_effect_shared["alpha_week0"]),
        "alpha_week1": float(policy_effect_shared["alpha_week1"]),
        "decay_type": str(policy_effect_shared["decay_type"]),
        "window_exclusive_upper": bool(policy_effect_shared["window_exclusive_upper"]),
        "delta_shock_ablation": reference_delta,
    },
}

policy_spec_json = json.dumps(policy_spec, sort_keys=True, ensure_ascii=False)
policy_spec_hash = sha256(policy_spec_json.encode("utf-8")).hexdigest()[:12]
policy_spec["spec_hash_short_sha256"] = policy_spec_hash

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

row = {
    "trigger_anchor_study": str(policy_spec["trigger"]["trigger_anchor_study"]),
    "trigger_anchor_definition": str(policy_spec["trigger"]["trigger_anchor_definition"]),
    "trigger_frequency": str(policy_spec["trigger"].get("trigger_frequency", "")),
    "r_star": int(policy_spec["trigger"]["r_star"]),
    "recency_definition": str(policy_spec["trigger"].get("recency_definition", "")),
    "recency_proxy_col": str(policy_spec["trigger"].get("recency_proxy_col", "")),
    "W_weeks": int(policy_spec["effect_shared"]["W"]),
    "alpha_week0": float(policy_spec["effect_shared"]["alpha_week0"]),
    "alpha_week1": float(policy_spec["effect_shared"]["alpha_week1"]),
    "decay_type": str(policy_spec["effect_shared"]["decay_type"]),
    "window_exclusive_upper": int(bool(policy_spec["effect_shared"]["window_exclusive_upper"])),
    "reference_scenario_id": reference_scenario_id,
    "reference_delta_shock_ablation": reference_delta,
    "scenario_count": int(len(policy_spec["shock_scenarios"])),
    "shock_scenario_ids": json.dumps([row["scenario_id"] for row in policy_spec["shock_scenarios"]], ensure_ascii=False),
    "policy_spec_hash_short_sha256": policy_spec_hash,
}

df = pd.DataFrame([row])
p_csv = os.path.join(TABLE_DIR, "table_policy_spec.csv")
df.to_csv(p_csv, index=False)

META_PATH = os.path.join(OUT_DIR, "metadata.json")
meta = {}
if os.path.exists(META_PATH):
    try:
        with open(META_PATH, "r", encoding="utf-8") as f:
            meta = json.load(f)
    except Exception:
        raise ValueError("P15.2: metadata.json exists but could not be parsed. Fix/delete it and re-run.")

meta.setdefault("policy", {})
meta["policy"]["policy_spec"] = policy_spec
meta["policy"]["policy_spec_table"] = "tables/table_policy_spec.csv"

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("=== P15.2 — Unified Policy Spec Contract (REPORTABLE) ===")
print("\n[Trigger]")
print("anchor_study     :", policy_spec["trigger"]["trigger_anchor_study"])
print("anchor_definition:", policy_spec["trigger"]["trigger_anchor_definition"])
print("r_star           :", policy_spec["trigger"]["r_star"])
print("frequency        :", policy_spec["trigger"].get("trigger_frequency", ""))
print("\n[Effect shared]")
print("W_weeks              :", policy_spec["effect_shared"]["W"])
print("alpha_week0          :", policy_spec["effect_shared"]["alpha_week0"])
print("alpha_week1          :", policy_spec["effect_shared"]["alpha_week1"])
print("decay_type           :", policy_spec["effect_shared"]["decay_type"])
print("window_exclusive_upper:", policy_spec["effect_shared"]["window_exclusive_upper"])
print("\n[Reference scenario bridge]")
print("reference_scenario_id      :", reference_scenario_id)
print("reference_delta_shock_ablation:", reference_delta)
print("\n[Shock scenarios]")
print(pd.DataFrame(policy_spec["shock_scenarios"])[["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status", "scenario_source"]].to_string(index=False))
print("\n[Provenance]")
print("policy_spec_hash_short_sha256:", policy_spec_hash)
print("\n[Exports]")
print("policy_spec_csv :", p_csv)
print("metadata_json   :", META_PATH)
print("=== P15.2 DONE ===")

[TABLE] saved: outputs_v2/tables/table_policy_spec.csv | shape=(1, 16)
columns: ['trigger_anchor_study', 'trigger_anchor_definition', 'trigger_frequency', 'r_star', 'recency_definition', 'recency_proxy_col', 'W_weeks', 'alpha_week0', 'alpha_week1', 'decay_type', 'window_exclusive_upper', 'reference_scenario_id', 'reference_delta_shock_ablation', 'scenario_count', 'shock_scenario_ids', 'policy_spec_hash_short_sha256']
sample (n=1):
                                trigger_anchor_study  \
0  Kay & Bostock (2023) — The Power of the Nudge:...   

                           trigger_anchor_definition trigger_frequency  \
0  flag if no LMS engagement in last 7 days (≈ re...            weekly   

   r_star                                 recency_definition  \
0       1  weeks since last active week (activity_flag = ...   

  recency_proxy_col  W_weeks  alpha_week0  alpha_week1      decay_type  \
0      total_clicks        2         0.35          0.1  kb2023_step_2w   

   window_exclusive_upper

In [959]:
# ==============================================================
# P15.0 — Sync policy_params from policy_spec (NO GUESSING; scenario-aware)
# --------------------------------------------------------------
# Purpose:
#   Keep backward compatibility with cells that still read `policy_params`
#   while making `policy_spec` the source of truth.
# ==============================================================

print("=== P15.0 — START ===")

if "policy_spec" not in globals():
    raise NameError("P15.0 requires `policy_spec` to exist (run P15.2 first).")
if not isinstance(policy_spec, dict):
    raise TypeError(f"P15.0: policy_spec must be dict, got {type(policy_spec)}")
for k in ["trigger", "effect_shared", "shock_scenarios", "reference_scenario_id", "effect"]:
    if k not in policy_spec:
        raise KeyError(f"P15.0: policy_spec missing required key '{k}'. Found keys: {list(policy_spec.keys())}")

trg = policy_spec["trigger"]
eff_shared = policy_spec["effect_shared"]
eff_ref = policy_spec["effect"]
shock_scenarios = list(policy_spec["shock_scenarios"])
reference_scenario_id = str(policy_spec["reference_scenario_id"])
reference_scenario = next(row for row in shock_scenarios if str(row["scenario_id"]) == reference_scenario_id)
reference_delta = float(reference_scenario["delta_shock_ablation"])

policy_params = {
    "anchor_study": str(trg["trigger_anchor_study"]),
    "anchor_window_days": 14,
    "anchor_metric_week0": "time_on_task_uplift_day7_35pct",
    "anchor_metric_week1": "uplift_day14_lt_10pct",
    "anchor_trigger_note": str(trg["trigger_anchor_definition"]),
    "r_star": int(trg["r_star"]),
    "trigger_anchor_study": str(trg["trigger_anchor_study"]),
    "trigger_anchor_definition": str(trg["trigger_anchor_definition"]),
    "trigger_frequency": str(trg.get("trigger_frequency", "")),
    "recency_definition": str(trg.get("recency_definition", "")),
    "recency_proxy_col": str(trg.get("recency_proxy_col", "")),
    "W": int(eff_shared["W"]),
    "W_weeks": int(eff_shared["W"]),
    "alpha_week0": float(eff_shared["alpha_week0"]),
    "alpha_week1": float(eff_shared["alpha_week1"]),
    "decay_type": str(eff_shared["decay_type"]),
    "window_exclusive_upper": bool(eff_shared["window_exclusive_upper"]),
    "delta": reference_delta,
    "delta_shock_ablation": reference_delta,
    "reference_scenario_id": reference_scenario_id,
    "reference_delta_shock_ablation": reference_delta,
    "shock_scenarios": shock_scenarios,
    "effect_shared": dict(eff_shared),
    "effect": dict(eff_ref),
}

print("[policy_params written]")
for k in [
    "anchor_study",
    "anchor_window_days",
    "anchor_metric_week0",
    "anchor_metric_week1",
    "anchor_trigger_note",
    "r_star",
    "W",
    "alpha_week0",
    "alpha_week1",
    "decay_type",
    "window_exclusive_upper",
    "delta",
    "reference_scenario_id",
]:
    v = policy_params.get(k, "MISSING")
    print(f"- {k}: {v} (type={type(v).__name__})")

print("\n[shock_scenarios]")
print(pd.DataFrame(shock_scenarios)[["scenario_id", "scenario_label", "delta_shock_ablation", "scenario_status"]].to_string(index=False))
print("=== P15.0 DONE ===")

=== P15.0 — START ===
[policy_params written]
- anchor_study: Kay & Bostock (2023) — The Power of the Nudge: Technology Driving Persistence (type=str)
- anchor_window_days: 14 (type=int)
- anchor_metric_week0: time_on_task_uplift_day7_35pct (type=str)
- anchor_metric_week1: uplift_day14_lt_10pct (type=str)
- anchor_trigger_note: flag if no LMS engagement in last 7 days (≈ recency>=1 week) (type=str)
- r_star: 1 (type=int)
- W: 2 (type=int)
- alpha_week0: 0.35 (type=float)
- alpha_week1: 0.1 (type=float)
- decay_type: kb2023_step_2w (type=str)
- window_exclusive_upper: True (type=bool)
- delta: 0.2 (type=float)
- reference_scenario_id: hypothetical_A (type=str)

[shock_scenarios]
          scenario_id        scenario_label  delta_shock_ablation scenario_status
anchored_conservative Conservative anchored                  0.08        anchored
       hypothetical_A        Hypothetical A                  0.20    hypothetical
       hypothetical_B        Hypothetical B                  0.60 

In [961]:
# ==============================================================
# P17-v5 — Survival Evaluation Metrics (TEST; Stable IPCW via P14.2 Dual Horizons)
# --------------------------------------------------------------
# Purpose:
#   Compute reportable survival metrics on the TEST split using:
#     - T_policy       : policy horizon (contrast / trigger horizon)
#     - T_eval_metrics : IPCW-stable evaluation horizon (G(t) >= g_min)
#
# Key design:
#   - This step DOES NOT depend on P14 trajectories.
#   - This step DOES NOT re-derive horizons.
#   - It consumes the dual-horizon decision already made in P14.2-v1.
#
# Required inputs (must exist in the kernel):
#   - pp_test: person-period TEST frame with:
#       KEYS, week, event, hazard_hat  (baseline hazard)
#     Optional: hazard_hat_ipcw (only used for extra diagnostics if present)
#   - enrollments_eventtime: enrollment backbone with:
#       KEYS, event_withdrawn, t_event_week, t_final_week
#   - enroll_test_keys: TEST enrollment keys (from P6 split)
#   - T_policy, T_eval_metrics, T_eval_policy (from P14.2-v1)
#   - dfG_test (or table_policy_horizons_dual.csv / table_censoring_survival_G_test_by_week.csv)
#
# Outputs:
#   - outputs_v2/tables/table_survival_metrics_ipcw.csv
#   - (prints) horizons + stability + metrics + sanity
#
# Notes:
#   - We evaluate event-by-horizon probabilities:
#       p_i(T) = 1 - S_i(T)   where S_i(T) = Π_{k=0..min(T, t_final_i)} (1 - h_hat_{ik})
#   - IPCW uses censoring survival G(t) estimated on TEST (P14.2 exports).
# ==============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ------------------------------
# 0) Preconditions (fail fast)
# ------------------------------
REQUIRED_GLOBALS = ["KEYS", "pp_test", "enrollments_eventtime", "enroll_test_keys", "T_policy", "T_eval_metrics", "T_eval_policy"]
missing_globals = [k for k in REQUIRED_GLOBALS if k not in globals()]
if missing_globals:
    raise NameError(f"P17-v5 missing required globals: {missing_globals}. Run P6, P7/P11, and P14.2-v1 first.")

T_policy_local = int(T_policy)
T_eval_metrics_local = int(T_eval_metrics)
T_eval_policy_local = int(T_eval_policy)

if T_eval_metrics_local > T_eval_policy_local:
    raise ValueError(f"P17-v5: expected T_eval_metrics <= T_eval_policy, got {T_eval_metrics_local} > {T_eval_policy_local}")

# Required columns in pp_test
required_cols_pp = KEYS + ["week", "event", "hazard_hat"]
missing_cols_pp = [c for c in required_cols_pp if c not in pp_test.columns]
if missing_cols_pp:
    raise KeyError(f"P17-v5 requires columns missing from pp_test: {missing_cols_pp}")

# Required columns in enrollments_eventtime
required_cols_enr = KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]
missing_cols_enr = [c for c in required_cols_enr if c not in enrollments_eventtime.columns]
if missing_cols_enr:
    raise KeyError(f"P17-v5 requires columns missing from enrollments_eventtime: {missing_cols_enr}")

# ------------------------------
# 1) Load/resolve G(t) from P14.2 exports
# ------------------------------
# Prefer in-kernel dfG (if user kept it), else read from outputs_v2 table.
OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

G_table_path = os.path.join(TABLE_DIR, "table_censoring_survival_G_test_by_week.csv")
dual_path = os.path.join(TABLE_DIR, "table_policy_horizons_dual.csv")

dfG = None
if "dfG_test" in globals() and isinstance(globals()["dfG_test"], pd.DataFrame):
    dfG = globals()["dfG_test"].copy()
elif os.path.exists(G_table_path):
    dfG = pd.read_csv(G_table_path)
else:
    # Fall back to dual horizons table ONLY if it contains week,G_t for all weeks (usually it doesn't).
    if os.path.exists(dual_path):
        raise FileNotFoundError(
            "P17-v5: Found table_policy_horizons_dual.csv but missing the per-week G(t) table. "
            "Run P14.2-v1 to generate outputs_v2/tables/table_censoring_survival_G_test_by_week.csv."
        )
    raise FileNotFoundError(
        "P17-v5: Missing censoring survival table from P14.2. "
        "Run P14.2-v1 to generate outputs_v2/tables/table_censoring_survival_G_test_by_week.csv."
    )

if "week" not in dfG.columns or "G_t" not in dfG.columns:
    raise KeyError(f"P17-v5: dfG must contain columns ['week','G_t']. Found: {list(dfG.columns)}")

dfG["week"] = pd.to_numeric(dfG["week"], errors="raise").astype(int)
dfG["G_t"] = pd.to_numeric(dfG["G_t"], errors="coerce").astype(float)
dfG = dfG.sort_values("week").reset_index(drop=True)

T_max_G = int(dfG["week"].max())
if T_eval_policy_local > T_max_G:
    raise ValueError(f"P17-v5: T_eval_policy={T_eval_policy_local} exceeds G(t) table max week={T_max_G}")

# Build G array for quick indexing
G_arr = np.full(T_max_G + 1, np.nan, dtype=float)
G_arr[dfG["week"].to_numpy(dtype=int)] = dfG["G_t"].to_numpy(dtype=float)

# Sanity for the two horizons
G_Tpolicy = float(G_arr[T_policy_local]) if np.isfinite(G_arr[T_policy_local]) else np.nan
G_Teval_metrics = float(G_arr[T_eval_metrics_local]) if np.isfinite(G_arr[T_eval_metrics_local]) else np.nan
G_Teval_policy = float(G_arr[T_eval_policy_local]) if np.isfinite(G_arr[T_eval_policy_local]) else np.nan

if not np.isfinite(G_Teval_metrics):
    raise ValueError(f"P17-v5: G(T_eval_metrics={T_eval_metrics_local}) is not finite; check P14.2 outputs.")
if not np.isfinite(G_Tpolicy):
    raise ValueError(f"P17-v5: G(T_policy={T_policy_local}) is not finite; check P14.2 outputs.")

# ------------------------------
# 2) Enrollment backbone restricted to TEST
# ------------------------------
bb = (
    enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner", validate="m:1")
    .loc[:, KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]]
    .drop_duplicates(subset=KEYS)
    .copy()
)
bb["event_withdrawn"] = pd.to_numeric(bb["event_withdrawn"], errors="raise").astype(int)

# t_event_week is nullable for censored
# use Int64 for safe NA handling, then to_numpy with float
bb["t_event_week"] = pd.to_numeric(bb["t_event_week"], errors="coerce").astype("Int64")
bb["t_final_week"] = pd.to_numeric(bb["t_final_week"], errors="raise").astype(int)

N_test_enr = int(bb.shape[0])
N_test_events = int(bb["event_withdrawn"].sum())

# ------------------------------
# 3) Build S_i(t) grid up to T_eval_policy_local using hazard_hat
# ------------------------------
EPS = 1e-12

pp = pp_test.loc[:, KEYS + ["week", "hazard_hat"]].copy()
pp["week"] = pd.to_numeric(pp["week"], errors="raise").astype(int)
pp["hazard_hat"] = pd.to_numeric(pp["hazard_hat"], errors="raise").astype(float).clip(EPS, 1.0 - EPS)

# restrict to weeks <= T_eval_policy_local (needed to compute S(T_policy) and S(T_eval_metrics))
pp = pp.loc[pp["week"] <= T_eval_policy_local].copy()

# sort and compute cumulative survival per enrollment
pp = pp.sort_values(KEYS + ["week"]).reset_index(drop=True)
gkey = pp[KEYS].apply(tuple, axis=1)
pp["S_hat"] = (1.0 - pp["hazard_hat"]).groupby(gkey).cumprod().astype(float).clip(EPS, 1.0)

# Pivot to enrollment x week (sparse weeks allowed)
S_grid = pp.pivot_table(index=KEYS, columns="week", values="S_hat", aggfunc="first")

# Align to backbone enrollments
S_grid = S_grid.reindex(index=pd.MultiIndex.from_frame(bb[KEYS]), copy=False)

# Ensure all needed weeks exist as columns (fill with NaN if absent)
need_weeks = list(range(0, T_eval_policy_local + 1))
for t in need_weeks:
    if t not in S_grid.columns:
        S_grid[t] = np.nan
S_grid = S_grid.reindex(columns=need_weeks)

# Fill missing early weeks with 1.0 at week 0 when enrollment has no row at week 0
if S_grid[0].isna().any():
    S_grid[0] = S_grid[0].fillna(1.0)

# Forward-fill within each enrollment trajectory (enrollment ends before some weeks => remains last observed S)
S_grid = S_grid.ffill(axis=1)

# Hard check: some enrollments might have no rows at all <= T_eval_policy_local (should not happen)
all_nan = S_grid.isna().all(axis=1)
if bool(all_nan.any()):
    n_bad = int(all_nan.sum())
    raise ValueError(f"P17-v5: Found TEST enrollments with no hazard rows <= T_eval_policy. n={n_bad}")

# ------------------------------
# 4) Event probability arrays
# ------------------------------
p_event_by_t = (1.0 - S_grid.to_numpy(dtype=float)).clip(0.0, 1.0)
p_Tpolicy = p_event_by_t[:, T_policy_local]
p_Teval_metrics = p_event_by_t[:, T_eval_metrics_local]

# ------------------------------
# 5) Discrete-time C-index at horizon T (ranking-based; event vs non-event-at-risk)
# ------------------------------
def c_index_discrete(E: np.ndarray, t_event: np.ndarray, t_final: np.ndarray, p_event_by_T: np.ndarray, T: int) -> float:
    """
    Simple horizon C-index:
      Compare each event i with each non-event j that is still at risk at i's event time.
      Uses p_event_by_T as risk score.
    """
    idx_event = np.where(E == 1)[0]
    idx_none = np.where(E == 0)[0]
    if idx_event.size == 0 or idx_none.size == 0:
        return np.nan

    conc = 0.0
    ties = 0.0
    total = 0.0

    for i in idx_event:
        ti = t_event[i]
        if not np.isfinite(ti):
            continue
        ti = int(ti)
        # non-events with observation beyond ti
        js = idx_none[t_final[idx_none] >= ti]
        if js.size == 0:
            continue
        pi = p_event_by_T[i]
        pj = p_event_by_T[js]
        total += js.size
        conc += float(np.sum(pi > pj))
        ties += float(np.sum(pi == pj))

    if total <= 0:
        return np.nan
    return float((conc + 0.5 * ties) / total)

# ------------------------------
# 6) IPCW Brier at horizon T and IBS(0..T)
# ------------------------------
def brier_ipcw_event_by_T(E: np.ndarray, t_event: np.ndarray, t_final: np.ndarray, p_event_T: np.ndarray, T: int, G: np.ndarray) -> float:
    """
    IPCW Brier score for event-by-horizon indicator I(T_event <= T).
    Weights use G(t) at observed time for censoring; stabilized by clipping in G upstream.
    """
    # observed label: event by horizon
    y = ((E == 1) & (t_event <= T)).astype(float)

    # weights:
    # - if event happens by T: weight 1 / G(t_event)
    # - else (no event by T): weight 1 / G(T) for those still observed at T (t_final >= T)
    w = np.zeros_like(y, dtype=float)

    # events by T
    idx_e = np.where((E == 1) & (t_event <= T))[0]
    if idx_e.size > 0:
        te = t_event[idx_e].astype(int)
        w[idx_e] = 1.0 / np.clip(G[te], 1e-12, 1.0)

    # non-events by T but at risk at T
    idx_ne = np.where(~((E == 1) & (t_event <= T)) & (t_final >= T))[0]
    if idx_ne.size > 0:
        w[idx_ne] = 1.0 / np.clip(G[T], 1e-12, 1.0)

    # If everyone is censored before T, denominator is 0
    denom = float(np.sum(w))
    if denom <= 0:
        return np.nan

    err = (y - p_event_T) ** 2
    return float(np.sum(w * err) / denom)

def ibs_ipcw_event(E: np.ndarray, t_event: np.ndarray, t_final: np.ndarray, p_event_by_t: np.ndarray, T: int, G: np.ndarray) -> float:
    """
    IPCW Integrated Brier Score over t=0..T for event-by-time indicator I(T_event <= t).
    """
    vals = []
    for t in range(0, T + 1):
        vals.append(brier_ipcw_event_by_T(E, t_event, t_final, p_event_by_t[:, t], t, G))
    vals = np.asarray(vals, dtype=float)
    if np.all(~np.isfinite(vals)):
        return np.nan
    return float(np.nanmean(vals))

# Extract arrays
E = bb["event_withdrawn"].to_numpy(dtype=int)
t_event = bb["t_event_week"].astype("Int64").to_numpy(dtype="float")  # keep NaN for censored
t_final = bb["t_final_week"].to_numpy(dtype=int)

# C-index
cidx_Tpolicy = c_index_discrete(E, t_event, t_final, p_Tpolicy, T_policy_local)
cidx_Teval = c_index_discrete(E, t_event, t_final, p_Teval_metrics, T_eval_metrics_local)

# IPCW Brier/IBS
brier_Tpolicy = brier_ipcw_event_by_T(E, t_event, t_final, p_Tpolicy, T_policy_local, G_arr)
ibs_0_Tpolicy = ibs_ipcw_event(E, t_event, t_final, p_event_by_t[:, : (T_policy_local + 1)], T_policy_local, G_arr)

brier_Teval = brier_ipcw_event_by_T(E, t_event, t_final, p_Teval_metrics, T_eval_metrics_local, G_arr)
ibs_0_Teval = ibs_ipcw_event(E, t_event, t_final, p_event_by_t[:, : (T_eval_metrics_local + 1)], T_eval_metrics_local, G_arr)

# ------------------------------
# 7) Export table + required prints
# ------------------------------
eval_summary = pd.DataFrame(
    [
        ("T_policy", float(T_policy_local)),
        ("T_eval_metrics", float(T_eval_metrics_local)),
        ("T_eval_policy", float(T_eval_policy_local)),
        ("T_max_G_table", float(T_max_G)),
        ("G_T_policy", float(G_Tpolicy)),
        ("G_T_eval_metrics", float(G_Teval_metrics)),
        ("G_T_eval_policy", float(G_Teval_policy)),
        ("C_index_discrete_T_policy", float(cidx_Tpolicy) if np.isfinite(cidx_Tpolicy) else np.nan),
        ("C_index_discrete_T_eval_metrics", float(cidx_Teval) if np.isfinite(cidx_Teval) else np.nan),
        ("IPCW_Brier_T_policy", float(brier_Tpolicy) if np.isfinite(brier_Tpolicy) else np.nan),
        ("IPCW_IBS_0_T_policy", float(ibs_0_Tpolicy) if np.isfinite(ibs_0_Tpolicy) else np.nan),
        ("IPCW_Brier_T_eval_metrics", float(brier_Teval) if np.isfinite(brier_Teval) else np.nan),
        ("IPCW_IBS_0_T_eval_metrics", float(ibs_0_Teval) if np.isfinite(ibs_0_Teval) else np.nan),
        ("N_test_enrollments", float(N_test_enr)),
        ("N_test_events", float(N_test_events)),
    ],
    columns=["metric", "value"],
)

p_out = os.path.join(TABLE_DIR, "table_survival_metrics_ipcw.csv")
eval_summary.to_csv(p_out, index=False)

print("=== P17-v5 — Survival Metrics (TEST; Stable IPCW via P14.2) ===")
print("[Horizons]")
print(f"T_policy      : {T_policy_local}")
print(f"T_eval_policy : {T_eval_policy_local} (raw TEST support)")
print(f"T_eval_metrics: {T_eval_metrics_local} (stable IPCW)")
print("\n[Censoring survival at horizons]")
print(f"G(T_policy)      : {G_Tpolicy}")
print(f"G(T_eval_policy) : {G_Teval_policy}")
print(f"G(T_eval_metrics): {G_Teval_metrics}")

print("\n[Metrics table]")
print(eval_summary.to_string(index=False))

print("\n[Sanity]")
print("Brier(T_policy):", brier_Tpolicy, "| IBS(0..T_policy):", ibs_0_Tpolicy)
print("Brier(T_eval_metrics):", brier_Teval, "| IBS(0..T_eval_metrics):", ibs_0_Teval)
print("C-index(T_policy):", cidx_Tpolicy, "| C-index(T_eval_metrics):", cidx_Teval)

print("\n[Export]")
print(p_out)
print("=== P17-v5 DONE ===")

[TABLE] saved: outputs_v2/tables/table_survival_metrics_ipcw.csv | shape=(15, 2)
columns: ['metric', 'value']
sample (n=5):
           metric      value
0        T_policy  18.000000
1  T_eval_metrics  37.000000
2   T_eval_policy  38.000000
3   T_max_G_table  38.000000
4      G_T_policy   0.795798
=== P17-v5 — Survival Metrics (TEST; Stable IPCW via P14.2) ===
[Horizons]
T_policy      : 18
T_eval_policy : 38 (raw TEST support)
T_eval_metrics: 37 (stable IPCW)

[Censoring survival at horizons]
G(T_policy)      : 0.7957980376262404
G(T_eval_policy) : 1e-12
G(T_eval_metrics): 0.0600718928600871

[Metrics table]
                         metric        value
                       T_policy 1.800000e+01
                 T_eval_metrics 3.700000e+01
                  T_eval_policy 3.800000e+01
                  T_max_G_table 3.800000e+01
                     G_T_policy 7.957980e-01
               G_T_eval_metrics 6.007189e-02
                G_T_eval_policy 1.000000e-12
      C_index_discrete_T_

# P17.6 — Unified Benchmark Pack (Temporal vs Non-Temporal)

In [968]:
# ==============================================================
# P17.6-v1 — Unified Benchmark Pack (Temporal vs Non-Temporal)
# --------------------------------------------------------------
# Purpose:
#   Consolidate benchmark evidence into one comparable view:
#   - Temporal hazard models (base/ipcw) at enrollment horizons
#   - Non-temporal enrollment baseline (P9.1)
#   - Row-level diagnostics (discrete-time logit + HistGB)
#
# Outputs (source of truth):
#   - outputs_v2/tables/table_benchmark_horizon_auc.csv
#   - outputs_v2/tables/table_benchmark_row_level_auc.csv
#   - outputs_v2/tables/table_benchmark_unified.csv
#   - outputs_v2/figures/fig_benchmark_dual_panel_auc.png
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# ------------------------------
# 0) Preconditions
# ------------------------------
required = ["KEYS", "pp_test", "enrollments_eventtime"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P17.6-v1 missing required objects: {missing}")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

T_policy_local = int(globals().get("T_policy", 18))
T_eval_local = int(globals().get("T_eval_metrics", max(T_policy_local, 30)))

# ------------------------------
# 1) Build enrollment backbone on TEST
# ------------------------------
if "enroll_test_keys" in globals():
    test_keys = enroll_test_keys[KEYS].drop_duplicates().copy()
else:
    test_keys = pp_test[KEYS].drop_duplicates().copy()

base_cols = KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]
missing_base = [c for c in base_cols if c not in enrollments_eventtime.columns]
if missing_base:
    raise KeyError(f"P17.6-v1 missing columns in enrollments_eventtime: {missing_base}")

bb = (
    enrollments_eventtime[base_cols]
    .drop_duplicates(subset=KEYS)
    .merge(test_keys, on=KEYS, how="inner", validate="1:1")
    .copy()
)

bb["event_withdrawn"] = pd.to_numeric(bb["event_withdrawn"], errors="coerce").fillna(0).astype(int)
bb["t_event_week"] = pd.to_numeric(bb["t_event_week"], errors="coerce")
bb["t_final_week"] = pd.to_numeric(bb["t_final_week"], errors="coerce").fillna(0).astype(int)

def event_by_horizon(df: pd.DataFrame, T: int) -> np.ndarray:
    t_event = pd.to_numeric(df["t_event_week"], errors="coerce").astype(float).fillna(np.inf).to_numpy(dtype=float)
    e = df["event_withdrawn"].astype(int).to_numpy()
    return ((e == 1) & (t_event <= T)).astype(int)

y_policy = event_by_horizon(bb, T_policy_local)
y_eval = event_by_horizon(bb, T_eval_local)

# ------------------------------
# 2) Temporal hazard -> enrollment event probability by horizon
# ------------------------------
def horizon_prob_from_hazard(pp_df: pd.DataFrame, hazard_col: str, bb_df: pd.DataFrame, T: int) -> np.ndarray:
    req = KEYS + ["week", hazard_col]
    miss = [c for c in req if c not in pp_df.columns]
    if miss:
        raise KeyError(f"P17.6-v1 requires {miss} in pp_test for hazard_col={hazard_col}")

    tmp = pp_df[req].copy()
    tmp["week"] = pd.to_numeric(tmp["week"], errors="coerce").fillna(0).astype(int)
    tmp[hazard_col] = pd.to_numeric(tmp[hazard_col], errors="coerce").fillna(0.0).clip(1e-12, 1 - 1e-12)
    tmp = tmp.loc[tmp["week"] <= T].sort_values(KEYS + ["week"]).reset_index(drop=True)

    gkey = tmp[KEYS].apply(tuple, axis=1)
    tmp["S_hat_tmp"] = (1.0 - tmp[hazard_col]).groupby(gkey).cumprod().astype(float).clip(1e-12, 1.0)

    s_grid = tmp.pivot_table(index=KEYS, columns="week", values="S_hat_tmp", aggfunc="first")
    for t in range(0, T + 1):
        if t not in s_grid.columns:
            s_grid[t] = np.nan
    s_grid = s_grid.reindex(columns=list(range(0, T + 1)))
    s_grid = s_grid.reindex(index=pd.MultiIndex.from_frame(bb_df[KEYS]))

    if 0 in s_grid.columns:
        s_grid[0] = s_grid[0].fillna(1.0)
    s_grid = s_grid.ffill(axis=1)

    if s_grid.isna().any().any():
        raise ValueError(f"P17.6-v1 could not build complete survival grid for hazard_col={hazard_col}")

    p_event = (1.0 - s_grid.to_numpy(dtype=float)[:, T]).clip(1e-12, 1 - 1e-12)
    return p_event

model_rows = []

# Base temporal hazard (active hazard_hat)
p_base_policy = horizon_prob_from_hazard(pp_test, "hazard_hat", bb, T_policy_local)
p_base_eval = horizon_prob_from_hazard(pp_test, "hazard_hat", bb, T_eval_local)
model_rows.append({
    "model": "temporal_hazard_active",
    "family": "discrete_time",
    "AUC_event_by_T_policy": float(roc_auc_score(y_policy, p_base_policy)) if len(np.unique(y_policy)) > 1 else np.nan,
    "AUC_event_by_T_eval": float(roc_auc_score(y_eval, p_base_eval)) if len(np.unique(y_eval)) > 1 else np.nan,
    "n_test_enrollments": int(len(bb)),
    "source": "pp_test.hazard_hat",
})

# Optional IPCW hazard if present
if "hazard_hat_ipcw" in pp_test.columns:
    p_ipcw_policy = horizon_prob_from_hazard(pp_test, "hazard_hat_ipcw", bb, T_policy_local)
    p_ipcw_eval = horizon_prob_from_hazard(pp_test, "hazard_hat_ipcw", bb, T_eval_local)
    model_rows.append({
        "model": "temporal_hazard_ipcw",
        "family": "discrete_time_ipcw",
        "AUC_event_by_T_policy": float(roc_auc_score(y_policy, p_ipcw_policy)) if len(np.unique(y_policy)) > 1 else np.nan,
        "AUC_event_by_T_eval": float(roc_auc_score(y_eval, p_ipcw_eval)) if len(np.unique(y_eval)) > 1 else np.nan,
        "n_test_enrollments": int(len(bb)),
        "source": "pp_test.hazard_hat_ipcw",
    })

# Non-temporal RSF baseline from in-memory P9.1 objects.
if "pred_df" in globals() and isinstance(pred_df, pd.DataFrame):
    rsf_pred = pred_df.copy()

    # Accept both naming styles used across sections.
    if {"risk_policy", "risk_eval"}.issubset(rsf_pred.columns):
        rsf_pred = rsf_pred.rename(
            columns={
                "risk_policy": "p_event_by_T_policy",
                "risk_eval": "p_event_by_T_eval",
            }
        )

    req_prob = ["p_event_by_T_policy", "p_event_by_T_eval"]
    if set(req_prob).issubset(rsf_pred.columns):
        join_keys = [k for k in KEYS if k in rsf_pred.columns and k in bb.columns]
        if (not join_keys) and ("id_student" in rsf_pred.columns) and ("id_student" in bb.columns):
            join_keys = ["id_student"]

        if join_keys:
            rsf_small = rsf_pred[join_keys + req_prob].drop_duplicates(subset=join_keys).copy()
            rsf_join = bb[join_keys].merge(rsf_small, on=join_keys, how="left", validate="1:1")

            if rsf_join[req_prob].isna().any().any():
                raise ValueError("P17.6-v1 RSF in-memory predictions do not fully match TEST enrollment backbone.")

            p_rsf_policy = rsf_join["p_event_by_T_policy"].to_numpy(dtype=float)
            p_rsf_eval = rsf_join["p_event_by_T_eval"].to_numpy(dtype=float)

            model_rows.append({
                "model": "non_temporal_rsf",
                "family": "non_temporal_survival",
                "AUC_event_by_T_policy": float(roc_auc_score(y_policy, p_rsf_policy)) if len(np.unique(y_policy)) > 1 else np.nan,
                "AUC_event_by_T_eval": float(roc_auc_score(y_eval, p_rsf_eval)) if len(np.unique(y_eval)) > 1 else np.nan,
                "n_test_enrollments": int(len(bb)),
                "source": "pred_df(in_memory)",
            })

horizon_df = pd.DataFrame(model_rows)

# ------------------------------
# 3) Row-level AUC diagnostics (in-memory)
# ------------------------------
row_rows = []

if {"event", "hazard_hat"}.issubset(pp_test.columns):
    row_rows.append({
        "model": "discrete_time_logit",
        "family": "row_level_temporal",
        "AUC_row_test": float(roc_auc_score(pp_test["event"].astype(int), pd.to_numeric(pp_test["hazard_hat"], errors="coerce").fillna(0.0))),
        "source": "pp_test.hazard_hat",
    })

if {"event", "hazard_hat_ipcw"}.issubset(pp_test.columns):
    row_rows.append({
        "model": "discrete_time_logit_ipcw",
        "family": "row_level_temporal",
        "AUC_row_test": float(roc_auc_score(pp_test["event"].astype(int), pd.to_numeric(pp_test["hazard_hat_ipcw"], errors="coerce").fillna(0.0))),
        "source": "pp_test.hazard_hat_ipcw",
    })

if {"event", "gb_hazard_hat"}.issubset(pp_test.columns):
    row_rows.append({
        "model": "histgb_optional_baseline",
        "family": "row_level_temporal",
        "AUC_row_test": float(roc_auc_score(pp_test["event"].astype(int), pd.to_numeric(pp_test["gb_hazard_hat"], errors="coerce").fillna(0.0))),
        "source": "pp_test.gb_hazard_hat",
    })

row_df = pd.DataFrame(row_rows)

# ------------------------------
# 4) Unified export table
# ------------------------------
unified_df = horizon_df.copy()
if not row_df.empty:
    row_aug = row_df.copy()
    row_aug["AUC_event_by_T_policy"] = np.nan
    row_aug["AUC_event_by_T_eval"] = np.nan
    row_aug["n_test_enrollments"] = np.nan
    row_aug = row_aug[["model", "family", "AUC_event_by_T_policy", "AUC_event_by_T_eval", "n_test_enrollments", "source"]]
    unified_df = pd.concat([unified_df, row_aug], axis=0, ignore_index=True)

p_horizon = os.path.join(TABLE_DIR, "table_benchmark_horizon_auc.csv")
p_row = os.path.join(TABLE_DIR, "table_benchmark_row_level_auc.csv")
p_unified = os.path.join(TABLE_DIR, "table_benchmark_unified.csv")

horizon_df.to_csv(p_horizon, index=False)
row_df.to_csv(p_row, index=False)
unified_df.to_csv(p_unified, index=False)

# ------------------------------
# 5) Paper figure (dual-panel benchmark)
# ------------------------------
fig_path = os.path.join(FIG_DIR, "fig_benchmark_dual_panel_auc.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# Left panel: row-level AUC
if not row_df.empty:
    plot_row = row_df.copy().sort_values("AUC_row_test", ascending=False)
    axes[0].bar(plot_row["model"], plot_row["AUC_row_test"])
    axes[0].set_title("Row-level AUC (test)")
    axes[0].set_ylabel("AUC")
    axes[0].set_ylim(0.4, 1.0)
    axes[0].tick_params(axis="x", rotation=25)
else:
    axes[0].text(0.5, 0.5, "No row-level benchmark available", ha="center", va="center")
    axes[0].set_axis_off()

# Right panel: enrollment-level horizon AUC at T_eval
if not horizon_df.empty:
    plot_h = horizon_df.copy().sort_values("AUC_event_by_T_eval", ascending=False)
    axes[1].bar(plot_h["model"], plot_h["AUC_event_by_T_eval"])
    axes[1].set_title(f"Enrollment event-by-horizon AUC (T_eval={T_eval_local})")
    axes[1].set_ylabel("AUC")
    axes[1].set_ylim(0.4, 1.0)
    axes[1].tick_params(axis="x", rotation=25)
else:
    axes[1].text(0.5, 0.5, "No horizon benchmark available", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.savefig(fig_path, dpi=220)
plt.close()

# ------------------------------
# 6) Required console outputs
# ------------------------------
print("=== P17.6-v1 — Unified Benchmark Pack (Temporal vs Non-Temporal) ===")
print("[Horizon setup]")
print("T_policy      :", T_policy_local)
print("T_eval_metrics:", T_eval_local)
print("N_test_enrollments(backbone):", len(bb))

print("\n[Horizon benchmark table]")
if horizon_df.empty:
    print("(empty)")
else:
    print(horizon_df.to_string(index=False))

print("\n[Row-level benchmark table]")
if row_df.empty:
    print("(empty)")
else:
    print(row_df.to_string(index=False))

print("\n[Unified benchmark table]")
print(unified_df.to_string(index=False))

print("\n[Artifacts]")
print("horizon_table :", p_horizon)
print("row_table     :", p_row)
print("unified_table :", p_unified)
print("benchmark_fig :", fig_path)
print("=== P17.6-v1 DONE ===")

[TABLE] saved: outputs_v2/tables/table_benchmark_horizon_auc.csv | shape=(3, 6)
columns: ['model', 'family', 'AUC_event_by_T_policy', 'AUC_event_by_T_eval', 'n_test_enrollments', 'source']
sample (n=3):
                    model                 family  AUC_event_by_T_policy  \
0  temporal_hazard_active          discrete_time               0.480270   
1    temporal_hazard_ipcw     discrete_time_ipcw               0.522024   
2        non_temporal_rsf  non_temporal_survival               0.645659   

   AUC_event_by_T_eval  n_test_enrollments                   source  
0             0.497923                9778       pp_test.hazard_hat  
1             0.529068                9778  pp_test.hazard_hat_ipcw  
2             0.638836                9778       pred_df(in_memory)  
[TABLE] saved: outputs_v2/tables/table_benchmark_row_level_auc.csv | shape=(2, 4)
columns: ['model', 'family', 'AUC_row_test', 'source']
sample (n=2):
                      model              family  AUC_row_test  \


In [969]:
# ==============================================================
# P18-v4 — Ablation Study (M4) WITHOUT leakage (aligned) + G_t resolver
# --------------------------------------------------------------
# Purpose:
#   Fix P18 failures caused by missing censoring survival object `G_t`.
#   This version resolves G(t) from (in priority order):
#     1) G_t (already in globals)
#     2) G_t_test (already in globals)
#     3) dfG_test (already in globals)
#     4) CSV on disk: outputs_v2/tables/table_censoring_survival_G_test_by_week.csv
#
# Key guarantee:
#   After this cell, BOTH are materialized:
#     - dfG_test  : DataFrame with columns ["week","G_t"]
#     - G_t       : numpy array aligned with week index (0..T_max_G_table)
#
# Horizons:
#   - T_policy       : from globals (usually 18)
#   - T_eval_metrics : from globals (stable IPCW horizon)
#
# Outputs:
#   - outputs_v2/tables/table_ablation_m4.csv
# ==============================================================

import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

# ------------------------------
# 0) Preconditions (fail fast)
# ------------------------------
required = [
    "KEYS", "pp_train", "pp_test",
    "enrollments_eventtime", "enroll_test_keys",
    "numeric_features", "categorical_features",
    "T_policy", "T_eval_metrics",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P18-v4 missing required objects: {missing}")

if "event" not in pp_train.columns or "event" not in pp_test.columns:
    raise KeyError("P18-v4 requires column 'event' in pp_train and pp_test.")

T_policy_local = int(T_policy)
T_eval_local = int(T_eval_metrics)

print("=== P18-v4 — Ablation horizons (aligned) ===")
print("T_policy:", T_policy_local)
print("T_eval  :", T_eval_local)

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)
path_out = os.path.join(TABLE_DIR, "table_ablation_m4.csv")

# ------------------------------
# 1) Resolve censoring survival G(t) robustly
# ------------------------------
def _materialize_dfG_and_Gt() -> tuple[pd.DataFrame, np.ndarray]:
    """
    Returns (dfG_test, G_t) with:
      dfG_test columns: ["week","G_t"] (week int ascending)
      G_t array: length = max_week+1, aligned so G_t[t] corresponds to week t
    Also writes dfG_test and G_t back into globals() for downstream stability.
    """
    # Case 1: G_t already exists
    if "G_t" in globals() and globals()["G_t"] is not None:
        G = np.asarray(globals()["G_t"], dtype=float)
        df = pd.DataFrame({"week": np.arange(len(G), dtype=int), "G_t": G})
        globals()["dfG_test"] = df
        globals()["G_t"] = G
        return df, G

    # Case 2: G_t_test exists
    if "G_t_test" in globals() and globals()["G_t_test"] is not None:
        G = np.asarray(globals()["G_t_test"], dtype=float)
        df = pd.DataFrame({"week": np.arange(len(G), dtype=int), "G_t": G})
        globals()["dfG_test"] = df
        globals()["G_t"] = G
        return df, G

    # Case 3: dfG_test exists
    if "dfG_test" in globals() and isinstance(globals()["dfG_test"], pd.DataFrame):
        df = globals()["dfG_test"].copy()
        if "week" not in df.columns or "G_t" not in df.columns:
            raise KeyError("P18-v4: dfG_test exists but must contain columns ['week','G_t'].")
        df["week"] = pd.to_numeric(df["week"], errors="raise").astype(int)
        df["G_t"] = pd.to_numeric(df["G_t"], errors="raise").astype(float)
        df = df.sort_values("week").reset_index(drop=True)

        max_w = int(df["week"].max())
        G = np.full(max_w + 1, np.nan, dtype=float)
        G[df["week"].to_numpy(dtype=int)] = df["G_t"].to_numpy(dtype=float)

        # Fill any gaps defensively (shouldn't happen, but keep stable)
        if np.isnan(G[0]):
            G[0] = float(df.loc[df["week"] == 0, "G_t"].iloc[0]) if (df["week"] == 0).any() else 1.0
        G = pd.Series(G).ffill().to_numpy(dtype=float)

        globals()["dfG_test"] = df
        globals()["G_t"] = G
        return df, G

    # Case 4: load from CSV on disk
    csv_path = os.path.join(TABLE_DIR, "table_censoring_survival_G_test_by_week.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            "P18-v4: could not resolve G(t). None of [G_t, G_t_test, dfG_test] exist, "
            f"and CSV not found: {csv_path}. Run P14.2 (or P17.1-v2) first."
        )

    df = pd.read_csv(csv_path)
    if "week" not in df.columns or "G_t" not in df.columns:
        raise KeyError(f"P18-v4: {csv_path} must contain columns ['week','G_t']. Found: {list(df.columns)}")

    df["week"] = pd.to_numeric(df["week"], errors="raise").astype(int)
    df["G_t"] = pd.to_numeric(df["G_t"], errors="raise").astype(float)
    df = df.sort_values("week").reset_index(drop=True)

    max_w = int(df["week"].max())
    G = np.full(max_w + 1, np.nan, dtype=float)
    G[df["week"].to_numpy(dtype=int)] = df["G_t"].to_numpy(dtype=float)
    if np.isnan(G[0]):
        # For safety; should not happen
        G[0] = 1.0
    G = pd.Series(G).ffill().to_numpy(dtype=float)

    globals()["dfG_test"] = df
    globals()["G_t"] = G
    return df, G

dfG_test_local, G = _materialize_dfG_and_Gt()

print("\n[P18-v4][G(t) resolved]")
print("dfG_test shape:", dfG_test_local.shape)
print("G_t length:", int(len(G)))
print("G_t[0]:", float(G[0]))
print("G_t[min,max]:", float(np.min(G)), float(np.max(G)))

if T_eval_local >= len(G):
    raise ValueError(f"P18-v4: T_eval_local={T_eval_local} exceeds length of G_t ({len(G)}). Check horizon selection.")

# ------------------------------
# 2) Enrollment backbone for IPCW evaluation (TEST)
# ------------------------------
bb = (
    enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner")
    .loc[:, KEYS + ["event_withdrawn", "t_event_week", "t_final_week"]]
    .drop_duplicates(subset=KEYS)
    .copy()
)

bb["event_withdrawn"] = pd.to_numeric(bb["event_withdrawn"], errors="raise").astype(int)
bb["t_final_week"] = pd.to_numeric(bb["t_final_week"], errors="coerce").fillna(0).astype(int)

# t_event_week can be missing for censored; keep as float and set placeholder -1
bb["t_event_week"] = pd.to_numeric(bb["t_event_week"], errors="coerce")
bb["t_event_week"] = bb["t_event_week"].fillna(-1).astype(int)

E = bb["event_withdrawn"].to_numpy(dtype=int)
t_event = bb["t_event_week"].to_numpy(dtype=int)
t_final = bb["t_final_week"].to_numpy(dtype=int)

# ------------------------------
# 3) Utility metrics (discrete proxy; consistent with your P17 style)
# ------------------------------
def c_index_discrete(E, t_event, t_final, p_event_by_T, T):
    idx_event = np.where((E == 1) & (t_event >= 0) & (t_event <= T))[0]
    idx_none = np.where((E == 0) & (t_final >= 0))[0]
    if idx_event.size == 0 or idx_none.size == 0:
        return np.nan

    conc = 0.0
    ties = 0.0
    total = 0.0
    for i in idx_event:
        ti = t_event[i]
        js = idx_none[t_final[idx_none] >= ti]
        if js.size == 0:
            continue
        pi = p_event_by_T[i]
        pj = p_event_by_T[js]
        total += js.size
        conc += float(np.sum(pi > pj))
        ties += float(np.sum(pi == pj))
    if total == 0:
        return np.nan
    return float((conc + 0.5 * ties) / total)

def brier_ipcw_event_by_T(E, t_event, t_final, p_event_by_T, T, G):
    # IPCW for event-by-horizon; conservative definition
    # Weight:
    #   if event by T -> 1/G(t_event)
    #   else (no event by T) -> 1/G(min(T, t_final))
    eps = 1e-12
    out = []
    for i in range(len(E)):
        yi = 1.0 if (E[i] == 1 and t_event[i] >= 0 and t_event[i] <= T) else 0.0
        ti = t_event[i] if yi == 1.0 else min(T, max(t_final[i], 0))
        gi = float(G[min(int(ti), len(G) - 1)])
        gi = max(gi, eps)
        wi = 1.0 / gi
        out.append(wi * (yi - float(p_event_by_T[i])) ** 2)
    return float(np.mean(out))

def ibs_ipcw_event(E, t_event, t_final, p_event_by_t, T, G):
    # Integrated Brier from 0..T (inclusive)
    scores = []
    for tt in range(0, T + 1):
        scores.append(brier_ipcw_event_by_T(E, t_event, t_final, p_event_by_t[:, tt], tt, G))
    return float(np.mean(scores))

# ------------------------------
# 4) Build “no leakage” feature lists
# ------------------------------
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration",
    "hazard_hat", "hazard_hat_ipcw", "hazard_hat_ipcw_raw",
    "S_hat", "S0_hat", "S_shock_hat", "S_mech_hat",
    "log1m_h", "logS_hat",
}

num_all = [c for c in list(numeric_features) if c in pp_train.columns and c not in FORBIDDEN]
cat_all = [c for c in list(categorical_features) if c in pp_train.columns and c not in FORBIDDEN]

# ------------------------------
# 5) Define ablation variants (minimal)
# ------------------------------
variants = [
    ("Full", dict(drop_num=[], drop_cat=[])),
    ("No Recency/Streak", dict(drop_num=["recency", "streak"], drop_cat=[])),
    ("No Activity (total_clicks)", dict(drop_num=["total_clicks"], drop_cat=[])),
]

results = []

for name, spec in variants:
    num_cols = [c for c in num_all if c not in set(spec["drop_num"])]
    cat_cols = [c for c in cat_all if c not in set(spec["drop_cat"])]

    # Must have at least 1 feature
    if (len(num_cols) + len(cat_cols)) == 0:
        raise ValueError(f"P18-v4: Variant '{name}' has no features after drops.")

    # Preprocessor (dense output via sparse_threshold=0.0 to avoid downstream surprises)
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    base = LogisticRegression(
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )

    pipe = Pipeline([("pre", pre), ("lr", base)])

    ytr = pp_train["event"].astype(int).to_numpy()
    yte = pp_test["event"].astype(int).to_numpy()

    calib = CalibratedClassifierCV(estimator=pipe, method="sigmoid", cv=5, ensemble=False)
    calib.fit(pp_train[num_cols + cat_cols], ytr)

    # Row-level AUC diagnostic
    p_row = calib.predict_proba(pp_test[num_cols + cat_cols])[:, 1].astype(float)
    auc_row = float(roc_auc_score(yte, p_row))

    # Build S_hat(t) on TEST up to T_eval_local
    tmp = pp_test.loc[pp_test["week"].astype(int) <= T_eval_local, KEYS + ["week"]].copy()
    tmp["week"] = pd.to_numeric(tmp["week"], errors="raise").astype(int)

    hazard = calib.predict_proba(
        pp_test.loc[pp_test["week"].astype(int) <= T_eval_local, num_cols + cat_cols]
    )[:, 1].astype(float)
    hazard = np.clip(hazard, 1e-12, 1 - 1e-12)
    tmp["hazard"] = hazard

    tmp = tmp.sort_values(KEYS + ["week"]).reset_index(drop=True)
    gkey_tmp = tmp[KEYS].apply(tuple, axis=1)
    tmp["S_hat"] = (1.0 - tmp["hazard"]).groupby(gkey_tmp).cumprod().astype(float).clip(1e-12, 1.0)

    S_grid = tmp.pivot_table(index=KEYS, columns="week", values="S_hat", aggfunc="first")
    for t in range(0, T_eval_local + 1):
        if t not in S_grid.columns:
            S_grid[t] = np.nan
    S_grid = S_grid.reindex(columns=list(range(0, T_eval_local + 1)))

    # Align to enrollment backbone
    S_grid = S_grid.loc[bb.set_index(KEYS).index]

    # Fill survival
    if 0 in S_grid.columns:
        S_grid[0] = S_grid[0].fillna(1.0)
    S_grid = S_grid.ffill(axis=1)

    if S_grid.isna().any().any():
        miss_rows = int(S_grid.isna().all(axis=1).sum())
        raise ValueError(f"P18-v4: Variant '{name}' has enrollments with no S_hat rows after alignment. n={miss_rows}")

    p_event_by_t = (1.0 - S_grid.to_numpy(dtype=float)).clip(0.0, 1.0)
    p_Teval = p_event_by_t[:, T_eval_local]

    cidx = c_index_discrete(E, t_event, t_final, p_Teval, T_eval_local)
    brier = brier_ipcw_event_by_T(E, t_event, t_final, p_Teval, T_eval_local, G)
    ibs = ibs_ipcw_event(E, t_event, t_final, p_event_by_t, T_eval_local, G)

    results.append({
        "model": name,
        "AUC_row_test": auc_row,
        "C_index_discrete_T_eval": cidx,
        "IPCW_Brier_T_eval": brier,
        "IPCW_IBS_0_T_eval": ibs,
        "n_numeric": len(num_cols),
        "n_categorical": len(cat_cols),
        "n_features_total": int(len(num_cols) + len(cat_cols)),
    })

# ------------------------------
# 6) Export + prints
# ------------------------------
ablation_results = pd.DataFrame(results)
ablation_results.to_csv(path_out, index=False)

print("\n=== P18-v4 — Ablation Results (NO leakage; aligned) ===")
print(ablation_results[[
    "model","AUC_row_test","C_index_discrete_T_eval",
    "IPCW_Brier_T_eval","IPCW_IBS_0_T_eval",
    "n_numeric","n_categorical","n_features_total"
]].to_string(index=False))

print("\n[Export]")
print(path_out)
print("=== P18-v4 DONE ===")

=== P18-v4 — Ablation horizons (aligned) ===
T_policy: 18
T_eval  : 37

[P18-v4][G(t) resolved]
dfG_test shape: (39, 2)
G_t length: 39
G_t[0]: 0.8989568418899571
G_t[min,max]: 1e-12 0.8989568418899571
[TABLE] saved: outputs_v2/tables/table_ablation_m4.csv | shape=(3, 8)
columns: ['model', 'AUC_row_test', 'C_index_discrete_T_eval', 'IPCW_Brier_T_eval', 'IPCW_IBS_0_T_eval', 'n_numeric', 'n_categorical', 'n_features_total']
sample (n=3):
                        model  AUC_row_test  C_index_discrete_T_eval  \
0                        Full      0.839575                 0.395071   
1           No Recency/Streak      0.798731                 0.363701   
2  No Activity (total_clicks)      0.826349                 0.336870   

   IPCW_Brier_T_eval  IPCW_IBS_0_T_eval  n_numeric  n_categorical  \
0           0.339244           0.174652          6              6   
1           0.372773           0.177165          4              6   
2           0.353013           0.178051          5              6

In [971]:
# ==============================================================
# P19.1-v4 — Leave-One-Run-Out (K runs) with Guardrails (NO duplicate columns)
# --------------------------------------------------------------
# Purpose:
#   Run a deterministic Leave-One-Run-Out evaluation on K course-runs
#   (code_module, code_presentation), filtered by minimum enrollment/event counts.
#
# Key fix:
#   Always build the run table from a "clean" slice of enrollments_eventtime
#   using safe_select()/uniq_cols() from P2.2 so groupby never breaks due to
#   duplicate column names.
#
# Requires:
#   - KEYS
#   - enrollments_eventtime
#   - pp_features (person-period full, with engineered features)
#   - P6 split keys: enroll_train_keys, enroll_test_keys (optional; not used here)
#   - Feature lists used by your main hazard model:
#       numeric_features, categorical_features
#   - Guardrails from P2.2:
#       uniq_cols(), safe_select()
#
# Outputs:
#   - outputs_v2/tables/table_loo_run_k{K}.csv
#   - Printed selection table + per-run metrics + aggregate summary
# ==============================================================

import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("=== P19.1-v4 — START ===")

# ------------------------------
# 0) Preconditions (fail fast)
# ------------------------------
required = [
    "KEYS", "enrollments_eventtime", "pp_features",
    "numeric_features", "categorical_features",
    "uniq_cols", "safe_select",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P19.1-v4 missing required objects: {missing}")

KEYS = list(KEYS)

# ------------------------------
# 1) Config (edit here)
# ------------------------------
K = 5
MIN_ENROLL = 250
MIN_EVENTS = 30
RANDOM_STATE = 42

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

# ------------------------------
# 2) Build a CLEAN run table (prevents groupby crash)
# ------------------------------
cols_needed = KEYS + ["code_module", "code_presentation", "event_withdrawn", "t_event_week", "t_final_week"]
enr = safe_select(enrollments_eventtime, cols_needed).drop_duplicates(subset=KEYS).copy()

# Ensure unique columns after any prior merges
enr = enr.loc[:, uniq_cols(enr.columns)].copy()

# Validate required columns exist and are 1-D
for c in ["code_module", "code_presentation", "event_withdrawn"]:
    if c not in enr.columns:
        raise KeyError(f"P19.1-v4: required column missing in enr: '{c}'")
    if isinstance(enr[c], pd.DataFrame):
        raise ValueError(f"P19.1-v4: '{c}' is not 1-D (duplicate column). Guardrail failed.")

enr["event_withdrawn"] = pd.to_numeric(enr["event_withdrawn"], errors="coerce").fillna(0).astype(int)

run_tab = (
    enr.groupby(["code_module", "code_presentation"], as_index=False)
       .agg(n_enrollments=("event_withdrawn", "size"),
            n_events=("event_withdrawn", "sum"))
)

eligible = run_tab[(run_tab["n_enrollments"] >= MIN_ENROLL) & (run_tab["n_events"] >= MIN_EVENTS)].copy()
if eligible.empty:
    raise ValueError(
        f"P19.1-v4: No eligible runs found with MIN_ENROLL={MIN_ENROLL} and MIN_EVENTS={MIN_EVENTS}."
    )

# Deterministic ordering (no randomness)
eligible = eligible.sort_values(
    ["n_enrollments", "n_events", "code_module", "code_presentation"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

selected = eligible.head(K).copy()

print("\n=== P19.1-v4 — Selected runs (deterministic) ===")
print(f"K={K} | MIN_ENROLL={MIN_ENROLL} | MIN_EVENTS={MIN_EVENTS}")
print(selected.to_string(index=False))

# ------------------------------
# 3) Feature set (NO leakage)
# ------------------------------
FORBIDDEN = {
    "event", "event_withdrawn",
    "t_event_week", "t_final_week", "t_last_obs_week",
    "date_unregistration",
    "hazard_hat", "hazard_hat_ipcw", "hazard_hat_ipcw_raw",
    "S_hat", "S0_hat", "S_shock_hat", "S_mech_hat",
    "log1m_h", "logS_hat",
}

num_cols = [c for c in list(numeric_features) if (c in pp_features.columns) and (c not in FORBIDDEN)]
cat_cols = [c for c in list(categorical_features) if (c in pp_features.columns) and (c not in FORBIDDEN)]

if len(num_cols) + len(cat_cols) == 0:
    raise ValueError("P19.1-v4: No usable features found after leakage filter.")

print("\n[P19.1-v4][Feature set]")
print("n_numeric:", len(num_cols), "| numeric:", num_cols)
print("n_categorical:", len(cat_cols), "| categorical:", cat_cols)

# ------------------------------
# 4) Build model (simple, consistent with notebook family)
# ------------------------------
pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

lr = LogisticRegression(
    solver="lbfgs",
    max_iter=4000,
    class_weight="balanced",
    n_jobs=None,
    random_state=RANDOM_STATE,
)

model = Pipeline([("pre", pre), ("lr", lr)])

# ------------------------------
# 5) Run loop
# ------------------------------
results = []

# person-period must contain event column
if "event" not in pp_features.columns:
    raise KeyError("P19.1-v4 requires 'event' in pp_features (person-period target).")

pp_all = pp_features.copy()
pp_all = pp_all.loc[:, uniq_cols(pp_all.columns)].copy()

for _, row in selected.iterrows():
    m = str(row["code_module"])
    p = str(row["code_presentation"])

    is_test = (pp_all["code_module"].astype(str) == m) & (pp_all["code_presentation"].astype(str) == p)
    train_df = pp_all.loc[~is_test].copy()
    test_df  = pp_all.loc[is_test].copy()

    # Basic sanity
    n_test_rows = int(test_df.shape[0])
    if n_test_rows == 0:
        raise ValueError(f"P19.1-v4: held-out run produced zero rows: {m} {p}")

    ytr = train_df["event"].astype(int).to_numpy()
    yte = test_df["event"].astype(int).to_numpy()

    # Fit + predict
    model.fit(train_df[num_cols + cat_cols], ytr)
    p_te = model.predict_proba(test_df[num_cols + cat_cols])[:, 1].astype(float)

    auc = float(roc_auc_score(yte, p_te)) if (len(np.unique(yte)) > 1) else np.nan

    results.append({
        "heldout_code_module": m,
        "heldout_code_presentation": p,
        "n_rows_test": n_test_rows,
        "pos_rate_test": float(np.mean(yte)),
        "AUC_row_test_ood": auc,
        "n_enrollments_test": int(row["n_enrollments"]),
        "n_events_test": int(row["n_events"]),
    })

    print("\n=== P19.1-v4 — Run DONE ===")
    print("heldout:", m, p)
    print("rows train/test:", int(train_df.shape[0]), int(test_df.shape[0]))
    print("enrollments/events (test):", int(row["n_enrollments"]), int(row["n_events"]))
    print("AUC_row_test_ood:", auc)

df_out = pd.DataFrame(results)

# Aggregate
auc_vals = df_out["AUC_row_test_ood"].dropna().to_numpy(dtype=float)
auc_mean = float(np.mean(auc_vals)) if auc_vals.size else np.nan
auc_std  = float(np.std(auc_vals, ddof=0)) if auc_vals.size else np.nan

print("\n=== P19.1-v4 — LOO-Run K Summary ===")
print(df_out.to_string(index=False))
print("\n[Aggregate]")
print("K:", int(df_out.shape[0]))
print("AUC_row_mean:", auc_mean)
print("AUC_row_std:", auc_std)

p_out = os.path.join(TABLE_DIR, f"table_loo_run_k{K}.csv")
df_out.to_csv(p_out, index=False)

print("\n[Export]")
print(p_out)
print("=== P19.1-v4 DONE ===")

=== P19.1-v4 — START ===

=== P19.1-v4 — Selected runs (deterministic) ===
K=5 | MIN_ENROLL=250 | MIN_EVENTS=30
code_module code_presentation  n_enrollments  n_events
        CCC             2014J           2498       836
        FFF             2014J           2365       636
        BBB             2014J           2292       497
        FFF             2013J           2283       504
        BBB             2013J           2237       410

[P19.1-v4][Feature set]
n_numeric: 6 | numeric: ['week', 'total_clicks', 'recency', 'streak', 'studied_credits', 'num_of_prev_attempts']
n_categorical: 3 | categorical: ['highest_education', 'age_band', 'gender']

=== P19.1-v4 — Run DONE ===
heldout: CCC 2014J
rows train/test: 717535 57760
enrollments/events (test): 2498 836
AUC_row_test_ood: 0.7652084862415396

=== P19.1-v4 — Run DONE ===
heldout: FFF 2014J
rows train/test: 718749 56546
enrollments/events (test): 2365 636
AUC_row_test_ood: 0.8392608600524878

=== P19.1-v4 — Run DONE ===
heldout: BBB 

In [972]:
# ==============================================================
# P20-v2 — Endpoint Sensitivity (Primary vs Composite) aligned to P17
# --------------------------------------------------------------
# Fixes horizon inconsistency by using:
#   - T_policy = from P17 (18)
#   - T_eval   = T_eval_metrics from P17 (stable IPCW horizon, e.g., 37)
#
# Computes IPCW Brier/IBS on TEST enrollment backbone for:
#   (A) primary endpoint: Withdrawn (event_withdrawn + t_event_week)
#   (B) composite endpoint: Fail OR Withdrawn (event_comp + t_event_comp_week)
#
# NOTE:
#   OULAD "Fail" does NOT have a native event time like Withdrawn.
#   For this sensitivity, we map Fail to an event time at t_final_week (end-of-observation),
#   so the composite becomes time-defined and single-event-per-enrollment.
#
# Required objects:
#   - KEYS
#   - pp_test_surv with S_hat (for predicted survival)
#   - enrollments_eventtime, enroll_test_keys
#   - studentInfo (for final_result if not already in enrollments_eventtime)
#   - T_policy, T_eval_metrics, G_t (from P17)
#
# Exports:
#   - outputs_v2/tables/table_endpoint_sensitivity.csv
# ==============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------
# 0) Preconditions
# ------------------------------
required = ["KEYS", "pp_test_surv", "enrollments_eventtime", "enroll_test_keys", "T_policy", "T_eval_metrics", "G_t"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"P20-v2 missing required objects: {missing}")

for c in ["week", "S_hat"]:
    if c not in pp_test_surv.columns:
        raise KeyError(f"P20-v2 requires '{c}' in pp_test_surv.")

T_policy_local = int(T_policy)
T_eval_local = int(T_eval_metrics)

G = np.asarray(G_t, dtype=float)
if T_eval_local >= len(G):
    raise ValueError(f"P20-v2: T_eval_local={T_eval_local} exceeds length of G_t ({len(G)}). Run P17 first.")

print("=== P20-v2 — Endpoint Sensitivity Setup (aligned to P17) ===")
print("T_policy:", T_policy_local)
print("T_eval  :", T_eval_local)

# ------------------------------
# 1) Build TEST backbone with final_result
# ------------------------------
bb = (
    enrollments_eventtime.merge(enroll_test_keys, on=KEYS, how="inner")
    .drop_duplicates(subset=KEYS)
    .copy()
)

# Ensure final_result exists; if missing, merge from studentInfo
if "final_result" not in bb.columns:
    if "studentInfo" not in globals():
        raise NameError("P20-v2: final_result missing in enrollments_eventtime and studentInfo not found.")
    bb = bb.merge(
        studentInfo[KEYS + ["final_result"]].drop_duplicates(subset=KEYS),
        on=KEYS,
        how="left",
        validate="m:1",
    )

need_cols = KEYS + ["final_result", "event_withdrawn", "t_event_week", "t_final_week"]
miss = [c for c in need_cols if c not in bb.columns]
if miss:
    raise KeyError(f"P20-v2 missing required columns in backbone: {miss}")

bb["event_withdrawn"] = pd.to_numeric(bb["event_withdrawn"], errors="raise").astype(int)
bb["t_event_week"] = pd.to_numeric(bb["t_event_week"], errors="coerce")
bb["t_final_week"] = pd.to_numeric(bb["t_final_week"], errors="raise").astype(int)

# Primary endpoint truth
E_w = bb["event_withdrawn"].to_numpy(dtype=int)
t_event_w = bb["t_event_week"].fillna(-1).astype(int).to_numpy(dtype=int)
t_final = bb["t_final_week"].to_numpy(dtype=int)

# Composite endpoint: Fail OR Withdrawn
is_fail = (bb["final_result"].astype(str).str.lower() == "fail").to_numpy(dtype=bool)
E_comp = ((E_w == 1) | is_fail).astype(int)

# Define composite event time:
# - if withdrawn: use t_event_week
# - else if fail: use t_final_week (end-of-observation mapping)
t_event_comp = np.where(E_w == 1, t_event_w, t_final).astype(int)

# Sanity: if composite event=0, its t_event_comp is irrelevant; keep defined anyway
# Also ensure single event per enrollment by construction
N_events_primary = int(E_w.sum())
N_events_comp = int(E_comp.sum())

# ------------------------------
# 2) Build S_hat(t) grid up to T_eval with forward-fill (from pp_test_surv)
# ------------------------------
ppS = pp_test_surv[KEYS + ["week", "S_hat"]].copy()
ppS["week"] = pd.to_numeric(ppS["week"], errors="raise").astype(int)
ppS["S_hat"] = pd.to_numeric(ppS["S_hat"], errors="raise").astype(float).clip(1e-12, 1.0)
ppS = ppS.loc[ppS["week"] <= T_eval_local].copy()

S_grid = ppS.pivot_table(index=KEYS, columns="week", values="S_hat", aggfunc="first")

for t in range(0, T_eval_local + 1):
    if t not in S_grid.columns:
        S_grid[t] = np.nan
S_grid = S_grid.reindex(columns=list(range(0, T_eval_local + 1)))
S_grid = S_grid.set_index(pd.MultiIndex.from_frame(bb[KEYS])).reindex(index=pd.MultiIndex.from_frame(bb[KEYS]))

# Fill week 0 if missing
if S_grid[0].isna().any():
    S_grid[0] = S_grid[0].fillna(1.0)

S_grid = S_grid.ffill(axis=1)

if S_grid.isna().any().any():
    miss_rows = int(S_grid.isna().all(axis=1).sum())
    raise ValueError(f"P20-v2: Some TEST enrollments have no S_hat rows after alignment. n={miss_rows}")

p_event_by_t = (1.0 - S_grid.to_numpy(dtype=float)).clip(0.0, 1.0)

p_policy = p_event_by_t[:, T_policy_local]
p_eval = p_event_by_t[:, T_eval_local]

# ------------------------------
# 3) IPCW helpers
# ------------------------------
def brier_ipcw_event_by_T(E, t_event, t_final, p_event_by_T, T, G):
    y = ((E == 1) & (t_event <= T)).astype(int)
    w = np.full_like(p_event_by_T, np.nan, dtype=float)

    mask_event = (y == 1)
    if np.any(mask_event):
        te = t_event[mask_event].astype(int)
        w[mask_event] = 1.0 / G[te]

    mask_obs = (y == 0) & (t_final >= T)
    if np.any(mask_obs):
        w[mask_obs] = 1.0 / G[T]

    se = (y - p_event_by_T) ** 2
    return float(np.nanmean(w * se))

def ibs_ipcw_event(E, t_event, t_final, p_event_by_t, T_max, G):
    vals = []
    for t in range(T_max + 1):
        vals.append(brier_ipcw_event_by_T(E, t_event, t_final, p_event_by_t[:, t], t, G))
    return float(np.nanmean(vals))

def c_index_discrete(E, t_event, t_final, p_event_by_T, T: int) -> float:
    y = ((E == 1) & (t_event <= T)).astype(int)
    idx_event = np.where(y == 1)[0]
    idx_none = np.where((y == 0) & (t_final >= 0))[0]
    if len(idx_event) == 0 or len(idx_none) == 0:
        return np.nan

    conc = 0.0
    ties = 0.0
    total = 0.0
    for i in idx_event:
        ti = t_event[i]
        js = idx_none[t_final[idx_none] >= ti]
        if js.size == 0:
            continue
        pi = p_event_by_T[i]
        pj = p_event_by_T[js]
        total += js.size
        conc += np.sum(pi > pj)
        ties += np.sum(pi == pj)

    if total == 0:
        return np.nan
    return float((conc + 0.5 * ties) / total)

# ------------------------------
# 4) Metrics (Primary)
# ------------------------------
brier_w = brier_ipcw_event_by_T(E_w, t_event_w, t_final, p_eval, T_eval_local, G)
ibs_w = ibs_ipcw_event(E_w, t_event_w, t_final, p_event_by_t, T_eval_local, G)
cidx_w = c_index_discrete(E_w, t_event_w, t_final, p_eval, T_eval_local)

# ------------------------------
# 5) Metrics (Composite)
# ------------------------------
brier_c = brier_ipcw_event_by_T(E_comp, t_event_comp, t_final, p_eval, T_eval_local, G)
ibs_c = ibs_ipcw_event(E_comp, t_event_comp, t_final, p_event_by_t, T_eval_local, G)
cidx_c = c_index_discrete(E_comp, t_event_comp, t_final, p_eval, T_eval_local)

# ------------------------------
# 6) Assemble table + export
# ------------------------------
table = pd.DataFrame(
    [
        ("T_policy", float(T_policy_local), float(T_policy_local)),
        ("T_eval", float(T_eval_local), float(T_eval_local)),
        ("C_index_discrete_T_eval", cidx_w, cidx_c),
        ("IPCW_Brier_T_eval", brier_w, brier_c),
        ("IPCW_IBS_0_T_eval", ibs_w, ibs_c),
        ("N_events", float(N_events_primary), float(N_events_comp)),
    ],
    columns=["metric", "primary_withdrawn", "composite_fail_or_withdrawn"],
)

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)
p_out = os.path.join(TABLE_DIR, "table_endpoint_sensitivity.csv")
table.to_csv(p_out, index=False)

print("\n=== P20-v2 — Endpoint Sensitivity (Primary vs Composite; aligned) ===")
print(table.to_string(index=False))
print("\n[Export]")
print(p_out)
print("=== P20-v2 DONE ===")

=== P20-v2 — Endpoint Sensitivity Setup (aligned to P17) ===
T_policy: 18
T_eval  : 37
[TABLE] saved: outputs_v2/tables/table_endpoint_sensitivity.csv | shape=(6, 3)
columns: ['metric', 'primary_withdrawn', 'composite_fail_or_withdrawn']
sample (n=5):
                    metric  primary_withdrawn  composite_fail_or_withdrawn
0                 T_policy          18.000000                    18.000000
1                   T_eval          37.000000                    37.000000
2  C_index_discrete_T_eval           0.496529                     0.496455
3        IPCW_Brier_T_eval           0.854225                     1.079949
4        IPCW_IBS_0_T_eval           0.242950                     0.329068

=== P20-v2 — Endpoint Sensitivity (Primary vs Composite; aligned) ===
                 metric  primary_withdrawn  composite_fail_or_withdrawn
               T_policy          18.000000                    18.000000
                 T_eval          37.000000                    37.000000
C_index_dis

# P22 - Select + Validate Group Variable (disability)

# P23 — RQ3 Step 1 (Ensure group_var is present in pp_train/pp_test)

# P24 — RQ3 Step 2: Train subgroup-aware discrete-time model

# P25 — RQ3 Step 3 (Policy simulation + group summaries, NO exports)

# P26 — RQ3 Step 4: Bootstrap CI for Δgap (NO exports, NO p-value)

# P27 — RQ3 Step 5: Paper-ready summaries (NO exports) [FIXED]

# P28 — Export tables + figures for RQ3 

In [ ]:
# ==============================================================
# P28-v2 — RQ3 Step 6: Export paper-used tables + figure
# --------------------------------------------------------------
# Export only the fairness artifacts cited in the paper.
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("\n[P28-v2:RQ3:S6] Starting RQ3 Step 6 (paper-only exports).")

if "RQ3_MODELS" not in globals() or "RQ3_POLICY" not in RQ3_MODELS or "RQ3_SUMMARIES" not in RQ3_MODELS:
    raise KeyError("P28-v2 requires RQ3_MODELS['RQ3_POLICY'] and ['RQ3_SUMMARIES'] (run P25–P27).")

policy = RQ3_MODELS["RQ3_POLICY"]
summ = RQ3_MODELS["RQ3_SUMMARIES"]

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

T_policy_local = int(policy["T_policy"])
T_eval_local = int(policy["T_eval"])

gm_tp = summ["group_means_Tpolicy"].copy()
gm_te = summ["group_means_Teval"].copy()
gap = summ["anchored_policy"]

boot_tp = policy.get("bootstrap_delta_gap_Tpolicy", None)
boot_te = policy.get("bootstrap_delta_gap_Teval", None)
if boot_tp is None or boot_te is None:
    raise KeyError("P28-v2 requires bootstrap arrays in policy (run P26-v2).")

ci_tp = policy.get("delta_gap_Tpolicy_ci95", None)
ci_te = policy.get("delta_gap_Teval_ci95", None)

df_gap = pd.DataFrame([{
    "group_var": gap["group_var"],
    "group0": gap["group0"],
    "group1": gap["group1"],
    "T_policy": T_policy_local,
    "T_eval": T_eval_local,
    "T_eval_metrics": T_eval_local,
    "r_star": gap["r_star"],
    "W": gap["W"],
    "alpha_week0": gap["alpha_week0"],
    "alpha_week1": gap["alpha_week1"],
    "decay_type": gap["decay_type"],
    "treated_enrollments": gap["treated_enrollments"],
    "delta_gap_Tpolicy": gap["delta_gap_Tpolicy"],
    "delta_gap_Tpolicy_ci95_lo": ci_tp[0] if ci_tp else np.nan,
    "delta_gap_Tpolicy_ci95_hi": ci_tp[1] if ci_tp else np.nan,
    "delta_gap_Teval": gap["delta_gap_Teval"],
    "delta_gap_Teval_ci95_lo": ci_te[0] if ci_te else np.nan,
    "delta_gap_Teval_ci95_hi": ci_te[1] if ci_te else np.nan,
    "delta_gap_Teval_metrics": gap["delta_gap_Teval"],
    "delta_gap_Teval_metrics_ci95_lo": ci_te[0] if ci_te else np.nan,
    "delta_gap_Teval_metrics_ci95_hi": ci_te[1] if ci_te else np.nan,
}])

gm_tp2 = gm_tp.copy()
gm_tp2["horizon"] = "T_policy"
gm_te2 = gm_te.copy()
gm_te2["horizon"] = "T_eval"
df_means = pd.concat([gm_tp2, gm_te2], axis=0, ignore_index=True)[["horizon", "group", "mean_S_base", "mean_S_policy"]]
df_means["horizon_explicit"] = df_means["horizon"].replace({"T_eval": "T_eval_metrics"})

df_boot_wide = pd.DataFrame({"delta_gap_Tpolicy": boot_tp, "delta_gap_Teval": boot_te})
df_boot_wide["delta_gap_Teval_metrics"] = df_boot_wide["delta_gap_Teval"]

p_gap = os.path.join(TABLE_DIR, "rq3_policy_gaps_unified.csv")
p_means = os.path.join(TABLE_DIR, "rq3_group_means_unified.csv")
p_boot_wide = os.path.join(TABLE_DIR, "rq3_policy_bootstrap_wide.csv")

df_gap.to_csv(p_gap, index=False)
df_means.to_csv(p_means, index=False)
df_boot_wide.to_csv(p_boot_wide, index=False)

fig_htp = os.path.join(FIG_DIR, "fig_rq3_bootstrap_hist_deltagap_Tpolicy.png")
plt.figure()
plt.hist(boot_tp, bins=30)
plt.title("Bootstrap Δgap (T_policy)")
plt.xlabel("Δgap")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(fig_htp, dpi=200)
plt.close()

print("[P28-v2:RQ3:S6] SUMMARY")
print("  horizons: T_policy=", T_policy_local, "| T_eval=", T_eval_local)
print("  Δgap_Tpolicy:", gap["delta_gap_Tpolicy"], "| CI95:", ci_tp)
print("  Δgap_Teval  :", gap["delta_gap_Teval"], "| CI95:", ci_te)

print("\n[Exports]")
print("-", p_gap)
print("-", p_means)
print("-", p_boot_wide)
print("-", fig_htp)
print("[P28-v2:RQ3:S6] Step 6 COMPLETE.")


[P28-v2:RQ3:S6] Starting RQ3 Step 6 (EXPORT tables + figures).
[TABLE] saved: outputs_v2/tables/rq3_policy_gaps_unified.csv | shape=(1, 21)
columns: ['group_var', 'group0', 'group1', 'T_policy', 'T_eval', 'T_eval_metrics', 'r_star', 'W', 'alpha_week0', 'alpha_week1', 'decay_type', 'treated_enrollments', 'delta_gap_Tpolicy', 'delta_gap_Tpolicy_ci95_lo', 'delta_gap_Tpolicy_ci95_hi', 'delta_gap_Teval', 'delta_gap_Teval_ci95_lo', 'delta_gap_Teval_ci95_hi', 'delta_gap_Teval_metrics', 'delta_gap_Teval_metrics_ci95_lo', 'delta_gap_Teval_metrics_ci95_hi']
sample (n=1):
  group_var group0 group1  T_policy  T_eval  T_eval_metrics  r_star  W  \
0    gender      M      F        18      37              37       1  2   

   alpha_week0  alpha_week1  ... treated_enrollments  delta_gap_Tpolicy  \
0         0.35          0.1  ...                8387          -0.000508   

   delta_gap_Tpolicy_ci95_lo  delta_gap_Tpolicy_ci95_hi  delta_gap_Teval  \
0                  -0.000662                  -0.000329

In [987]:

# ==============================================================
# P28.2 — Fairness Extended (AUC-PR, TPR/FPR at operating point)
# --------------------------------------------------------------
# Purpose:
#   Extend subgroup reporting beyond AUC/Brier/ECE with:
#     - Average Precision (AUC-PR)
#     - TPR/FPR at a fixed operating point chosen by top-k risk
#       where k matches overall event prevalence on TEST.
#
# Requires:
#   - pp_test with: event, hazard_hat, and group_var (=RQ3_GROUP_VAR)
#
# Exports:
#   - outputs_v2/tables/rq3_fairness_extended.csv
# ==============================================================
import os
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score

# ------------------------------
# Preconditions
# ------------------------------
for name in ["pp_test", "RQ3_GROUP_VAR"]:
    if name not in globals():
        raise NameError(f"P28.2 missing required object: {name}")

group_var = str(RQ3_GROUP_VAR)

for c in ["event", "hazard_hat", group_var]:
    if c not in pp_test.columns:
        raise KeyError(f"P28.2 requires '{c}' in pp_test.")

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

df = pp_test.copy()
df["y"] = pd.to_numeric(df["event"], errors="coerce").fillna(0).astype(int)
df["p"] = pd.to_numeric(df["hazard_hat"], errors="coerce").fillna(0.0).astype(float)
df[group_var] = df[group_var].astype("string").fillna("MISSING").astype(str)

# overall operating point: top-k where k equals overall prevalence
prev = float(df["y"].mean())
k_frac = prev  # top fraction
k_frac = min(max(k_frac, 0.001), 0.5)  # keep sane
thr = float(np.quantile(df["p"].to_numpy(), 1.0 - k_frac))

def _tpr_fpr(sub):
    y = sub["y"].to_numpy()
    p = sub["p"].to_numpy()
    yhat = (p >= thr).astype(int)
    tp = int(((yhat == 1) & (y == 1)).sum())
    fp = int(((yhat == 1) & (y == 0)).sum())
    tn = int(((yhat == 0) & (y == 0)).sum())
    fn = int(((yhat == 0) & (y == 1)).sum())
    tpr = tp / max(tp + fn, 1)
    fpr = fp / max(fp + tn, 1)
    return tpr, fpr, tp, fp, tn, fn

rows = []
for g, sub in df.groupby(group_var, sort=True):
    ap = float(average_precision_score(sub["y"], sub["p"])) if sub["y"].nunique() > 1 else np.nan
    tpr, fpr, tp, fp, tn, fn = _tpr_fpr(sub)
    rows.append({
        "group_var": group_var,
        "group": g,
        "n_rows": int(len(sub)),
        "pos_rate": float(sub["y"].mean()),
        "avg_precision_aucpr": ap,
        "operating_point_top_frac": k_frac,
        "threshold": thr,
        "TPR_at_op": tpr,
        "FPR_at_op": fpr,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    })

out = pd.DataFrame(rows)
p_out = os.path.join(TABLE_DIR, "rq3_fairness_extended.csv")
out.to_csv(p_out, index=False)

print("=== P28.2 — Fairness Extended (TEST) ===")
print("[Operating point]")
print("overall prevalence:", prev, "| top_frac:", k_frac, "| threshold:", thr)
print("\n[By group]")
print(out.to_string(index=False))
print("\n[Export]")
print(p_out)
print("=== P28.2 DONE ===")


[TABLE] saved: outputs_v2/tables/rq3_fairness_extended.csv | shape=(2, 13)
columns: ['group_var', 'group', 'n_rows', 'pos_rate', 'avg_precision_aucpr', 'operating_point_top_frac', 'threshold', 'TPR_at_op', 'FPR_at_op', 'TP', 'FP', 'TN', 'FN']
sample (n=2):
  group_var group  n_rows  pos_rate  avg_precision_aucpr  \
0    gender     F  104924  0.009045             0.054354   
1    gender     M  127493  0.009938             0.057685   

   operating_point_top_frac  threshold  TPR_at_op  FPR_at_op   TP    FP  \
0                  0.009535   0.064767   0.076923   0.007685   73   799   
1                  0.009535   0.064767   0.093133   0.009721  118  1227   

       TN    FN  
0  103176   876  
1  124999  1149  
=== P28.2 — Fairness Extended (TEST) ===
[Operating point]
overall prevalence: 0.009534586540571472 | top_frac: 0.009534586540571472 | threshold: 0.06476651619226206

[By group]
group_var group  n_rows  pos_rate  avg_precision_aucpr  operating_point_top_frac  threshold  TPR_at_op  

In [ ]:
# ==============================================================
# P29 — Final Paper Artifact Cleanup
# --------------------------------------------------------------
# Purpose:
#   Ensure outputs_v2 ends with only the figures/tables cited in the paper.
# ==============================================================

import json
import os

OUT_DIR = "outputs_v2"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
META_PATH = os.path.join(OUT_DIR, "metadata.json")

KEEP_TABLES = {
    "rq3_fairness_extended.csv",
    "rq3_group_means_unified.csv",
    "rq3_policy_bootstrap_wide.csv",
    "rq3_policy_gaps_unified.csv",
    "table_ablation_m4.csv",
    "table_benchmark_horizon_auc.csv",
    "table_calibration_bin_support_tail3_test.csv",
    "table_calibration_metrics.csv",
    "table_censoring_survival_G_test_by_week.csv",
    "table_endpoint_sensitivity.csv",
    "table_loo_run_k5.csv",
    "table_policy_deltaS_at_horizons_by_scenario.csv",
    "table_policy_deltaS_by_week_by_scenario.csv",
    "table_policy_horizons_dual.csv",
    "table_policy_scenario_params.csv",
    "table_policy_scenarios_main.csv",
    "table_policy_spec.csv",
}

KEEP_FIGURES = {
    "fig_calibration_reliability_test.png",
    "fig_censoring_survival_G_test.png",
    "fig_policy_deltaS_by_week_shock_scenarios.png",
    "fig_policy_survival_by_week_baseline_shock_mech.png",
    "fig_rq3_bootstrap_hist_deltagap_Tpolicy.png",
}

def _prune_dir(dir_path: str, keep_names: set[str]) -> list[str]:
    removed = []
    if not os.path.isdir(dir_path):
        return removed
    for name in sorted(os.listdir(dir_path)):
        path = os.path.join(dir_path, name)
        if not os.path.isfile(path):
            continue
        if name not in keep_names:
            os.remove(path)
            removed.append(name)
    return removed

removed_tables = _prune_dir(TABLE_DIR, KEEP_TABLES)
removed_figures = _prune_dir(FIG_DIR, KEEP_FIGURES)

kept_table_paths = [
    f"tables/{name}"
    for name in sorted(KEEP_TABLES)
    if os.path.exists(os.path.join(TABLE_DIR, name))
]
kept_figure_paths = [
    f"figures/{name}"
    for name in sorted(KEEP_FIGURES)
    if os.path.exists(os.path.join(FIG_DIR, name))
]

missing_tables = sorted(name for name in KEEP_TABLES if not os.path.exists(os.path.join(TABLE_DIR, name)))
missing_figures = sorted(name for name in KEEP_FIGURES if not os.path.exists(os.path.join(FIG_DIR, name)))

metadata = {}
if os.path.exists(META_PATH):
    with open(META_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

metadata["tables"] = kept_table_paths
metadata["figures"] = kept_figure_paths
metadata.setdefault("policy", {})
metadata["policy"]["scenario_catalog_main_table"] = "tables/table_policy_scenarios_main.csv"
metadata["policy"]["scenario_catalog_table"] = "tables/table_policy_scenario_params.csv"
metadata["policy"]["deltaS_tables"] = {
    "weekly_by_scenario": "tables/table_policy_deltaS_by_week_by_scenario.csv",
    "horizons_by_scenario": "tables/table_policy_deltaS_at_horizons_by_scenario.csv",
}
metadata["policy"]["figures"] = {
    "survival": "figures/fig_policy_survival_by_week_baseline_shock_mech.png",
    "scenario_deltaS": "figures/fig_policy_deltaS_by_week_shock_scenarios.png",
}
if os.path.exists(os.path.join(TABLE_DIR, "table_policy_spec.csv")):
    metadata["policy"]["policy_spec_table"] = "tables/table_policy_spec.csv"
metadata.setdefault("artifacts", {})
metadata["artifacts"]["paper_notebook"] = "TCM-Student-Dropout_updated_2.ipynb"

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, sort_keys=True)

print("=== P29 — Final Paper Artifact Cleanup ===")
print("[Removed tables]")
print(removed_tables if removed_tables else "(none)")
print("\n[Removed figures]")
print(removed_figures if removed_figures else "(none)")
print("\n[Kept tables]")
print(kept_table_paths)
print("\n[Kept figures]")
print(kept_figure_paths)
print("\n[Missing expected tables]")
print(missing_tables if missing_tables else "(none)")
print("\n[Missing expected figures]")
print(missing_figures if missing_figures else "(none)")
print("\n[Metadata]")
print(META_PATH)
print("=== P29 DONE ===")